# Experimento VQE — versão 19.4  
## Hipótese financeira multi-$k$ e controle de permutação da ordem dos ativos

### Pergunta científica

A investigação anterior mostrou que, no problema original com dez ativos e
$k=4$, os parâmetros $\theta_2$, $\theta_{14}$ e $\theta_{17}$ concentram os
maiores efeitos causais. Esses parâmetros tocam os qubits associados a AMD e
COST.

Ainda existem duas explicações compatíveis com esse resultado:

1. **hipótese financeira:** a importância causal segue os ativos com decisões
   financeiras mais robustas;
2. **hipótese de coincidência topológica:** AMD e COST parecem especiais apenas
   porque ficaram vizinhos na ordem original do circuito.

Este notebook executa dois testes complementares:

- **teste multi-$k$:** muda a cardinalidade $k$ mantendo os mesmos dez ativos;
- **teste de permutação:** mantém o mesmo problema financeiro, mas altera a
  posição dos ativos nos qubits.

### Hipótese falsificável

Se a explicação financeira estiver correta:

- o par previsto deve ser recalculado em cada configuração;
- os parâmetros fortes devem acompanhar os qubits desse par;
- uma permutação de colunas não deve alterar o espectro clássico nem o conjunto
  físico de ativos selecionados;
- ordens com um par vizinho financeiramente mais robusto devem apresentar maior
  efeito causal nos parâmetros que tocam esse par;
- parâmetros previstos devem superar controles do mesmo tipo de porta, mas com
  baixa exposição financeira.

### O que não será considerado evidência

O notebook não tratará como confirmação:

- rankings formados somente por efeitos minúsculos;
- bancos com menos de 100 vetores na conclusão multi-$k$;
- bancos sem comprovação de 250 avaliações e `warmup=0`;
- o modo piloto do teste de permutação;
- um p-valor pequeno sem tamanho de efeito relevante;
- a ordem deliberada misturada à correlação das ordens aleatórias.

### Convenções essenciais

- `bitstring_asset_order`: $x_0,x_1,\ldots,x_9$;
- `bitstring_qiskit_order`: $q_9,q_8,\ldots,q_0$;
- `CRY`: período físico usado na análise angular igual a $4\pi$;
- `RY`: período físico usado igual a $2\pi$;
- energia final e probabilidade ótima são reavaliadas por `Statevector`, sem
  ruído de shots.


## Dependências e ambiente

O notebook usa:

- `numpy`, `pandas`, `matplotlib` e `scipy`;
- `cvxpy` para a formulação clássica binária;
- `docplex` e `qiskit-optimization` para QUBO/Ising;
- `qiskit`, `qiskit-aer` e `qiskit-algorithms`;
- um solver MIQP opcional para CVXPY.

Instalação sugerida no mesmo kernel do notebook:

```python
# %pip install --upgrade \
#     numpy pandas matplotlib scipy cvxpy \
#     docplex qiskit qiskit-aer \
#     qiskit-optimization qiskit-algorithms
```

O SCIP é opcional. Quando nenhum solver MIQP estiver disponível, a enumeração
exata de todas as carteiras continua fornecendo a solução clássica de referência.

Depois de instalar ou atualizar Qiskit, reinicie o kernel antes de executar as
células seguintes.


In [ ]:
# ============================================================
# 1. IMPORTS, CAMINHOS E CONFIGURAÇÃO GLOBAL
# ============================================================

from __future__ import annotations

from itertools import combinations
from pathlib import Path
from time import perf_counter
from typing import Any
import ast
import hashlib
import json
import math
import os
import warnings

import cvxpy as cp
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from scipy.stats import (
    spearmanr,
    wilcoxon,
)

from docplex.mp.model import Model

from qiskit import (
    QuantumCircuit,
    QuantumRegister,
)

from qiskit.circuit import (
    ParameterVector,
)

from qiskit.primitives import (
    BackendEstimatorV2,
)

from qiskit.quantum_info import (
    Statevector,
)

from qiskit_aer import (
    AerSimulator,
)

from qiskit_optimization.translators import (
    from_docplex_mp,
)

from qiskit_optimization.converters import (
    QuadraticProgramToQubo,
)

try:
    from qiskit_algorithms.minimum_eigensolvers import (
        NumPyMinimumEigensolver,
    )

    from qiskit_algorithms.optimizers import (
        COBYLA,
    )

except ImportError:
    # Compatibilidade com instalações antigas do Qiskit.
    from qiskit.algorithms.minimum_eigensolvers import (
        NumPyMinimumEigensolver,
    )

    from qiskit.algorithms.optimizers import (
        COBYLA,
    )


NOTEBOOK_VERSION = "19.4"


# ------------------------------------------------------------
# CAMINHOS DO PROJETO
# ------------------------------------------------------------

def locate_project_file(
    relative_candidates,
    max_parent_levels=6,
):
    """
    Procura um arquivo a partir da pasta atual do kernel e das
    pastas superiores.

    Isso elimina a dependência de o VS Code/Jupyter iniciar
    exatamente na pasta em que o notebook está salvo.
    """
    current_directory = Path.cwd().resolve()

    search_roots = [
        current_directory,
        *list(
            current_directory.parents
        )[
            :max_parent_levels
        ],
    ]

    checked_paths = []

    for candidate in relative_candidates:
        candidate = Path(
            candidate
        ).expanduser()

        if candidate.is_absolute():
            resolved = candidate.resolve()

            checked_paths.append(
                resolved
            )

            if resolved.is_file():
                return (
                    resolved,
                    checked_paths,
                )

            continue

        for root in search_roots:
            resolved = (
                root
                / candidate
            ).resolve()

            if resolved not in checked_paths:
                checked_paths.append(
                    resolved
                )

            if resolved.is_file():
                return (
                    resolved,
                    checked_paths,
                )

    return (
        None,
        checked_paths,
    )


stock_path_candidates = [
    Path("data/assets/stocks_price.csv"),
    Path("data/assets/stocks_prices.csv"),
    Path("../data/assets/stocks_price.csv"),
    Path("../data/assets/stocks_prices.csv"),
]

stock_path, checked_stock_paths = (
    locate_project_file(
        stock_path_candidates
    )
)

if stock_path is None:
    checked_text = "\n".join(
        f" - {path}"
        for path
        in checked_stock_paths
    )

    raise FileNotFoundError(
        "O CSV de preços não foi localizado.\n"
        f"Pasta atual do kernel: {Path.cwd().resolve()}\n"
        "Caminhos verificados:\n"
        f"{checked_text}"
    )


# ------------------------------------------------------------
# DEFINIÇÃO DO PROBLEMA ORIGINAL
# ------------------------------------------------------------

n_assets = 10
budget = 4

q_value = 0.5
risk_free = 0.0475

# Referências históricas usadas exclusivamente para auditar
# a reconstrução do caso original. Elas NÃO são reutilizadas
# nas cópias embaralhadas.
known_exact_bitstring = "1001011000"
known_exact_energy = -0.8181062577496281


# ------------------------------------------------------------
# CONTROLE DAS FRENTES EXPERIMENTAIS
# ------------------------------------------------------------

# Parte VIII:
# testa a hipótese para k = 2, 3, 4 e 5.
run_multi_k_analysis = True

# False: fluxo preliminar.
# True: exige bancos com 100 vetores por configuração.
financial_robust_mode = False

financial_runs_per_config = (
    100
    if financial_robust_mode
    else 8
)

# O baseline robusto confirmado no relatório é:
#   - nenhum aquecimento;
#   - 250 avaliações COBYLA.
financial_cobyla_maxiter = 250

shots = 4096
random_seed = 42
near_optimal_threshold = 1e-3

# True gera ou completa bancos multi-k.
# False apenas carrega bancos já existentes.
generate_financial_banks = False

# Permite ler bancos antigos somente para depuração.
# Eles nunca recebem status de evidência final.
allow_legacy_bank_fallback = True

financial_discovery_fraction = 0.50

financial_n_test_vectors = (
    20
    if financial_robust_mode
    else 3
)

financial_angle_bins = 8

financial_min_runs_per_bin = (
    4
    if financial_robust_mode
    else 1
)

financial_configs = [
    (10, 2),
    (10, 3),
    (10, 4),
    (10, 5),
]

# O score causal comparável está na escala aproximada [0, 1].
# O limiar abaixo é operacional e evita declarar como sinal
# relevante o maior valor de uma tabela formada apenas por ruído.
minimum_causal_score = 0.01


# ------------------------------------------------------------
# SOLVERS CVXPY
# ------------------------------------------------------------

# O primeiro solver MIQP disponível será usado.
# A enumeração exata permanece como referência independente.
MIQP_SOLVER_PRIORITY = [
    "GUROBI",
    "CPLEX",
    "MOSEK",
    "SCIP",
    "XPRESS",
    "COPT",
    "ECOS_BB",
]


# ------------------------------------------------------------
# DIRETÓRIO CURTO DE RESULTADOS
# ------------------------------------------------------------

# Caminhos curtos evitam WinError 206 no Windows.
custom_results_root = os.environ.get(
    "VQE_RESULTS_DIR"
)

if custom_results_root:
    results_root = Path(
        custom_results_root
    ).expanduser()

elif os.name == "nt":
    results_root = (
        Path.home()
        / "vqe_r"
    )

else:
    results_root = Path(
        "vqe_r"
    )

output_dir = (
    results_root
    / "base"
)

output_dir.mkdir(
    parents=True,
    exist_ok=True,
)

rng = np.random.default_rng(
    random_seed
)


# ------------------------------------------------------------
# RESUMO DA CONFIGURAÇÃO
# ------------------------------------------------------------

print(
    f"notebook: versão {NOTEBOOK_VERSION}"
)

print(
    "pasta atual do kernel:",
    Path.cwd().resolve(),
)

print(
    "arquivo de preços:",
    stock_path,
)

print(
    "diretório de resultados:",
    output_dir.resolve(),
)

print(
    "problema original:",
    f"{n_assets} ativos, escolher {budget}",
)

print(
    "análise multi-k:",
    run_multi_k_analysis,
)

print(
    "modo robusto multi-k:",
    financial_robust_mode,
)

print(
    "gerar bancos multi-k:",
    generate_financial_banks,
)


# Parte I — reconstrução e auditoria do problema original

## 1. Preços, retornos e covariância

O CSV é carregado com:

```python
pd.read_csv(stock_path, index_col=0)
```

A função objetivo usa:

$$
f(x)
=
q\,x^\top\Sigma x
-
(1-q)\,\mu^\top x
+
r_f,
$$

sujeita a:

$$
x_i\in\{0,1\},
\qquad
\sum_i x_i=k.
$$

Neste projeto:

- $q=0{,}5$;
- $r_f=0{,}0475$;
- $\mu$ é a **soma** dos retornos diários;
- $\Sigma$ é a covariância dos retornos diários.

Trocar `sum()` por `mean()` não é uma refatoração: cria outro problema e outro
Hamiltoniano.


In [ ]:
# ============================================================
# 2. PREÇOS -> RETORNOS DIÁRIOS -> RETORNO ACUMULADO
# ============================================================

# A leitura usa o caminho já resolvido na célula de configuração.
# O CSV original é carregado uma única vez e nunca é sobrescrito.
stocks_prices = pd.read_csv(
    stock_path,
    index_col=0,
)

# Converte todas as colunas para número. Valores inválidos viram
# NaN e são tratados de maneira explícita.
stocks_prices = (
    stocks_prices
    .apply(
        pd.to_numeric,
        errors="coerce",
    )
    .replace(
        [
            np.inf,
            -np.inf,
        ],
        np.nan,
    )
    .dropna(
        axis=1,
        how="all",
    )
    .ffill()
    .bfill()
    .dropna()
)

if stocks_prices.empty:
    raise ValueError(
        "O CSV foi localizado, mas não contém dados numéricos válidos."
    )

if stocks_prices.columns.duplicated().any():
    duplicated = (
        stocks_prices.columns[
            stocks_prices.columns.duplicated()
        ]
        .tolist()
    )

    raise ValueError(
        "O CSV possui tickers repetidos: "
        f"{duplicated}"
    )

if stocks_prices.shape[1] != n_assets:
    raise ValueError(
        f"Esperados {n_assets} ativos, mas o CSV possui "
        f"{stocks_prices.shape[1]} colunas numéricas.\n"
        f"Colunas encontradas: {stocks_prices.columns.tolist()}"
    )

if len(stocks_prices) < 3:
    raise ValueError(
        "São necessárias ao menos três datas para calcular "
        "retornos e covariância."
    )

# Relação usada no experimento original.
daily_returns = (
    stocks_prices
    .pct_change(
        fill_method=None
    )
    .dropna()
)

if daily_returns.empty:
    raise ValueError(
        "Não foi possível calcular retornos diários válidos."
    )

# IMPORTANTE:
# O alvo original usa a SOMA dos retornos diários, não a média.
# Trocar sum() por mean() criaria outro Hamiltoniano.
assets_returns = (
    daily_returns.sum()
)

covariance = (
    daily_returns.cov()
)

# A simetrização remove apenas assimetrias de arredondamento.
covariance_values = (
    covariance.to_numpy(
        dtype=float
    )
)

covariance_values = 0.5 * (
    covariance_values
    + covariance_values.T
)

assets_return_values = (
    assets_returns.to_numpy(
        dtype=float
    )
)

tickers = list(
    stocks_prices.columns
)

# Auditorias numéricas básicas.
if not np.all(
    np.isfinite(
        assets_return_values
    )
):
    raise ValueError(
        "O vetor de retornos contém valores não finitos."
    )

if not np.all(
    np.isfinite(
        covariance_values
    )
):
    raise ValueError(
        "A matriz de covariância contém valores não finitos."
    )

covariance_eigenvalues = (
    np.linalg.eigvalsh(
        covariance_values
    )
)

if covariance_eigenvalues.min() < -1e-10:
    raise ValueError(
        "A covariância possui autovalor negativo além da "
        "tolerância numérica: "
        f"{covariance_eigenvalues.min():.3e}"
    )

print(
    "tickers:",
    tickers,
)

print(
    "preços:",
    stocks_prices.shape,
)

print(
    "retornos diários:",
    daily_returns.shape,
)

print(
    "menor autovalor da covariância:",
    f"{covariance_eigenvalues.min():.3e}",
)

display(
    assets_returns.to_frame(
        "assets_return"
    )
)

display(
    covariance
)


## 2. Referência clássica CVXPY

O CVXPY representa diretamente a variável binária de seleção. A tentativa de
solução percorre uma lista de solvers MIQP instalados.

A função do CVXPY aqui é dupla:

1. confirmar que a formulação clássica foi montada corretamente;
2. oferecer uma solução independente do pipeline DOcplex $\rightarrow$ QUBO.

A ausência de solver não interrompe o notebook, pois a próxima célula enumera
todas as $\binom{10}{4}=210$ carteiras.


In [ ]:
# ============================================================
# 3. FORMULAÇÃO CLÁSSICA COM CVXPY
# ============================================================

def build_cvxpy_portfolio_problem(
    return_values,
    covariance_matrix,
    k_value,
):
    """
    Constrói a mesma função objetivo binária usada no DOcplex.

    O termo risk_free é constante: desloca todas as energias
    igualmente, mas não altera o portfólio ótimo.
    """
    return_values = np.asarray(
        return_values,
        dtype=float,
    ).reshape(-1)

    covariance_matrix = np.asarray(
        covariance_matrix,
        dtype=float,
    )

    n_value = int(
        len(
            return_values
        )
    )

    variable = cp.Variable(
        shape=(
            n_value,
        ),
        boolean=True,
        name="portfolio_selection",
    )

    objective = cp.Minimize(
        q_value
        * cp.quad_form(
            variable,
            cp.psd_wrap(
                covariance_matrix
            ),
        )
        - (
            1
            - q_value
        )
        * variable
        @ return_values
        + risk_free
    )

    constraints = [
        cp.sum(
            variable
        )
        == int(
            k_value
        ),
    ]

    problem = cp.Problem(
        objective=objective,
        constraints=constraints,
    )

    return (
        problem,
        variable,
    )


def try_solve_cvxpy_miqp(
    problem,
    variable,
):
    """
    Tenta os solvers MIQP instalados, na ordem de prioridade.

    Falhar aqui não invalida o experimento porque a enumeração
    exata fornece uma segunda referência clássica independente.
    """
    installed = set(
        cp.installed_solvers()
    )

    errors = []

    for solver_name in MIQP_SOLVER_PRIORITY:
        if solver_name not in installed:
            continue

        try:
            value = problem.solve(
                solver=solver_name,
                verbose=False,
            )

            if variable.value is None:
                continue

            solution = np.rint(
                np.asarray(
                    variable.value
                ).reshape(-1)
            ).astype(int)

            return {
                "solver": solver_name,
                "status": problem.status,
                "value": float(
                    value
                ),
                "solution": solution,
                "errors": errors,
            }

        except Exception as exc:
            errors.append(
                {
                    "solver": solver_name,
                    "error": (
                        f"{type(exc).__name__}: {exc}"
                    ),
                }
            )

    return {
        "solver": None,
        "status": "miqp_solver_unavailable",
        "value": np.nan,
        "solution": None,
        "errors": errors,
    }


problem_cvx, var_x = (
    build_cvxpy_portfolio_problem(
        return_values=assets_return_values,
        covariance_matrix=covariance_values,
        k_value=budget,
    )
)

cvxpy_result = try_solve_cvxpy_miqp(
    problem=problem_cvx,
    variable=var_x,
)

cvxpy_status = cvxpy_result[
    "status"
]

cvxpy_solution = cvxpy_result[
    "solution"
]

cvxpy_value = cvxpy_result[
    "value"
]

print(
    "solver CVXPY:",
    cvxpy_result[
        "solver"
    ],
)

print(
    "status CVXPY:",
    cvxpy_status,
)

print(
    "valor CVXPY:",
    cvxpy_value,
)

print(
    "solução CVXPY:",
    cvxpy_solution,
)

if cvxpy_result[
    "solver"
] is None:
    warnings.warn(
        "Nenhum solver MIQP compatível foi encontrado. "
        "A enumeração exata das 210 carteiras será usada."
    )


## 3. Enumeração exata e ordem dos bits

A enumeração calcula a energia de toda carteira viável. Ela também resolve uma
fonte frequente de erro: a diferença entre a ordem econômica e a ordem exibida
pelo Qiskit.

Exemplo:

```text
ordem econômica: x0 x1 x2 ... x9
ordem Qiskit:    q9 q8 q7 ... q0
```

O notebook conserva as duas representações em colunas separadas e mede o gap
entre a melhor e a segunda melhor carteira.


In [ ]:
# ============================================================
# 4. ENUMERAÇÃO EXATA DAS C(10,4) = 210 CARTEIRAS
# ============================================================

def original_classical_objective(
    x_binary,
):
    """
    Função objetivo clássica compartilhada por CVXPY e DOcplex.
    """
    x_binary = np.asarray(
        x_binary,
        dtype=float,
    ).reshape(-1)

    risk_term = (
        q_value
        * x_binary
        @ covariance_values
        @ x_binary
    )

    return_term = (
        (
            1
            - q_value
        )
        * x_binary
        @ assets_return_values
    )

    return float(
        risk_term
        - return_term
        + risk_free
    )


enumeration_rows = []

for selected_indices in combinations(
    range(
        n_assets
    ),
    budget,
):
    x_candidate = np.zeros(
        n_assets,
        dtype=int,
    )

    x_candidate[
        list(
            selected_indices
        )
    ] = 1

    # Ordem econômica:
    # x_0, x_1, ..., x_9.
    bitstring_asset_order = "".join(
        str(
            int(
                value
            )
        )
        for value
        in x_candidate
    )

    # Ordem mostrada pelo Qiskit:
    # q_9, q_8, ..., q_0.
    bitstring_qiskit_order = (
        bitstring_asset_order[
            ::-1
        ]
    )

    enumeration_rows.append(
        {
            "selected_indices": tuple(
                selected_indices
            ),
            "x_asset_order": (
                x_candidate
            ),
            "bitstring_asset_order": (
                bitstring_asset_order
            ),
            "bitstring_qiskit_order": (
                bitstring_qiskit_order
            ),
            "objective": (
                original_classical_objective(
                    x_candidate
                )
            ),
        }
    )

enumeration_df = (
    pd.DataFrame(
        enumeration_rows
    )
    .sort_values(
        "objective"
    )
    .reset_index(
        drop=True
    )
)

expected_portfolio_count = math.comb(
    n_assets,
    budget,
)

if len(
    enumeration_df
) != expected_portfolio_count:
    raise RuntimeError(
        "A enumeração não produziu o número esperado de carteiras: "
        f"{len(enumeration_df)} != {expected_portfolio_count}"
    )

enumeration_best = (
    enumeration_df.iloc[
        0
    ]
)

classical_exact_value = float(
    enumeration_best[
        "objective"
    ]
)

classical_exact_x = np.asarray(
    enumeration_best[
        "x_asset_order"
    ],
    dtype=int,
)

classical_asset_bitstring = str(
    enumeration_best[
        "bitstring_asset_order"
    ]
)

classical_qiskit_bitstring = str(
    enumeration_best[
        "bitstring_qiskit_order"
    ]
)

selected_tickers = [
    ticker
    for ticker, selected
    in zip(
        tickers,
        classical_exact_x,
    )
    if selected == 1
]

# A diferença para a segunda melhor carteira mede a unicidade
# e a estabilidade da solução exata.
second_best_gap = float(
    enumeration_df.loc[
        1,
        "objective",
    ]
    - classical_exact_value
)

print(
    "carteiras avaliadas:",
    len(
        enumeration_df
    ),
)

print(
    "valor clássico exato:",
    classical_exact_value,
)

print(
    "gap para a segunda melhor:",
    second_best_gap,
)

print(
    "bitstring em ordem dos ativos:",
    classical_asset_bitstring,
)

print(
    "bitstring em ordem Qiskit:",
    classical_qiskit_bitstring,
)

print(
    "ativos selecionados:",
    selected_tickers,
)

if cvxpy_solution is not None:
    cvxpy_objective_audit = (
        original_classical_objective(
            cvxpy_solution
        )
    )

    print(
        "diferença enumeração - CVXPY:",
        abs(
            classical_exact_value
            - cvxpy_objective_audit
        ),
    )

display(
    enumeration_df[
        [
            "selected_indices",
            "bitstring_asset_order",
            "bitstring_qiskit_order",
            "objective",
        ]
    ].head(
        10
    )
)


## 4. QUBO e Hamiltoniano de Ising

O modelo DOcplex é convertido em:

```text
QuadraticProgram → QUBO → operador de Ising
```

Como o custo é um QUBO clássico, o operador deve conter somente termos
diagonais da família $I$, $Z$ e $ZZ$.

A célula interrompe a execução caso encontre qualquer termo $X$ ou $Y$. Essa
checagem formaliza a razão pela qual a antiga frente ADAPT por comutador não
forneceu vantagem para este Hamiltoniano.


In [ ]:
# ============================================================
# 5. DOCPLEX -> QUADRATIC PROGRAM -> QUBO -> ISING
# ============================================================

model = Model(
    name="portfolio_optimization"
)

variables = np.array(
    [
        model.binary_var(
            name=f"x_{index}"
        )
        for index
        in range(
            n_assets
        )
    ],
    dtype=object,
)

# Soma explícita para preservar compatibilidade entre versões
# de NumPy e objetos simbólicos do DOcplex.
risk_expression = model.sum(
    float(
        covariance_values[
            row,
            column,
        ]
    )
    * variables[
        row
    ]
    * variables[
        column
    ]
    for row
    in range(
        n_assets
    )
    for column
    in range(
        n_assets
    )
)

return_expression = model.sum(
    float(
        assets_return_values[
            index
        ]
    )
    * variables[
        index
    ]
    for index
    in range(
        n_assets
    )
)

model.minimize(
    q_value
    * risk_expression
    - (
        1
        - q_value
    )
    * return_expression
    + risk_free
)

model.add_constraint(
    model.sum(
        variables.tolist()
    )
    == budget,
    ctname="budget_exactly_four",
)

quad_model = from_docplex_mp(
    model=model
)

qubo_converter = (
    QuadraticProgramToQubo()
)

qubo = qubo_converter.convert(
    quad_model
)

ising, offset = (
    qubo.to_ising()
)

# Esta auditoria é central para o relatório:
# um QUBO convertido em Ising deve conter somente I, Z e ZZ.
pauli_labels = [
    str(
        label
    )
    for label
    in ising.paulis.to_labels()
]

non_diagonal_paulis = [
    label
    for label
    in pauli_labels
    if (
        "X" in label
        or "Y" in label
    )
]

if non_diagonal_paulis:
    raise RuntimeError(
        "O Hamiltoniano deveria ser diagonal, mas contém "
        f"termos não diagonais: {non_diagonal_paulis[:10]}"
    )

# O LP exportado documenta diretamente a ordem x_0...x_9
# e os coeficientes usados pelo modelo.
lp_export_path = (
    output_dir
    / "portfolio_original.lp"
)

lp_export_path.write_text(
    model.export_as_lp_string(),
    encoding="utf-8",
)

print(
    "modelo LP salvo em:",
    lp_export_path.resolve(),
)

print(
    "qubits do Ising:",
    ising.num_qubits,
)

print(
    "offset:",
    offset,
)

print(
    "termos de Pauli:",
    len(
        pauli_labels
    ),
)

print(
    "Hamiltoniano diagonal?",
    not non_diagonal_paulis,
)


## 5. Solução exata e checagem cruzada

A solução do Ising é comparada simultaneamente com:

- a energia histórica do experimento;
- o bitstring histórico;
- a energia da enumeração;
- o bitstring da enumeração.

A análise só continua quando todas as verificações concordam. Nas cópias
embaralhadas, as referências fixas não são usadas: o bitstring é recalculado
para a nova ordem.


In [ ]:
# ============================================================
# 6. SOLUÇÃO EXATA E AUDITORIA CRUZADA
# ============================================================

exact_solver = (
    NumPyMinimumEigensolver()
)

exact_result = (
    exact_solver
    .compute_minimum_eigenvalue(
        operator=ising
    )
)

exact_energy = float(
    np.real(
        exact_result.eigenvalue
        + offset
    )
)

exact_state_dict = (
    exact_result
    .eigenstate
    .to_dict()
)

# Trabalhar com um conjunto evita assumir unicidade sem verificar.
exact_bitstrings = sorted(
    {
        str(
            bitstring
        ).replace(
            " ",
            "",
        )
        for bitstring, amplitude
        in exact_state_dict.items()
        if abs(
            amplitude
        )
        > 1e-12
    }
)

if not exact_bitstrings:
    raise RuntimeError(
        "O solver exato não retornou nenhum bitstring."
    )

# No caso original há uma solução única. Quando houver mais de uma,
# a análise posterior soma a probabilidade de todas as soluções ótimas.
exact_bitstring = (
    known_exact_bitstring
    if known_exact_bitstring
    in exact_bitstrings
    else exact_bitstrings[
        0
    ]
)

energy_matches_reference = np.isclose(
    exact_energy,
    known_exact_energy,
    atol=1e-10,
    rtol=0.0,
)

energy_matches_enumeration = np.isclose(
    exact_energy,
    classical_exact_value,
    atol=1e-10,
    rtol=0.0,
)

bitstring_matches_reference = (
    known_exact_bitstring
    in exact_bitstrings
)

bitstring_matches_enumeration = (
    classical_qiskit_bitstring
    in exact_bitstrings
)

print(
    "energia exata Ising + offset:",
    exact_energy,
)

print(
    "bitstring(s) ótimo(s):",
    exact_bitstrings,
)

print(
    "energia coincide com a referência histórica?",
    energy_matches_reference,
)

print(
    "energia coincide com a enumeração?",
    energy_matches_enumeration,
)

print(
    "bitstring coincide com a referência histórica?",
    bitstring_matches_reference,
)

print(
    "bitstring coincide com a enumeração?",
    bitstring_matches_enumeration,
)

if not (
    energy_matches_reference
    and energy_matches_enumeration
    and bitstring_matches_reference
    and bitstring_matches_enumeration
):
    raise RuntimeError(
        "A reconstrução não coincide com o experimento original. "
        "Não prossiga para as intervenções antes de revisar o CSV, "
        "a relação retorno=sum(), q, risk_free e a ordem dos bits."
    )

EXACT_ENERGY = exact_energy
EXACT_BITSTRING = exact_bitstring
EXACT_BITSTRINGS = exact_bitstrings


# Parte II — infraestrutura do ansatz de Dicke

## 6. Portas parametrizadas `CY` e `CCY`

Estas funções reproduzem os blocos de baixo nível usados no circuito original.

Os identificadores aleatórios servem apenas para criar nomes simbólicos únicos.
A ordem física de um parâmetro deve ser recuperada pelo circuito decomposto, e
não interpretada a partir do texto do nome.


In [ ]:

# ============================================================
# 7. PORTAS PARAMETRIZADAS
#
# Estas funções são infraestrutura do ansatz. Os nomes
# aleatórios servem somente para garantir identificadores
# simbólicos únicos; não têm interpretação física.
# ============================================================
# 7. PORTAS PARAMETRIZADAS E DickeStateAnsatz
#    IMPLEMENTAÇÃO EMBUTIDA — SEM DEPENDER DE OUTRO NOTEBOOK
# ============================================================

def CY_parameterized(i):
    """
    Cria a porta parametrizada chamada CY no código original.

    Parâmetro
    ---------
    i:
        Identificador usado somente para gerar um nome simbólico
        único para o ângulo da porta.

    Funcionamento
    -------------
    A porta interna é uma rotação controlada em Y:
        controle = qubit 1
        alvo     = qubit 0

    O valor numérico do ângulo só será inserido mais tarde por
    `assign_parameters`.
    """
    param = ParameterVector(
        name=f"x{i}",
        length=1,
    )

    qc = QuantumCircuit(2)

    qc.cry(
        param[0],
        1,
        0,
    )

    gate = qc.to_gate(
        label="CY"
    )

    return gate


def CCY_parameterized(i):
    """
    Cria a porta parametrizada chamada CCY no código original.

    Ela usa:
    1. uma rotação RY;
    2. uma porta Toffoli CCX;
    3. a rotação inversa;
    4. uma segunda CCX.

    O parâmetro continua simbólico até a chamada de
    `assign_parameters`.
    """
    param = ParameterVector(
        name=f"y{i}",
        length=1,
    )

    qc = QuantumCircuit(3)

    qc.ry(
        param[0],
        0,
    )

    qc.ccx(
        2,
        1,
        0,
    )

    qc.ry(
        -param[0],
        0,
    )

    qc.ccx(
        2,
        1,
        0,
    )

    gate = qc.to_gate(
        label="CCY"
    )

    return gate


# Parte III — infraestrutura multi-$k$

O notebook constrói quatro problemas independentes:

$$
(10,2),\ (10,3),\ (10,4),\ (10,5).
$$

Cada contexto contém:

- tickers e coeficientes financeiros;
- Hamiltoniano e offset;
- energia e bitstring ótimos;
- ansatz de Dicke correspondente;
- número e mapa estrutural dos parâmetros;
- simulador e estimador.

Os bancos atuais usam 250 avaliações COBYLA e nenhum aquecimento. Bancos
legados podem ser usados para depuração, mas são marcados como preliminares.


In [ ]:

# ============================================================
# 34. CONFIGURAÇÃO DOS BANCOS MULTI-K
# ============================================================

nk_configs = list(
    financial_configs
)

nk_runs_per_config = int(
    financial_runs_per_config
)

nk_cobyla_maxiter = int(
    financial_cobyla_maxiter
)

nk_shots = int(
    shots
)

# O diretório dos bancos permanece "financial_v19_2" para
# reutilizar os checkpoints robustos já calculados. As novas tabelas
# da comparação direta são gravadas em subpastas v19_4 separadas.
financial_output_dir = (
    output_dir
    / "financial_v19_2"
)

financial_bank_dir = (
    financial_output_dir
    / (
        "bank_r100_b250"
        if financial_robust_mode
        else "bank_pilot_b250"
    )
)

financial_table_dir = (
    financial_output_dir
    / "tables"
)

financial_figure_dir = (
    financial_output_dir
    / "figures"
)

for directory in [
    financial_bank_dir,
    financial_table_dir,
    financial_figure_dir,
]:
    directory.mkdir(
        parents=True,
        exist_ok=True,
    )

print(
    "configurações:",
    nk_configs,
)

print(
    "vetores por configuração:",
    nk_runs_per_config,
)

print(
    "avaliações COBYLA por vetor:",
    nk_cobyla_maxiter,
)

print(
    "diretório:",
    financial_output_dir.resolve(),
)


## 7. Ansatz rastreável

A quantidade de parâmetros desta construção é:

$$
m(n,k)
=
\frac{k(2n-k-1)}{2}.
$$

O mapa estrutural registra:

- `l` e `i`: qubits usados pelo bloco;
- `distance = l-i`;
- `gate_type`: `CY` ou `CCY`;
- coordenadas normalizadas para auditoria.

O índice `theta_index` não é tratado como propriedade física.


In [ ]:
# ============================================================
# 35. ANSATZ DE DICKE COM MAPA ESTRUTURAL
# ============================================================

def dicke_parameter_count(
    n_value,
    k_value,
):
    """
    Quantidade exata de theta desta construção de Dicke.
    """
    return int(
        k_value
        * (
            2
            * n_value
            - k_value
            - 1
        )
        / 2
    )


def build_tracked_dicke_ansatz(
    n_value,
    k_value,
    seed,
):
    """
    Constrói o mesmo ansatz de Dicke e registra a origem estrutural
    de cada parâmetro.

    ATENÇÃO:
    o índice theta_index é apenas a ordem produzida pelo Qiskit.
    A interpretação física deve usar gate_type, l, i e distance.

    O mapa usa:
        l: qubit final do bloco;
        i: qubit inicial;
        d = l - i: distância dentro do bloco;
        gate_type: CY quando d=1, CCY nos demais casos.
    """
    numpy_state = np.random.get_state()

    try:
        np.random.seed(
            int(seed)
        )

        qr = QuantumRegister(
            n_value,
            "q",
        )

        qc = QuantumCircuit(
            qr
        )

        for excitation_index in range(
            k_value
        ):
            qc.x(
                n_value
                - excitation_index
                - 1
            )

        records = []
        aux = 1

        for l_value in range(
            n_value
        )[::-1]:
            for i_value in range(
                l_value - 1,
                l_value - 1 - k_value,
                -1,
            ):
                if i_value >= 0:
                    unique_name = (
                        str(l_value)
                        + str(i_value)
                        + str(aux)
                        + str(
                            np.random.randint(
                                0,
                                int(1e8),
                            )
                        )
                    )

                    if i_value == l_value - 1:
                        gate = CY_parameterized(
                            unique_name
                        )

                        gate_parameter = list(
                            gate.params[0].parameters
                        )[0]

                        qc.cx(
                            qr[i_value],
                            qr[l_value],
                        )

                        qc.append(
                            gate,
                            [
                                qr[i_value],
                                qr[l_value],
                            ],
                        )

                        qc.cx(
                            qr[i_value],
                            qr[l_value],
                        )

                        gate_type = "CY"

                    else:
                        gate = CCY_parameterized(
                            unique_name
                        )

                        gate_parameter = list(
                            gate.params[0].parameters
                        )[0]

                        qc.cx(
                            i_value,
                            l_value,
                        )

                        qc.append(
                            gate,
                            [
                                qr[i_value],
                                qr[i_value + 1],
                                qr[l_value],
                            ],
                        )

                        qc.cx(
                            i_value,
                            l_value,
                        )

                        gate_type = "CCY"

                    records.append(
                        {
                            "parameter_object": gate_parameter,
                            "parameter_name": str(
                                gate_parameter
                            ),
                            "l": int(
                                l_value
                            ),
                            "i": int(
                                i_value
                            ),
                            "distance": int(
                                l_value
                                - i_value
                            ),
                            "gate_type": gate_type,
                        }
                    )

                aux += 1

    finally:
        np.random.set_state(
            numpy_state
        )

    ordered_parameters = list(
        qc.parameters
    )

    parameter_to_index = {
        parameter: index
        for index, parameter
        in enumerate(
            ordered_parameters
        )
    }

    structure_rows = []

    for record in records:
        parameter_object = record.pop(
            "parameter_object"
        )

        theta_index = int(
            parameter_to_index[
                parameter_object
            ]
        )

        structure_rows.append(
            {
                "theta_index": theta_index,
                **record,
            }
        )

    structure_df = (
        pd.DataFrame(
            structure_rows
        )
        .sort_values(
            "theta_index"
        )
        .reset_index(
            drop=True
        )
    )

    expected_count = dicke_parameter_count(
        n_value,
        k_value,
    )

    if (
        len(
            structure_df
        )
        != expected_count
    ):
        raise RuntimeError(
            "Quantidade estrutural incorreta: "
            f"esperado {expected_count}, "
            f"encontrado {len(structure_df)}."
        )

    structure_df[
        "n"
    ] = int(
        n_value
    )

    structure_df[
        "k"
    ] = int(
        k_value
    )

    structure_df[
        "rho_k_over_n"
    ] = (
        k_value
        / n_value
    )

    structure_df[
        "u_l"
    ] = (
        structure_df[
            "l"
        ]
        / max(
            n_value - 1,
            1,
        )
    )

    structure_df[
        "u_distance"
    ] = (
        structure_df[
            "distance"
        ]
        / k_value
    )

    structure_df[
        "u_theta_index"
    ] = (
        structure_df[
            "theta_index"
        ]
        / max(
            expected_count - 1,
            1,
        )
    )

    return (
        qc.decompose(),
        structure_df,
    )


## 8. Contexto completo para cada configuração

A função `build_nk_problem_context` impede misturas acidentais entre:

- theta de um ansatz;
- Hamiltoniano de outro $k$;
- bitstring de outra ordem;
- número incorreto de parâmetros.

A solução exata é recalculada em cada configuração.


In [ ]:
# ============================================================
# 36. CONSTRUIR UM PROBLEMA PARA CADA n E k
# ============================================================

def build_nk_problem_context(
    n_value,
    k_value,
):
    """
    Reconstrói completamente uma configuração (n, k).

    O contexto mantém juntos os dados financeiros, o Hamiltoniano,
    a solução exata e o ansatz correspondente. Isso reduz o risco
    de combinar theta de um circuito com o Hamiltoniano errado.
    """
    if n_value > stocks_prices.shape[1]:
        raise ValueError(
            f"O CSV possui somente {stocks_prices.shape[1]} ativos. "
            f"Não é possível usar n={n_value}."
        )

    selected_tickers = list(
        stocks_prices.columns[
            :n_value
        ]
    )

    local_returns = (
        daily_returns[
            selected_tickers
        ]
    )

    local_assets_returns = (
        local_returns.sum()
    )

    local_covariance = (
        local_returns.cov()
    )

    local_covariance_values = (
        local_covariance.to_numpy(
            dtype=float
        )
    )

    local_covariance_values = 0.5 * (
        local_covariance_values
        + local_covariance_values.T
    )

    local_return_values = (
        local_assets_returns.to_numpy(
            dtype=float
        )
    )

    local_model = Model(
        name=(
            f"portfolio_n{n_value}_k{k_value}"
        )
    )

    local_variables = np.array(
        [
            local_model.binary_var(
                name=f"x_{index}"
            )
            for index in range(
                n_value
            )
        ],
        dtype=object,
    )

    local_risk = local_model.sum(
        float(
            local_covariance_values[
                row,
                column,
            ]
        )
        * local_variables[
            row
        ]
        * local_variables[
            column
        ]
        for row in range(
            n_value
        )
        for column in range(
            n_value
        )
    )

    local_return = local_model.sum(
        float(
            local_return_values[
                index
            ]
        )
        * local_variables[
            index
        ]
        for index in range(
            n_value
        )
    )

    local_model.minimize(
        q_value
        * local_risk
        - (
            1
            - q_value
        )
        * local_return
        + risk_free
    )

    local_model.add_constraint(
        local_model.sum(
            local_variables.tolist()
        )
        == k_value,
        ctname="budget",
    )

    local_quadratic_program = (
        from_docplex_mp(
            model=local_model
        )
    )

    local_qubo = (
        QuadraticProgramToQubo()
        .convert(
            local_quadratic_program
        )
    )

    local_ising, local_offset = (
        local_qubo.to_ising()
    )

    local_exact_result = (
        NumPyMinimumEigensolver()
        .compute_minimum_eigenvalue(
            operator=local_ising
        )
    )

    local_exact_energy = float(
        np.real(
            local_exact_result.eigenvalue
            + local_offset
        )
    )

    local_exact_state = (
        local_exact_result
        .eigenstate
        .to_dict()
    )

    local_exact_bitstrings = sorted(
        {
            str(
                bitstring
            ).replace(
                " ",
                "",
            )
            for bitstring, amplitude
            in local_exact_state.items()
            if abs(
                amplitude
            )
            > 1e-12
        }
    )

    if not local_exact_bitstrings:
        raise RuntimeError(
            f"n={n_value}, k={k_value}: "
            "nenhum bitstring ótimo foi encontrado."
        )

    local_exact_bitstring = (
        local_exact_bitstrings[
            0
        ]
    )

    local_ansatz, local_structure = (
        build_tracked_dicke_ansatz(
            n_value=n_value,
            k_value=k_value,
            seed=(
                random_seed
                + 100
                * n_value
                + k_value
            ),
        )
    )

    local_n_parameters = int(
        local_ansatz.num_parameters
    )

    expected_parameters = (
        dicke_parameter_count(
            n_value,
            k_value,
        )
    )

    if (
        local_n_parameters
        != expected_parameters
    ):
        raise RuntimeError(
            f"n={n_value}, k={k_value}: "
            f"esperados {expected_parameters} theta, "
            f"encontrados {local_n_parameters}."
        )

    local_simulator = AerSimulator(
        method="statevector",
        device="CPU",
    )

    local_estimator = BackendEstimatorV2(
        backend=local_simulator
    )

    return {
        "n": int(
            n_value
        ),
        "k": int(
            k_value
        ),
        "tickers": selected_tickers,
        "assets_returns": local_assets_returns,
        "covariance": local_covariance,
        "ising": local_ising,
        "offset": float(
            local_offset
        ),
        "exact_energy": local_exact_energy,
        "exact_bitstring": local_exact_bitstring,
        "exact_bitstrings": local_exact_bitstrings,
        "ansatz": local_ansatz,
        "n_parameters": local_n_parameters,
        "structure_df": local_structure,
        "simulator": local_simulator,
        "estimator": local_estimator,
    }


## 9. Uma rodada COBYLA

Cada `run_index` gera um ponto inicial determinístico. Isso garante:

- reprodutibilidade;
- retomada de checkpoint;
- comparação pareada;
- ausência de duplicação silenciosa.

A otimização usa o estimador durante a busca e, posteriormente, todo theta é
reavaliado exatamente por `Statevector`.


In [ ]:
# ============================================================
# 37. OTIMIZAR UMA RODADA DO BANCO MULTI-N/K
# ============================================================

def run_nk_cobyla(
    context,
    run_index,
):
    """
    Executa uma única otimização COBYLA.

    A semente depende de (n, k, run_index), permitindo retomar
    checkpoints sem alterar os pontos iniciais já definidos.
    """
    local_rng = np.random.default_rng(
        random_seed
        + 1_000_000
        + 10_000
        * context[
            "n"
        ]
        + 100
        * context[
            "k"
        ]
        + int(
            run_index
        )
    )

    theta_initial = (
        0.5
        * np.pi
        * local_rng.random(
            context[
                "n_parameters"
            ]
        )
    )

    # Guarda o histórico para contar avaliações de energia.
    # Não é usado para escolher o melhor ponto depois do fato.
    local_callback = []

    def local_exp_val(
        theta,
    ):
        theta = np.asarray(
            theta,
            dtype=float,
        ).reshape(-1)

        result = context[
            "estimator"
        ].run(
            pubs=[
                (
                    context[
                        "ansatz"
                    ],
                    context[
                        "ising"
                    ],
                    theta,
                )
            ]
        ).result()

        energy = float(
            np.real(
                np.asarray(
                    result[
                        0
                    ].data.evs
                ).reshape(-1)[
                    0
                ]
            )
            + context[
                "offset"
            ]
        )

        local_callback.append(
            (
                theta.copy(),
                energy,
            )
        )

        return energy

    optimizer = COBYLA(
        maxiter=nk_cobyla_maxiter
    )

    start = perf_counter()

    result = optimizer.minimize(
        fun=local_exp_val,
        x0=theta_initial,
    )

    optimizer_time = (
        perf_counter()
        - start
    )

    theta_final = np.asarray(
        result.x,
        dtype=float,
    ).reshape(-1)

    assigned = (
        context[
            "ansatz"
        ]
        .copy()
        .assign_parameters(
            theta_final,
            inplace=False,
        )
    )

    assigned.measure_all()

    simulator_start = perf_counter()

    counts = (
        context[
            "simulator"
        ]
        .run(
            assigned,
            shots=nk_shots,
            seed_simulator=(
                random_seed
                + 2_000_000
                + 10_000
                * context[
                    "n"
                ]
                + 100
                * context[
                    "k"
                ]
                + int(
                    run_index
                )
            ),
        )
        .result()
        .get_counts()
    )

    simulator_time = (
        perf_counter()
        - simulator_start
    )

    counts_clean = {
        str(key).replace(
            " ",
            "",
        ): int(value)
        for key, value
        in counts.items()
    }

    total_counts = int(
        sum(
            counts_clean.values()
        )
    )

    most_frequent = max(
        counts_clean,
        key=counts_clean.get,
    )

    exact_probability = (
        counts_clean.get(
            context[
                "exact_bitstring"
            ],
            0,
        )
        / total_counts
    )

    final_energy = float(
        result.fun
    )

    return {
        "n": context[
            "n"
        ],
        "k": context[
            "k"
        ],
        "run": int(
            run_index
        ),
        "tickers": context[
            "tickers"
        ],
        "n_parameters": context[
            "n_parameters"
        ],
        "initial_point": theta_initial,
        "best_parameters": theta_final,
        "objective_function_value": final_energy,
        "energy_gap": abs(
            final_energy
            - context[
                "exact_energy"
            ]
        ),
        "exact_energy": context[
            "exact_energy"
        ],
        "exact_bitstring": context[
            "exact_bitstring"
        ],
        "probability_best_answer": exact_probability,
        "most_frequent_bitstring": most_frequent,
        "is_exact_dominant": (
            most_frequent
            == context[
                "exact_bitstring"
            ]
        ),
        "nfev": len(
            local_callback
        ),
        "optimizer_time": float(
            optimizer_time
        ),
        "simulator_time": float(
            simulator_time
        ),
        "status": "ok",
    }


## 10. Geração opcional dos bancos

Esta é a parte cara do teste multi-$k$.

```python
generate_financial_banks = False
```

apenas constrói os contextos. Quando a flag é `True`, o notebook gera ou
completa os bancos e salva checkpoints após cada execução.

O teste de permutação possui bancos próprios e não depende de gerar os quatro
bancos multi-$k$.


In [ ]:

# ============================================================
# 38. CONSTRUIR CONTEXTOS
#
# A construção dos contextos é barata. A parte cara é executada
# somente quando generate_financial_banks=True.
# ============================================================
# 38. CONSTRUIR CONTEXTOS E, OPCIONALMENTE, GERAR OS BANCOS
# ============================================================

nk_contexts = {}

for n_value, k_value in nk_configs:
    print(
        f"construindo contexto n={n_value}, k={k_value}"
    )

    nk_contexts[
        (
            n_value,
            k_value,
        )
    ] = build_nk_problem_context(
        n_value=n_value,
        k_value=k_value,
    )


if generate_financial_banks:

    for (
        n_value,
        k_value,
    ), context in nk_contexts.items():

        config_dir = (
            financial_bank_dir
            / f"n{n_value}k{k_value}"
        )

        config_dir.mkdir(
            parents=True,
            exist_ok=True,
        )

        checkpoint_path = (
            config_dir
            / "bank.pkl"
        )

        if checkpoint_path.exists():
            config_runs = pd.read_pickle(
                checkpoint_path
            )

            if not isinstance(
                config_runs,
                list,
            ):
                config_runs = []
        else:
            config_runs = []

        completed_runs = {
            int(
                row[
                    "run"
                ]
            )
            for row in config_runs
            if row.get(
                "status"
            )
            == "ok"
        }

        for run_index in range(
            nk_runs_per_config
        ):
            if run_index in completed_runs:
                continue

            result_row = run_nk_cobyla(
                context=context,
                run_index=run_index,
            )

            result_row[
                "cobyla_maxiter"
            ] = nk_cobyla_maxiter

            result_row[
                "warmup_budget"
            ] = 0

            config_runs.append(
                result_row
            )

            pd.to_pickle(
                config_runs,
                checkpoint_path,
            )

            print(
                f"n={n_value}, k={k_value}",
                f"run {run_index + 1}/{nk_runs_per_config}",
                f"gap={result_row['energy_gap']:.6e}",
            )

        bank_df = pd.DataFrame(
            config_runs
        )

        bank_df.to_pickle(
            config_dir
            / "bank_df.pkl"
        )

else:
    print(
        "Geração desligada. "
        "Os contextos foram construídos, mas nenhum COBYLA foi executado."
    )


# Parte IV — previsão financeira e validação causal multi-$k$

## 11. Limite da evidência anterior

Para o caso original, a relação AMD/COST está bem sustentada. Isso ainda não
prova que exista uma regra geral.

A previsão desta parte é:

> o efeito causal deve acompanhar o par de qubits com decisões financeiras
> robustas e opostas, mesmo quando $k$ muda.

O desenho separa:

- **previsão:** derivada da função objetivo e do portfólio exato;
- **descoberta angular:** metade do banco;
- **intervenção causal:** outra metade;
- **veredito:** tamanho de efeito, controle positivo e qualidade do banco.


## 12. Mapa físico dos parâmetros

O ansatz é decomposto e cada parâmetro é ligado a:

- operação;
- qubits tocados;
- número de ocorrências;
- primeira e última instrução;
- período angular apropriado.

A célula exige que todos os parâmetros livres apareçam no mapa.


In [ ]:
# ============================================================
# 39. MAPA FÍSICO DOS PARÂMETROS APÓS A DECOMPOSIÇÃO
# ============================================================

def build_physical_parameter_map(
    ansatz,
):
    """
    Descobre em qual porta e em quais qubits cada theta aparece.

    CRY usa período 4*pi.
    RY usa período 2*pi.

    O mapa é feito após a decomposição, porque essa é a estrutura
    que realmente recebe os valores em assign_parameters.
    """
    parameter_order = list(
        ansatz.parameters
    )

    parameter_to_index = {
        parameter: index
        for index, parameter
        in enumerate(
            parameter_order
        )
    }

    occurrence_rows = []

    for instruction_position, instruction in enumerate(
        ansatz.data
    ):
        operation = (
            instruction.operation
        )

        operation_name = str(
            operation.name
        ).lower()

        qubits = tuple(
            int(
                ansatz.find_bit(
                    qubit
                ).index
            )
            for qubit
            in instruction.qubits
        )

        for parameter_slot, expression in enumerate(
            operation.params
        ):
            expression_parameters = getattr(
                expression,
                "parameters",
                set(),
            )

            for parameter in expression_parameters:
                if parameter not in parameter_to_index:
                    continue

                occurrence_rows.append(
                    {
                        "theta_index": int(
                            parameter_to_index[
                                parameter
                            ]
                        ),
                        "parameter_name": str(
                            parameter
                        ),
                        "operation": operation_name,
                        "qubits": qubits,
                        "instruction_position": int(
                            instruction_position
                        ),
                        "parameter_slot": int(
                            parameter_slot
                        ),
                    }
                )

    occurrence_df = pd.DataFrame(
        occurrence_rows
    )

    parameter_rows = []

    for theta_index, group in occurrence_df.groupby(
        "theta_index"
    ):
        operations = sorted(
            set(
                group[
                    "operation"
                ].astype(str)
            )
        )

        touched_qubits = sorted(
            {
                int(qubit)
                for qubit_tuple
                in group[
                    "qubits"
                ]
                for qubit
                in qubit_tuple
            }
        )

        if "cry" in operations:
            physical_type = "CRY"
            angular_period = float(
                4
                * np.pi
            )
        else:
            physical_type = "RY"
            angular_period = float(
                2
                * np.pi
            )

        parameter_rows.append(
            {
                "theta_index": int(
                    theta_index
                ),
                "parameter_name": str(
                    group[
                        "parameter_name"
                    ].iloc[0]
                ),
                "physical_type": physical_type,
                "physical_qubits": tuple(
                    touched_qubits
                ),
                "angular_period": angular_period,
                "n_occurrences": int(
                    len(
                        group
                    )
                ),
                "first_instruction": int(
                    group[
                        "instruction_position"
                    ].min()
                ),
                "last_instruction": int(
                    group[
                        "instruction_position"
                    ].max()
                ),
            }
        )

    parameter_map = (
        pd.DataFrame(
            parameter_rows
        )
        .sort_values(
            "theta_index"
        )
        .reset_index(
            drop=True
        )
    )

    # Toda variável livre do circuito precisa aparecer no mapa.
    expected_parameters = int(
        ansatz.num_parameters
    )

    mapped_parameters = int(
        parameter_map[
            "theta_index"
        ].nunique()
    )

    if mapped_parameters != expected_parameters:
        raise RuntimeError(
            "Mapa físico incompleto: "
            f"{mapped_parameters} de {expected_parameters} parâmetros."
        )

    allowed_types = {
        "CRY",
        "RY",
    }

    unexpected_types = set(
        parameter_map[
            "physical_type"
        ]
    ) - allowed_types

    if unexpected_types:
        raise RuntimeError(
            "Tipos físicos inesperados no mapa: "
            f"{sorted(unexpected_types)}"
        )

    return (
        parameter_map,
        occurrence_df,
    )


physical_maps = {}
physical_occurrences = {}

for configuration, context in nk_contexts.items():
    parameter_map, occurrence_map = (
        build_physical_parameter_map(
            context[
                "ansatz"
            ]
        )
    )

    physical_maps[
        configuration
    ] = parameter_map

    physical_occurrences[
        configuration
    ] = occurrence_map

    n_value, k_value = configuration

    parameter_map.to_csv(
        financial_table_dir
        / f"physical_map_n{n_value}k{k_value}.csv",
        index=False,
    )

print(
    "Mapas físicos construídos."
)


## 13. Margem de decisão financeira

Para cada ativo, a margem é a menor perda causada por uma troca viável que
inverte sua decisão no ótimo.

- ativo incluído: melhor carteira que o remove;
- ativo excluído: melhor carteira que o inclui.

Depois, o algoritmo avalia pares de qubits vizinhos e privilegia pares com
decisões opostas. Essa é uma previsão calculada antes da intervenção causal.


In [ ]:
# ============================================================
# 40. MARGEM FINANCEIRA DE CADA ATIVO
# ============================================================

def context_objective(
    context,
    x_binary,
):
    x_binary = np.asarray(
        x_binary,
        dtype=float,
    ).reshape(-1)

    mu = (
        context[
            "assets_returns"
        ]
        .to_numpy(
            dtype=float
        )
    )

    sigma = (
        context[
            "covariance"
        ]
        .to_numpy(
            dtype=float
        )
    )

    return float(
        q_value
        * x_binary
        @ sigma
        @ x_binary
        - (
            1
            - q_value
        )
        * mu
        @ x_binary
        + risk_free
    )


def enumerate_context_portfolios(
    context,
):
    n_value = int(
        context[
            "n"
        ]
    )

    k_value = int(
        context[
            "k"
        ]
    )

    rows = []

    for selected_indices in combinations(
        range(
            n_value
        ),
        k_value,
    ):
        x_binary = np.zeros(
            n_value,
            dtype=int,
        )

        x_binary[
            list(
                selected_indices
            )
        ] = 1

        rows.append(
            {
                "selected_indices": tuple(
                    selected_indices
                ),
                "x_binary": x_binary,
                "objective": context_objective(
                    context,
                    x_binary,
                ),
            }
        )

    return (
        pd.DataFrame(
            rows
        )
        .sort_values(
            "objective"
        )
        .reset_index(
            drop=True
        )
    )


def financial_decision_table(
    context,
):
    """
    Mede quão cara é a melhor troca que inverte a decisão de um ativo.

    Essa margem usa a função objetivo completa, incluindo retorno,
    variância, covariâncias e cardinalidade. Ela é mais adequada
    que interpretar somente um coeficiente linear.
    a decisão de um ativo.

    Margem grande:
        é caro retirar um ativo selecionado;
        ou é caro inserir um ativo excluído.

    Portanto, a decisão é menos ambígua.
    """
    enumeration = (
        enumerate_context_portfolios(
            context
        )
    )

    optimum = enumeration.iloc[
        0
    ]

    x_opt = np.asarray(
        optimum[
            "x_binary"
        ],
        dtype=int,
    )

    optimum_energy = float(
        optimum[
            "objective"
        ]
    )

    selected = np.flatnonzero(
        x_opt
        == 1
    )

    excluded = np.flatnonzero(
        x_opt
        == 0
    )

    mu = (
        context[
            "assets_returns"
        ]
        .to_numpy(
            dtype=float
        )
    )

    sigma = (
        context[
            "covariance"
        ]
        .to_numpy(
            dtype=float
        )
    )

    rows = []

    for asset_index in range(
        len(
            x_opt
        )
    ):
        swap_deltas = []

        if x_opt[
            asset_index
        ] == 1:
            for replacement in excluded:
                trial = x_opt.copy()
                trial[
                    asset_index
                ] = 0
                trial[
                    replacement
                ] = 1

                swap_deltas.append(
                    context_objective(
                        context,
                        trial,
                    )
                    - optimum_energy
                )
        else:
            for removed in selected:
                trial = x_opt.copy()
                trial[
                    asset_index
                ] = 1
                trial[
                    removed
                ] = 0

                swap_deltas.append(
                    context_objective(
                        context,
                        trial,
                    )
                    - optimum_energy
                )

        decision_margin = float(
            np.min(
                swap_deltas
            )
        )

        # Como partimos do ótimo global, nenhuma troca viável deveria
        # melhorar a energia. Pequenos valores negativos podem surgir
        # apenas por arredondamento.
        if decision_margin < -1e-10:
            raise RuntimeError(
                "A margem de decisão ficou negativa além da "
                f"tolerância no ativo {asset_index}: "
                f"{decision_margin:.3e}"
            )

        decision_margin = max(
            decision_margin,
            0.0,
        )

        rows.append(
            {
                "asset_index": int(
                    asset_index
                ),
                "ticker": context[
                    "tickers"
                ][
                    asset_index
                ],
                "selected_exact": int(
                    x_opt[
                        asset_index
                    ]
                ),
                "decision_margin": decision_margin,
                "return_coefficient": float(
                    -(
                        1
                        - q_value
                    )
                    * mu[
                        asset_index
                    ]
                ),
                "self_risk_coefficient": float(
                    q_value
                    * sigma[
                        asset_index,
                        asset_index,
                    ]
                ),
                "risk_connection": float(
                    q_value
                    * np.sum(
                        np.abs(
                            sigma[
                                asset_index
                            ]
                        )
                    )
                ),
            }
        )

    decision_df = pd.DataFrame(
        rows
    )

    pair_rows = []

    for left in range(
        len(
            x_opt
        )
        - 1
    ):
        right = left + 1

        opposite_decisions = bool(
            x_opt[
                left
            ]
            != x_opt[
                right
            ]
        )

        pair_rows.append(
            {
                "left_qubit": int(
                    left
                ),
                "right_qubit": int(
                    right
                ),
                "left_ticker": context[
                    "tickers"
                ][
                    left
                ],
                "right_ticker": context[
                    "tickers"
                ][
                    right
                ],
                "opposite_decisions": (
                    opposite_decisions
                ),
                "pair_margin_sum": float(
                    decision_df.loc[
                        decision_df[
                            "asset_index"
                        ]
                        .isin(
                            [
                                left,
                                right,
                            ]
                        ),
                        "decision_margin",
                    ].sum()
                ),
                "pair_margin_min": float(
                    decision_df.loc[
                        decision_df[
                            "asset_index"
                        ]
                        .isin(
                            [
                                left,
                                right,
                            ]
                        ),
                        "decision_margin",
                    ].min()
                ),
            }
        )

    pair_df = pd.DataFrame(
        pair_rows
    )

    opposite_pair_df = pair_df[
        pair_df[
            "opposite_decisions"
        ]
    ]

    if opposite_pair_df.empty:
        decisive_pair_row = (
            pair_df
            .sort_values(
                [
                    "pair_margin_min",
                    "pair_margin_sum",
                ],
                ascending=False,
            )
            .iloc[
                0
            ]
        )
    else:
        decisive_pair_row = (
            opposite_pair_df
            .sort_values(
                [
                    "pair_margin_min",
                    "pair_margin_sum",
                ],
                ascending=False,
            )
            .iloc[
                0
            ]
        )

    decisive_pair = (
        int(
            decisive_pair_row[
                "left_qubit"
            ]
        ),
        int(
            decisive_pair_row[
                "right_qubit"
            ]
        ),
    )

    return {
        "enumeration_df": enumeration,
        "x_opt": x_opt,
        "optimum_energy": optimum_energy,
        "decision_df": decision_df,
        "pair_df": pair_df,
        "decisive_pair": decisive_pair,
        "decisive_pair_row": decisive_pair_row,
    }


financial_context_analysis = {}

for configuration, context in nk_contexts.items():
    analysis = financial_decision_table(
        context
    )

    financial_context_analysis[
        configuration
    ] = analysis

    n_value, k_value = configuration

    analysis[
        "decision_df"
    ].to_csv(
        financial_table_dir
        / f"asset_decisions_n{n_value}k{k_value}.csv",
        index=False,
    )

    analysis[
        "pair_df"
    ].to_csv(
        financial_table_dir
        / f"adjacent_pairs_n{n_value}k{k_value}.csv",
        index=False,
    )

    pair = analysis[
        "decisive_pair"
    ]

    print(
        f"n={n_value}, k={k_value}:",
        "par previsto =",
        pair,
        context[
            "tickers"
        ][
            pair[
                0
            ]
        ],
        "/",
        context[
            "tickers"
        ][
            pair[
                1
            ]
        ],
    )


## 14. Da previsão financeira para as portas do circuito

Um parâmetro é previsto quando:

- controla a `CRY` que liga diretamente o par decisivo; ou
- controla uma `RY` local em um dos qubits do par.

Os demais parâmetros permanecem como comparação. O objetivo não é congelar os
outros theta, mas verificar onde a intervenção produz maior efeito.


In [ ]:
# ============================================================
# 41. PARÂMETROS QUE TOCAM O PAR FINANCEIRO PREVISTO
# ============================================================

# A previsão é feita por identidade física da porta/qubit,
# nunca pelo número arbitrário theta_index.
def annotate_parameter_financial_exposure(
    parameter_map,
    decision_df,
    decisive_pair,
):
    margin_lookup = dict(
        zip(
            decision_df[
                "asset_index"
            ].astype(int),
            decision_df[
                "decision_margin"
            ].astype(float),
        )
    )

    exact_decision_lookup = dict(
        zip(
            decision_df[
                "asset_index"
            ].astype(int),
            decision_df[
                "selected_exact"
            ].astype(int),
        )
    )

    rows = []

    decisive_pair_set = set(
        decisive_pair
    )

    for _, parameter_row in parameter_map.iterrows():
        touched_qubits = tuple(
            parameter_row[
                "physical_qubits"
            ]
        )

        touched_set = set(
            touched_qubits
        )

        touched_margins = [
            margin_lookup[
                int(qubit)
            ]
            for qubit
            in touched_qubits
        ]

        if (
            parameter_row[
                "physical_type"
            ]
            == "CRY"
            and touched_set
            == decisive_pair_set
        ):
            predicted_role = (
                "direct_edge"
            )
        elif (
            parameter_row[
                "physical_type"
            ]
            == "RY"
            and bool(
                touched_set
                & decisive_pair_set
            )
        ):
            predicted_role = (
                "local_on_pair"
            )
        else:
            predicted_role = (
                "other"
            )

        decisions_touched = {
            exact_decision_lookup[
                int(qubit)
            ]
            for qubit
            in touched_qubits
        }

        rows.append(
            {
                **parameter_row.to_dict(),
                "financial_exposure_mean": float(
                    np.mean(
                        touched_margins
                    )
                ),
                "financial_exposure_max": float(
                    np.max(
                        touched_margins
                    )
                ),
                "touches_opposite_decisions": bool(
                    len(
                        decisions_touched
                    )
                    > 1
                ),
                "predicted_role": predicted_role,
                "predicted_by_finance": bool(
                    predicted_role
                    != "other"
                ),
            }
        )

    return pd.DataFrame(
        rows
    )


parameter_hypothesis_tables = {}

for configuration, context in nk_contexts.items():
    parameter_table = (
        annotate_parameter_financial_exposure(
            parameter_map=physical_maps[
                configuration
            ],
            decision_df=financial_context_analysis[
                configuration
            ][
                "decision_df"
            ],
            decisive_pair=financial_context_analysis[
                configuration
            ][
                "decisive_pair"
            ],
        )
    )

    parameter_hypothesis_tables[
        configuration
    ] = parameter_table

    n_value, k_value = configuration

    parameter_table.to_csv(
        financial_table_dir
        / f"parameter_hypothesis_n{n_value}k{k_value}.csv",
        index=False,
    )

    print()
    print(
        f"Parâmetros previstos para n={n_value}, k={k_value}:"
    )

    display(
        parameter_table[
            parameter_table[
                "predicted_by_finance"
            ]
        ][
            [
                "theta_index",
                "physical_type",
                "physical_qubits",
                "predicted_role",
                "financial_exposure_mean",
            ]
        ]
    )


## 15. Carregamento e auditoria dos bancos multi-$k$

O nome da pasta não é suficiente para certificar um banco. A célula verifica nas
próprias linhas:

- comprimento dos vetores;
- ausência de valores não finitos;
- `cobyla_maxiter = 250`;
- `warmup_budget = 0`;
- pelo menos 100 vetores para evidência final.

Para executar somente o teste de permutação:

```python
run_multi_k_analysis = False
```

Nesse modo nenhum banco multi-$k$ é exigido.


In [ ]:
# ============================================================
# 42. CARREGAR BANCOS SEM MISTURAR VERSÕES
# ============================================================

def parse_theta_vector(
    value,
):
    """
    Converte diferentes representações de best_parameters
    em um vetor NumPy unidimensional.
    """
    if isinstance(
        value,
        np.ndarray,
    ):
        return np.asarray(
            value,
            dtype=float,
        ).reshape(-1)

    if isinstance(
        value,
        (
            list,
            tuple,
            pd.Series,
        ),
    ):
        return np.asarray(
            value,
            dtype=float,
        ).reshape(-1)

    if isinstance(
        value,
        str,
    ):
        try:
            parsed = json.loads(
                value
            )

        except Exception:
            parsed = ast.literal_eval(
                value
            )

        return np.asarray(
            parsed,
            dtype=float,
        ).reshape(-1)

    raise TypeError(
        f"Tipo inválido para theta: {type(value)}"
    )


def bank_candidates_for_configuration(
    n_value,
    k_value,
):
    """
    Prioridade:
    1. banco atual v19.2;
    2. bancos robustos da v18/v19;
    3. bancos legados de 150 avaliações, somente para depuração.
    """
    candidates = [
        (
            financial_bank_dir
            / f"n{n_value}k{k_value}"
            / "bank_df.pkl",
            "v19_2_b250",
            250,
        ),
        (
            output_dir
            / "financial_v18"
            / "bank_r100_b250"
            / f"n{n_value}k{k_value}"
            / "bank_df.pkl",
            "v18_b250",
            250,
        ),
    ]

    if allow_legacy_bank_fallback:
        candidates.extend(
            [
                (
                    output_dir
                    / "nk"
                    / "r100"
                    / f"n{n_value}k{k_value}"
                    / "bank_df.pkl",
                    "legacy_b150",
                    150,
                ),
                (
                    output_dir
                    / "nk"
                    / "r5"
                    / f"n{n_value}k{k_value}"
                    / "bank_df.pkl",
                    "legacy_pilot",
                    150,
                ),
            ]
        )

    return candidates


loaded_banks = {}
bank_metadata_rows = []

if not run_multi_k_analysis:
    print(
        "Análise multi-k desativada. "
        "Nenhum banco multi-k será exigido."
    )

    bank_metadata_df = pd.DataFrame(
        columns=[
            "n",
            "k",
            "runs",
            "bank_source",
            "cobyla_budget",
            "budget_metadata_ok",
            "warmup_metadata_ok",
            "bank_path",
            "final_evidence",
        ]
    )

else:
    for (
        n_value,
        k_value,
    ), context in nk_contexts.items():

        selected_path = None
        selected_source = None
        selected_budget = None

        for (
            candidate_path,
            source_label,
            budget_label,
        ) in bank_candidates_for_configuration(
            n_value,
            k_value,
        ):
            if candidate_path.exists():
                selected_path = candidate_path
                selected_source = source_label
                selected_budget = budget_label
                break

        if selected_path is None:
            raise FileNotFoundError(
                f"Nenhum banco encontrado para n={n_value}, k={k_value}.\n"
                "Opções:\n"
                "  1. generate_financial_banks = True\n"
                "  2. run_multi_k_analysis = False para rodar somente "
                "o teste de permutação."
            )

        bank_df = pd.read_pickle(
            selected_path
        ).copy()

        required_columns = {
            "best_parameters",
        }

        missing_columns = (
            required_columns
            - set(
                bank_df.columns
            )
        )

        if missing_columns:
            raise KeyError(
                f"Banco {selected_path} sem colunas: "
                f"{sorted(missing_columns)}"
            )

        bank_df[
            "best_parameters"
        ] = bank_df[
            "best_parameters"
        ].map(
            parse_theta_vector
        )

        valid_length = (
            bank_df[
                "best_parameters"
            ].map(
                len
            )
            == int(
                context[
                    "n_parameters"
                ]
            )
        )

        finite_theta = bank_df[
            "best_parameters"
        ].map(
            lambda theta: bool(
                np.all(
                    np.isfinite(
                        theta
                    )
                )
            )
        )

        bank_df = (
            bank_df.loc[
                valid_length
                & finite_theta
            ]
            .reset_index(
                drop=True
            )
        )

        if bank_df.empty:
            raise ValueError(
                f"O banco {selected_path} não contém theta válidos."
            )

        # O status de evidência final é calculado a partir dos
        # metadados das linhas, não apenas do nome da pasta.
        if "cobyla_maxiter" in bank_df.columns:
            budget_metadata_ok = bool(
                pd.to_numeric(
                    bank_df[
                        "cobyla_maxiter"
                    ],
                    errors="coerce",
                ).eq(
                    250
                ).all()
            )

        else:
            budget_metadata_ok = False

        if "warmup_budget" in bank_df.columns:
            warmup_metadata_ok = bool(
                pd.to_numeric(
                    bank_df[
                        "warmup_budget"
                    ],
                    errors="coerce",
                ).eq(
                    0
                ).all()
            )

        else:
            warmup_metadata_ok = False

        loaded_banks[
            (
                n_value,
                k_value,
            )
        ] = bank_df

        final_evidence = bool(
            len(
                bank_df
            )
            >= 100
            and selected_budget
            == 250
            and budget_metadata_ok
            and warmup_metadata_ok
        )

        bank_metadata_rows.append(
            {
                "n": int(
                    n_value
                ),
                "k": int(
                    k_value
                ),
                "runs": int(
                    len(
                        bank_df
                    )
                ),
                "bank_source": selected_source,
                "cobyla_budget": int(
                    selected_budget
                ),
                "budget_metadata_ok": (
                    budget_metadata_ok
                ),
                "warmup_metadata_ok": (
                    warmup_metadata_ok
                ),
                "bank_path": str(
                    selected_path.resolve()
                ),
                "final_evidence": (
                    final_evidence
                ),
            }
        )

    bank_metadata_df = pd.DataFrame(
        bank_metadata_rows
    )

    bank_metadata_df.to_csv(
        financial_table_dir
        / "bank_metadata.csv",
        index=False,
    )

    display(
        bank_metadata_df
    )


## 16. Reavaliação exata dos vetores

Rótulos salvos com 4096 shots podem confundir bitstrings próximos. Portanto,
energia, probabilidade ótima e bitstring dominante são recalculados com
`Statevector`.

Quando houver mais de um bitstring ótimo, as probabilidades são somadas.


In [ ]:
# ============================================================
# 43. REAVALIAR CADA THETA COM STATEVECTOR
# ============================================================

def exact_theta_metrics(
    context,
    theta,
):
    """
    Avalia energia e probabilidade sem shots.

    Quando houver degenerescência, p_exact_eval é a soma das
    probabilidades de todos os bitstrings ótimos.
    """
    theta = np.asarray(
        theta,
        dtype=float,
    ).reshape(-1)

    expected_length = int(
        context[
            "n_parameters"
        ]
    )

    if len(
        theta
    ) != expected_length:
        raise ValueError(
            f"Vetor theta com {len(theta)} elementos; "
            f"esperados {expected_length}."
        )

    assigned = (
        context[
            "ansatz"
        ]
        .assign_parameters(
            theta,
            inplace=False,
        )
    )

    state = Statevector.from_instruction(
        assigned
    )

    energy = float(
        np.real(
            state.expectation_value(
                context[
                    "ising"
                ]
            )
        )
        + context[
            "offset"
        ]
    )

    probabilities_raw = (
        state.probabilities_dict()
    )

    probabilities = {
        str(
            key
        ).replace(
            " ",
            "",
        ): float(
            value
        )
        for key, value
        in probabilities_raw.items()
    }

    optimal_bitstrings = context.get(
        "exact_bitstrings",
        [
            context[
                "exact_bitstring"
            ]
        ],
    )

    exact_probability = float(
        sum(
            probabilities.get(
                bitstring,
                0.0,
            )
            for bitstring
            in optimal_bitstrings
        )
    )

    dominant_bitstring = max(
        probabilities,
        key=probabilities.get,
    )

    return {
        "energy_exact_eval": energy,
        "gap_exact_eval": abs(
            energy
            - context[
                "exact_energy"
            ]
        ),
        "p_exact_eval": exact_probability,
        "dominant_bitstring_exact_eval": (
            dominant_bitstring
        ),
        "is_exact_dominant_eval": bool(
            dominant_bitstring
            in set(
                optimal_bitstrings
            )
        ),
    }


exact_banks = {}

if not run_multi_k_analysis:
    print(
        "Reavaliação multi-k ignorada porque "
        "run_multi_k_analysis=False."
    )

else:
    for configuration, context in nk_contexts.items():
        bank_df = (
            loaded_banks[
                configuration
            ].copy()
        )

        metric_rows = [
            exact_theta_metrics(
                context,
                theta,
            )
            for theta
            in bank_df[
                "best_parameters"
            ]
        ]

        metric_df = pd.DataFrame(
            metric_rows
        )

        exact_bank = pd.concat(
            [
                bank_df.reset_index(
                    drop=True
                ),
                metric_df,
            ],
            axis=1,
        )

        exact_banks[
            configuration
        ] = exact_bank

        n_value, k_value = configuration

        exact_bank.to_pickle(
            financial_table_dir
            / f"exact_bank_n{n_value}k{k_value}.pkl"
        )

        print(
            f"n={n_value}, k={k_value}:",
            f"gap mediano={exact_bank['gap_exact_eval'].median():.6e}",
            f"taxa exata="
            f"{100 * exact_bank['is_exact_dominant_eval'].mean():.2f}%",
        )


## 17. Descoberta angular sem vazamento

O banco é dividido aleatoriamente, mas de forma reprodutível:

- metade de descoberta;
- metade de validação.

A região “boa” e a região “ruim” de cada theta são definidas somente na metade
de descoberta. A metade de validação nunca participa dessa escolha.


In [ ]:
# ============================================================
# 44. DESCOBRIR REGIÕES BOAS E RUINS EM UMA METADE DO BANCO
# ============================================================

# Cada porta é reduzida ao seu próprio intervalo físico.
# CRY e RY não recebem necessariamente o mesmo período.
def canonical_angle(
    angle,
    period,
):
    return (
        float(
            angle
        )
        + period / 2
    ) % period - period / 2


def discover_parameter_regions(
    discovery_df,
    parameter_table,
):
    rows = []

    for _, parameter_row in parameter_table.iterrows():
        theta_index = int(
            parameter_row[
                "theta_index"
            ]
        )

        period = float(
            parameter_row[
                "angular_period"
            ]
        )

        angles = np.asarray(
            [
                canonical_angle(
                    theta[
                        theta_index
                    ],
                    period,
                )
                for theta
                in discovery_df[
                    "best_parameters"
                ]
            ],
            dtype=float,
        )

        bin_edges = np.linspace(
            -period / 2,
            period / 2,
            financial_angle_bins + 1,
        )

        bin_ids = np.digitize(
            angles,
            bin_edges[
                1:-1
            ],
            right=False,
        )

        local_rows = []

        for bin_id in range(
            financial_angle_bins
        ):
            mask = (
                bin_ids
                == bin_id
            )

            count = int(
                mask.sum()
            )

            if (
                count
                < financial_min_runs_per_bin
            ):
                continue

            local_rows.append(
                {
                    "bin_id": int(
                        bin_id
                    ),
                    "count": count,
                    "angle_center": float(
                        0.5
                        * (
                            bin_edges[
                                bin_id
                            ]
                            + bin_edges[
                                bin_id + 1
                            ]
                        )
                    ),
                    "gap_median": float(
                        discovery_df.loc[
                            mask,
                            "gap_exact_eval",
                        ].median()
                    ),
                    "p_exact_mean": float(
                        discovery_df.loc[
                            mask,
                            "p_exact_eval",
                        ].mean()
                    ),
                }
            )

        local_df = pd.DataFrame(
            local_rows
        )

        if len(
            local_df
        ) < 2:
            rows.append(
                {
                    "theta_index": theta_index,
                    "region_valid": False,
                    "good_angle": np.nan,
                    "bad_angle": np.nan,
                    "valid_bins": int(
                        len(
                            local_df
                        )
                    ),
                }
            )

            continue

        gap_min = float(
            local_df[
                "gap_median"
            ].min()
        )

        gap_max = float(
            local_df[
                "gap_median"
            ].max()
        )

        p_min = float(
            local_df[
                "p_exact_mean"
            ].min()
        )

        p_max = float(
            local_df[
                "p_exact_mean"
            ].max()
        )

        if gap_max > gap_min:
            local_df[
                "gap_normalized"
            ] = (
                local_df[
                    "gap_median"
                ]
                - gap_min
            ) / (
                gap_max
                - gap_min
            )
        else:
            local_df[
                "gap_normalized"
            ] = 0.0

        if p_max > p_min:
            local_df[
                "p_normalized"
            ] = (
                local_df[
                    "p_exact_mean"
                ]
                - p_min
            ) / (
                p_max
                - p_min
            )
        else:
            local_df[
                "p_normalized"
            ] = 0.0

        local_df[
            "quality_score"
        ] = (
            local_df[
                "p_normalized"
            ]
            - local_df[
                "gap_normalized"
            ]
        )

        good_row = local_df.sort_values(
            "quality_score",
            ascending=False,
        ).iloc[
            0
        ]

        bad_row = local_df.sort_values(
            "quality_score",
            ascending=True,
        ).iloc[
            0
        ]

        region_valid = bool(
            int(
                good_row[
                    "bin_id"
                ]
            )
            != int(
                bad_row[
                    "bin_id"
                ]
            )
        )

        rows.append(
            {
                "theta_index": theta_index,
                "region_valid": region_valid,
                "good_angle": float(
                    good_row[
                        "angle_center"
                    ]
                ),
                "bad_angle": float(
                    bad_row[
                        "angle_center"
                    ]
                ),
                "valid_bins": int(
                    len(
                        local_df
                    )
                ),
                "good_gap_median": float(
                    good_row[
                        "gap_median"
                    ]
                ),
                "bad_gap_median": float(
                    bad_row[
                        "gap_median"
                    ]
                ),
                "good_p_exact_mean": float(
                    good_row[
                        "p_exact_mean"
                    ]
                ),
                "bad_p_exact_mean": float(
                    bad_row[
                        "p_exact_mean"
                    ]
                ),
            }
        )

    return pd.DataFrame(
        rows
    )

if run_multi_k_analysis:
    split_banks = {}
    parameter_regions = {}

    for configuration, exact_bank in exact_banks.items():
        n_value, k_value = configuration

        local_rng = np.random.default_rng(
            random_seed
            + 4_000_000
            + 100
            * n_value
            + k_value
        )

        permutation = local_rng.permutation(
            len(
                exact_bank
            )
        )

        discovery_size = int(
            np.floor(
                financial_discovery_fraction
                * len(
                    exact_bank
                )
            )
        )

        discovery_indices = permutation[
            :discovery_size
        ]

        validation_indices = permutation[
            discovery_size:
        ]

        discovery_df = (
            exact_bank.iloc[
                discovery_indices
            ]
            .copy()
            .reset_index(
                drop=True
            )
        )

        validation_df = (
            exact_bank.iloc[
                validation_indices
            ]
            .copy()
            .reset_index(
                drop=True
            )
        )

        split_banks[
            configuration
        ] = {
            "discovery": discovery_df,
            "validation": validation_df,
        }

        regions = discover_parameter_regions(
            discovery_df=discovery_df,
            parameter_table=parameter_hypothesis_tables[
                configuration
            ],
        )

        parameter_regions[
            configuration
        ] = regions

        regions.to_csv(
            financial_table_dir
            / f"regions_n{n_value}k{k_value}.csv",
            index=False,
        )

        print(
            f"n={n_value}, k={k_value}:",
            f"descoberta={len(discovery_df)}",
            f"validação={len(validation_df)}",
            f"regiões válidas={int(regions['region_valid'].sum())}",
        )
else:

    split_banks = {}
    parameter_regions = {}

    print(
        "Descoberta angular multi-k ignorada."
    )


## 18. Intervenções de necessidade e suficiência

### Necessidade

Parte de vetores bons, altera um theta para a região ruim e mede:

$$
\Delta E_{\mathrm{nec}}
=
E_{\mathrm{depois}}-E_{\mathrm{antes}},
$$

$$
\Delta P_{\mathrm{nec}}
=
P^\star_{\mathrm{depois}}-P^\star_{\mathrm{antes}}.
$$

Espera-se $\Delta E_{\mathrm{nec}}>0$ e
$\Delta P_{\mathrm{nec}}<0$.

### Suficiência

Parte de vetores ruins e move um theta para a região boa. Espera-se
$\Delta E_{\mathrm{suf}}<0$ e $\Delta P_{\mathrm{suf}}>0$.

Os outros parâmetros permanecem fixos.


In [ ]:
# ============================================================
# 45. INTERVENÇÃO CAUSAL EM TODOS OS PARÂMETROS
# ============================================================

# O p-valor é complementar. A decisão de relevância será
# baseada principalmente no tamanho do efeito.
def safe_wilcoxon(
    values,
    alternative,
):
    values = np.asarray(
        values,
        dtype=float,
    )

    values = values[
        np.isfinite(
            values
        )
    ]

    if (
        len(
            values
        )
        == 0
        or np.allclose(
            values,
            0.0,
        )
    ):
        return np.nan

    try:
        return float(
            wilcoxon(
                values,
                alternative=alternative,
                zero_method="wilcox",
            ).pvalue
        )
    except ValueError:
        return np.nan


def choose_validation_extremes(
    validation_df,
):
    ordered_good = validation_df.sort_values(
        [
            "gap_exact_eval",
            "p_exact_eval",
        ],
        ascending=[
            True,
            False,
        ],
    )

    ordered_bad = validation_df.sort_values(
        [
            "gap_exact_eval",
            "p_exact_eval",
        ],
        ascending=[
            False,
            True,
        ],
    )

    maximum_without_overlap = max(
        1,
        len(
            validation_df
        )
        // 2,
    )

    n_selected = min(
        int(
            financial_n_test_vectors
        ),
        maximum_without_overlap,
    )

    good_df = (
        ordered_good.head(
            n_selected
        )
        .copy()
        .reset_index(
            drop=True
        )
    )

    bad_df = (
        ordered_bad.head(
            n_selected
        )
        .copy()
        .reset_index(
            drop=True
        )
    )

    return (
        good_df,
        bad_df,
    )


def intervene_one_parameter(
    context,
    theta,
    theta_index,
    new_angle,
):
    modified = np.asarray(
        theta,
        dtype=float,
    ).copy()

    modified[
        int(
            theta_index
        )
    ] = float(
        new_angle
    )

    return exact_theta_metrics(
        context,
        modified,
    )

if run_multi_k_analysis:
    causal_rows = []

    for configuration, context in nk_contexts.items():
        n_value, k_value = configuration

        validation_df = split_banks[
            configuration
        ][
            "validation"
        ]

        good_df, bad_df = (
            choose_validation_extremes(
                validation_df
            )
        )

        region_table = (
            parameter_regions[
                configuration
            ]
        )

        hypothesis_table = (
            parameter_hypothesis_tables[
                configuration
            ]
        )

        merged_parameter_table = (
            hypothesis_table.merge(
                region_table,
                on="theta_index",
                how="left",
                validate="one_to_one",
            )
        )

        for _, parameter_row in merged_parameter_table.iterrows():
            if not bool(
                parameter_row.get(
                    "region_valid",
                    False,
                )
            ):
                continue

            theta_index = int(
                parameter_row[
                    "theta_index"
                ]
            )

            good_angle = float(
                parameter_row[
                    "good_angle"
                ]
            )

            bad_angle = float(
                parameter_row[
                    "bad_angle"
                ]
            )

            necessity_delta_energy = []
            necessity_delta_probability = []

            for _, base_row in good_df.iterrows():
                after = intervene_one_parameter(
                    context=context,
                    theta=base_row[
                        "best_parameters"
                    ],
                    theta_index=theta_index,
                    new_angle=bad_angle,
                )

                necessity_delta_energy.append(
                    after[
                        "energy_exact_eval"
                    ]
                    - float(
                        base_row[
                            "energy_exact_eval"
                        ]
                    )
                )

                necessity_delta_probability.append(
                    after[
                        "p_exact_eval"
                    ]
                    - float(
                        base_row[
                            "p_exact_eval"
                        ]
                    )
                )

            sufficiency_delta_energy = []
            sufficiency_delta_probability = []

            for _, base_row in bad_df.iterrows():
                after = intervene_one_parameter(
                    context=context,
                    theta=base_row[
                        "best_parameters"
                    ],
                    theta_index=theta_index,
                    new_angle=good_angle,
                )

                sufficiency_delta_energy.append(
                    after[
                        "energy_exact_eval"
                    ]
                    - float(
                        base_row[
                            "energy_exact_eval"
                        ]
                    )
                )

                sufficiency_delta_probability.append(
                    after[
                        "p_exact_eval"
                    ]
                    - float(
                        base_row[
                            "p_exact_eval"
                        ]
                    )
                )

            causal_rows.append(
                {
                    "n": int(
                        n_value
                    ),
                    "k": int(
                        k_value
                    ),
                    "theta_index": theta_index,
                    "physical_type": parameter_row[
                        "physical_type"
                    ],
                    "physical_qubits": parameter_row[
                        "physical_qubits"
                    ],
                    "predicted_role": parameter_row[
                        "predicted_role"
                    ],
                    "predicted_by_finance": bool(
                        parameter_row[
                            "predicted_by_finance"
                        ]
                    ),
                    "financial_exposure_mean": float(
                        parameter_row[
                            "financial_exposure_mean"
                        ]
                    ),
                    "necessity_delta_energy_median": float(
                        np.median(
                            necessity_delta_energy
                        )
                    ),
                    "necessity_delta_probability_median": float(
                        np.median(
                            necessity_delta_probability
                        )
                    ),
                    "necessity_energy_p": safe_wilcoxon(
                        necessity_delta_energy,
                        alternative="greater",
                    ),
                    "necessity_probability_p": safe_wilcoxon(
                        necessity_delta_probability,
                        alternative="less",
                    ),
                    "sufficiency_delta_energy_median": float(
                        np.median(
                            sufficiency_delta_energy
                        )
                    ),
                    "sufficiency_delta_probability_median": float(
                        np.median(
                            sufficiency_delta_probability
                        )
                    ),
                    "sufficiency_energy_p": safe_wilcoxon(
                        sufficiency_delta_energy,
                        alternative="less",
                    ),
                    "sufficiency_probability_p": safe_wilcoxon(
                        sufficiency_delta_probability,
                        alternative="greater",
                    ),
                    "n_necessity": int(
                        len(
                            necessity_delta_energy
                        )
                    ),
                    "n_sufficiency": int(
                        len(
                            sufficiency_delta_energy
                        )
                    ),
                }
            )


    causal_effects_df = pd.DataFrame(
        causal_rows
    )

    causal_effects_df.to_csv(
        financial_table_dir
        / "causal_effects_all_k.csv",
        index=False,
    )

    print(
        "Intervenções concluídas:",
        len(
            causal_effects_df
        ),
    )
else:

    causal_rows = []

    causal_effects_df = pd.DataFrame()

    print(
        "Intervenções multi-k ignoradas."
    )


## 19. Score causal baseado em magnitude

A versão auditada não normaliza o maior efeito de cada tabela automaticamente
para 1. Esse procedimento poderia promover ruído quando todos os efeitos fossem
pequenos.

O score usa magnitudes absolutas, com energia dividida pelo intervalo clássico
da configuração:

$$
S
=
\frac14
\left[
\frac{\max(\Delta E_{\rm nec},0)}{\Delta E_{\rm span}}
+
\max(-\Delta P_{\rm nec},0)
+
\frac{\max(-\Delta E_{\rm suf},0)}{\Delta E_{\rm span}}
+
\max(\Delta P_{\rm suf},0)
\right].
$$

O p-valor é reportado, mas não substitui o tamanho de efeito.


In [ ]:
# ============================================================
# 46. RANKING CAUSAL E TESTE DA HIPÓTESE FINANCEIRA
# ============================================================

def comparable_causal_score(
    row,
    energy_span,
):
    """
    Score baseado em tamanho de efeito absoluto.

    Componentes:
    - necessidade em energia;
    - necessidade em probabilidade;
    - suficiência em energia;
    - suficiência em probabilidade.

    As energias são divididas pelo span clássico da configuração.
    Assim, o score não transforma automaticamente o maior ruído
    de cada tabela em valor 1.
    """
    energy_span = max(
        float(
            energy_span
        ),
        1e-15,
    )

    return float(
        0.25
        * (
            max(
                float(
                    row[
                        "necessity_delta_energy_median"
                    ]
                ),
                0.0,
            )
            / energy_span
            + max(
                -float(
                    row[
                        "necessity_delta_probability_median"
                    ]
                ),
                0.0,
            )
            + max(
                -float(
                    row[
                        "sufficiency_delta_energy_median"
                    ]
                ),
                0.0,
            )
            / energy_span
            + max(
                float(
                    row[
                        "sufficiency_delta_probability_median"
                    ]
                ),
                0.0,
            )
        )
    )


ranked_causal_tables = {}
hypothesis_summary_rows = []

if not run_multi_k_analysis:
    hypothesis_summary_df = pd.DataFrame()

    print(
        "Ranking causal multi-k ignorado."
    )

else:
    for configuration in nk_configs:
        n_value, k_value = configuration

        local_df = causal_effects_df[
            (
                causal_effects_df[
                    "n"
                ]
                == n_value
            )
            & (
                causal_effects_df[
                    "k"
                ]
                == k_value
            )
        ].copy()

        if local_df.empty:
            warnings.warn(
                f"n={n_value}, k={k_value}: "
                "nenhum efeito causal válido."
            )

            continue

        enumeration_values = (
            financial_context_analysis[
                configuration
            ][
                "enumeration_df"
            ][
                "objective"
            ]
            .to_numpy(
                dtype=float
            )
        )

        energy_span = float(
            np.max(
                enumeration_values
            )
            - np.min(
                enumeration_values
            )
        )

        local_df[
            "causal_score"
        ] = local_df.apply(
            lambda row: comparable_causal_score(
                row,
                energy_span,
            ),
            axis=1,
        )

        local_df[
            "causal_effect_relevant"
        ] = (
            local_df[
                "causal_score"
            ]
            >= minimum_causal_score
        )

        local_df[
            "causal_rank"
        ] = (
            local_df[
                "causal_score"
            ]
            .rank(
                method="min",
                ascending=False,
            )
            .astype(int)
        )

        local_df = local_df.sort_values(
            [
                "causal_rank",
                "theta_index",
            ]
        ).reset_index(
            drop=True
        )

        ranked_causal_tables[
            configuration
        ] = local_df

        local_df.to_csv(
            financial_table_dir
            / f"causal_ranking_n{n_value}k{k_value}.csv",
            index=False,
        )

        valid_correlation = local_df[
            [
                "financial_exposure_mean",
                "causal_score",
            ]
        ].dropna()

        if (
            len(
                valid_correlation
            )
            >= 3
            and valid_correlation[
                "financial_exposure_mean"
            ].nunique()
            > 1
            and valid_correlation[
                "causal_score"
            ].nunique()
            > 1
        ):
            correlation_result = spearmanr(
                valid_correlation[
                    "financial_exposure_mean"
                ],
                valid_correlation[
                    "causal_score"
                ],
            )

            spearman_rho = float(
                correlation_result.statistic
            )

            spearman_p = float(
                correlation_result.pvalue
            )

        else:
            spearman_rho = np.nan
            spearman_p = np.nan

        relevant_df = local_df[
            local_df[
                "causal_effect_relevant"
            ]
        ]

        signal_present = bool(
            not relevant_df.empty
        )

        top_count = min(
            3,
            len(
                relevant_df
            ),
        )

        top_causal = (
            relevant_df.head(
                top_count
            )
        )

        if top_causal.empty:
            top3_hit_rate = np.nan
            top_causal_thetas = ""

        else:
            top3_hit_rate = float(
                100
                * top_causal[
                    "predicted_by_finance"
                ].mean()
            )

            top_causal_thetas = ",".join(
                str(
                    int(
                        value
                    )
                )
                for value
                in top_causal[
                    "theta_index"
                ]
            )

        predicted_parameters = local_df[
            local_df[
                "predicted_by_finance"
            ]
        ]

        top_quartile_limit = max(
            1,
            int(
                np.ceil(
                    len(
                        local_df
                    )
                    / 4
                )
            ),
        )

        predicted_top_quartile_rate = (
            float(
                100
                * (
                    predicted_parameters[
                        "causal_rank"
                    ]
                    <= top_quartile_limit
                ).mean()
            )
            if not predicted_parameters.empty
            else np.nan
        )

        decisive_pair = (
            financial_context_analysis[
                configuration
            ][
                "decisive_pair"
            ]
        )

        tickers_local = (
            nk_contexts[
                configuration
            ][
                "tickers"
            ]
        )

        metadata_row = bank_metadata_df[
            (
                bank_metadata_df[
                    "n"
                ]
                == n_value
            )
            & (
                bank_metadata_df[
                    "k"
                ]
                == k_value
            )
        ].iloc[
            0
        ]

        hypothesis_summary_rows.append(
            {
                "n": int(
                    n_value
                ),
                "k": int(
                    k_value
                ),
                "decisive_pair": str(
                    decisive_pair
                ),
                "decisive_tickers": (
                    f"{tickers_local[decisive_pair[0]]}"
                    f"/{tickers_local[decisive_pair[1]]}"
                ),
                "predicted_parameter_count": int(
                    predicted_parameters.shape[
                        0
                    ]
                ),
                "causal_signal_present": (
                    signal_present
                ),
                "max_causal_score": float(
                    local_df[
                        "causal_score"
                    ].max()
                ),
                "top_causal_thetas": (
                    top_causal_thetas
                ),
                "top3_hit_rate_pct": (
                    top3_hit_rate
                ),
                "predicted_top_quartile_rate_pct": (
                    predicted_top_quartile_rate
                ),
                "spearman_finance_causal": (
                    spearman_rho
                ),
                "spearman_p": (
                    spearman_p
                ),
                "bank_source": metadata_row[
                    "bank_source"
                ],
                "cobyla_budget": int(
                    metadata_row[
                        "cobyla_budget"
                    ]
                ),
                "bank_runs": int(
                    metadata_row[
                        "runs"
                    ]
                ),
                "final_evidence": bool(
                    metadata_row[
                        "final_evidence"
                    ]
                ),
            }
        )

    hypothesis_summary_df = pd.DataFrame(
        hypothesis_summary_rows
    )

    hypothesis_summary_df.to_csv(
        financial_table_dir
        / "hypothesis_summary.csv",
        index=False,
    )

    display(
        hypothesis_summary_df
    )


## 20. Controle positivo e veredito multi-$k$

Antes de generalizar, o notebook exige que o caso $k=4$ recupere ao menos dois
dos três parâmetros históricos no Top-5.

O veredito final também exige:

- quatro bancos robustos;
- metadados corretos;
- sinal causal acima do limiar em todas as configurações;
- controle positivo aprovado.

Caso contrário, o resultado permanece preliminar.


In [ ]:
# ============================================================
# 47. CONTROLE POSITIVO DO CASO ORIGINAL E VEREDITO MULTI-K
# ============================================================

k4_sanity_row = None

if not run_multi_k_analysis:
    final_evidence_available = False
    final_verdict = (
        "SKIPPED_MULTI_K_ANALYSIS"
    )

else:
    if (
        10,
        4,
    ) in ranked_causal_tables:
        k4_table = ranked_causal_tables[
            (
                10,
                4,
            )
        ]

        expected_original = {
            2,
            14,
            17,
        }

        top5_original = set(
            k4_table.head(
                5
            )[
                "theta_index"
            ].astype(int)
        )

        recovered = (
            expected_original
            & top5_original
        )

        k4_sanity_row = {
            "expected_original": sorted(
                expected_original
            ),
            "top5_observed": sorted(
                top5_original
            ),
            "expected_recovered_in_top5": sorted(
                recovered
            ),
            "sanity_pass": bool(
                len(
                    recovered
                )
                >= 2
            ),
        }

        display(
            pd.DataFrame(
                [
                    k4_sanity_row
                ]
            )
        )

    final_evidence_available = bool(
        not hypothesis_summary_df.empty
        and hypothesis_summary_df[
            "final_evidence"
        ].all()
    )

    all_configs_have_signal = bool(
        not hypothesis_summary_df.empty
        and hypothesis_summary_df[
            "causal_signal_present"
        ].all()
    )

    k4_sanity_pass = bool(
        k4_sanity_row is not None
        and k4_sanity_row[
            "sanity_pass"
        ]
    )

    if (
        final_evidence_available
        and all_configs_have_signal
        and k4_sanity_pass
    ):
        median_top3_hit = float(
            hypothesis_summary_df[
                "top3_hit_rate_pct"
            ].median(
                skipna=True
            )
        )

        median_quartile_hit = float(
            hypothesis_summary_df[
                "predicted_top_quartile_rate_pct"
            ].median(
                skipna=True
            )
        )

        median_correlation = float(
            hypothesis_summary_df[
                "spearman_finance_causal"
            ].median(
                skipna=True
            )
        )

        if (
            median_top3_hit >= 66.0
            and median_quartile_hit >= 50.0
            and median_correlation > 0.30
        ):
            final_verdict = (
                "SUPPORTED_ACROSS_K"
            )

        elif (
            median_top3_hit <= 33.0
            and median_correlation <= 0.10
        ):
            final_verdict = (
                "NOT_SUPPORTED_ACROSS_K"
            )

        else:
            final_verdict = (
                "MIXED_EVIDENCE"
            )

    else:
        final_verdict = (
            "PRELIMINARY_OR_FAILED_POSITIVE_CONTROL"
        )


verdict_df = pd.DataFrame(
    [
        {
            "verdict": final_verdict,
            "all_banks_b250_r100": (
                final_evidence_available
            ),
            "k4_sanity_pass": (
                None
                if k4_sanity_row is None
                else k4_sanity_row[
                    "sanity_pass"
                ]
            ),
        }
    ]
)

verdict_df.to_csv(
    financial_table_dir
    / "final_verdict.csv",
    index=False,
)

display(
    verdict_df
)


# Parte V — controle de permutação da ordem dos ativos

## 21. O teste que separa finanças de coincidência topológica

Embaralhar as colunas altera qual ativo ocupa cada qubit, mas não altera as
séries financeiras. Portanto, devem permanecer invariantes:

- o conjunto das 210 energias;
- a energia ótima;
- o conjunto físico de ativos selecionados.

O bitstring muda porque sua representação depende da nova ordem.

### Desenho

Para cada ordem aleatória:

1. cria uma cópia profunda do DataFrame;
2. salva uma cópia física do CSV;
3. relê a cópia;
4. reconstrói CVXPY, DOcplex, QUBO, Ising e solução exata;
5. encontra o melhor par vizinho;
6. executa os mesmos pontos iniciais COBYLA;
7. separa descoberta e validação;
8. testa parâmetros previstos e controles do mesmo tipo;
9. correlaciona margem financeira com efeito causal.

Uma ordenação deliberada coloca a inclusão e a exclusão mais robustas lado a
lado. Ela é comparada com a distribuição aleatória, mas não entra na correlação
principal.

### Custo

Modo piloto:

```text
3 ordens aleatórias + 1 deliberada
14 restarts por ordem
```

Modo robusto:

```text
15 ordens aleatórias + 1 deliberada
40 restarts por ordem
250 avaliações por restart
```


## Correção da ordem deliberada — versão 19.4

A versão 19.2 não continha um erro aritmético em `insertion_position`. Ela
executava corretamente o que estava programado: escolhia automaticamente

```text
ativo incluído com maior margem
+
ativo excluído com maior margem
```

No problema atual, esse critério produziu o par **DAL/AMD**. Portanto, embora a
documentação mencionasse AMD/COST, o controle específico AMD/COST não havia sido
executado.

A versão 19.4 transforma a hipótese em uma condição explícita:

```python
shuffle_deliberate_pair = ("AMD", "COST")
```

Depois de construir a ordem, o código verifica:

```python
abs(posicao_AMD - posicao_COST) == 1
```

e interrompe o teste se a condição não for atendida.

### Reutilização dos resultados anteriores

As 15 ordens aleatórias conservam os mesmos identificadores, hashes, pontos
iniciais e checkpoints. Logo, não precisam ser otimizadas novamente.

A nova ordem deliberada possui outro `order_hash`, pois sua sequência de colunas
mudou. Assim, somente os 40 restarts da ordem deliberada correta serão gerados.

### Tamanho efetivo da correlação

A tabela passa a distinguir:

- `n_random_orders_total`;
- `n_random_orders_effective`;
- `excluded_order_ids`.

Ordens com `predicted_causal_score_mean = NaN` são explicitamente registradas e
retiradas da correlação. O valor de $\rho$ nunca mais será apresentado apenas
com o número nominal de permutações.


In [ ]:
# ============================================================
# 48. TESTE DE PERMUTAÇÃO DO CSV — SOMENTE EM CÓPIAS
# ============================================================
#
# CONTRATO CIENTÍFICO
# -------------------
# 1. O CSV original nunca é alterado.
# 2. Cada ordem é uma permutação das mesmas séries de preço.
# 3. O espectro das 210 carteiras deve permanecer invariante.
# 4. O bitstring ótimo é recalculado na nova ordem.
# 5. Os mesmos pontos iniciais COBYLA são usados em todas as ordens.
# 6. Descoberta angular e intervenção usam subconjuntos separados.
# 7. A ordem deliberada não entra na correlação das ordens aleatórias.
#

# ------------------------------------------------------------
# CONFIGURAÇÃO
# ------------------------------------------------------------

# False: apenas define o experimento.
# True: cria as cópias e executa o teste.
run_csv_permutation_test = False

# False: teste rápido de fluxo.
# True: teste recomendado pelo relatório.
shuffle_robust_mode = False

shuffle_n_random_orders = (
    15
    if shuffle_robust_mode
    else 3
)

# O piloto precisa de pelo menos 12 restarts:
# metade para descoberta (>=6) e metade para validação.
shuffle_restarts_per_order = (
    40
    if shuffle_robust_mode
    else 14
)

shuffle_cobyla_maxiter = 250
shuffle_k = 4
shuffle_seed = 123

# Inclui uma ordenação deliberada.
shuffle_include_deliberate_order = True

# CONTROLE POSITIVO ESPECÍFICO:
# esta versão testa explicitamente se AMD e COST, quando colocados
# em posições consecutivas, deslocam a sensibilidade causal para
# as portas que tocam esse par.
#
# A versão 19.2 não possuía erro no índice de inserção. O problema
# era outro: ela escolhia automaticamente o ativo incluído e o
# ativo excluído com maior margem. No conjunto atual, isso produziu
# DAL/AMD, e não AMD/COST.
shuffle_deliberate_pair = (
    "AMD",
    "COST",
)

# Mantém o par no centro da cadeia para reduzir efeitos puramente
# associados às bordas do circuito.
shuffle_deliberate_insertion_position = None

# A descoberta e a validação usam metades separadas do banco.
shuffle_discovery_fraction = 0.50

shuffle_n_intervention_vectors = (
    10
    if shuffle_robust_mode
    else 2
)

shuffle_output_dir = (
    financial_output_dir
    / "shuffle_test"
)

shuffle_csv_dir = (
    shuffle_output_dir
    / "csv"
)

shuffle_bank_dir = (
    shuffle_output_dir
    / "bank"
)

shuffle_table_dir = (
    shuffle_output_dir
    / "tables"
)

shuffle_figure_dir = (
    shuffle_output_dir
    / "figures"
)

for directory in [
    shuffle_csv_dir,
    shuffle_bank_dir,
    shuffle_table_dir,
    shuffle_figure_dir,
]:
    directory.mkdir(
        parents=True,
        exist_ok=True,
    )


# ------------------------------------------------------------
# FUNÇÕES AUXILIARES
# ------------------------------------------------------------

def build_problem_context_from_price_copy(
    copied_prices,
    k_value,
    context_seed,
):
    """
    Reconstrói o mesmo problema usando exclusivamente
    a cópia embaralhada do DataFrame.
    """
    copied_prices = (
        copied_prices
        .copy(
            deep=True
        )
        .apply(
            pd.to_numeric,
            errors="coerce",
        )
        .replace(
            [
                np.inf,
                -np.inf,
            ],
            np.nan,
        )
        .dropna(
            axis=1,
            how="all",
        )
        .ffill()
        .bfill()
        .dropna()
    )

    n_value = int(
        copied_prices.shape[
            1
        ]
    )

    if n_value != n_assets:
        raise ValueError(
            f"A cópia deveria possuir {n_assets} ativos, "
            f"mas possui {n_value}."
        )

    local_daily_returns = (
        copied_prices
        .pct_change(
            fill_method=None
        )
        .dropna()
    )

    local_assets_returns = (
        local_daily_returns.sum()
    )

    local_covariance = (
        local_daily_returns.cov()
    )

    sigma = (
        local_covariance
        .to_numpy(
            dtype=float
        )
    )

    sigma = 0.5 * (
        sigma
        + sigma.T
    )

    mu = (
        local_assets_returns
        .to_numpy(
            dtype=float
        )
    )

    # Reconstrói também a formulação CVXPY, usando exatamente
    # os mesmos coeficientes da cópia embaralhada.
    local_cvx_problem, local_cvx_variable = (
        build_cvxpy_portfolio_problem(
            return_values=mu,
            covariance_matrix=sigma,
            k_value=k_value,
        )
    )

    local_cvx_result = (
        try_solve_cvxpy_miqp(
            problem=local_cvx_problem,
            variable=local_cvx_variable,
        )
    )

    local_model = Model(
        name=(
            f"shuffle_portfolio_k{k_value}"
        )
    )

    local_variables = np.array(
        [
            local_model.binary_var(
                name=f"x_{index}"
            )
            for index in range(
                n_value
            )
        ],
        dtype=object,
    )

    risk_expression = local_model.sum(
        float(
            sigma[
                row,
                column,
            ]
        )
        * local_variables[
            row
        ]
        * local_variables[
            column
        ]
        for row in range(
            n_value
        )
        for column in range(
            n_value
        )
    )

    return_expression = local_model.sum(
        float(
            mu[
                index
            ]
        )
        * local_variables[
            index
        ]
        for index in range(
            n_value
        )
    )

    local_model.minimize(
        q_value
        * risk_expression
        - (
            1
            - q_value
        )
        * return_expression
        + risk_free
    )

    local_model.add_constraint(
        local_model.sum(
            local_variables.tolist()
        )
        == int(
            k_value
        ),
        ctname="budget",
    )

    local_quadratic_program = (
        from_docplex_mp(
            model=local_model
        )
    )

    local_qubo = (
        QuadraticProgramToQubo()
        .convert(
            local_quadratic_program
        )
    )

    local_ising, local_offset = (
        local_qubo.to_ising()
    )

    local_exact_result = (
        NumPyMinimumEigensolver()
        .compute_minimum_eigenvalue(
            operator=local_ising
        )
    )

    local_exact_energy = float(
        np.real(
            local_exact_result.eigenvalue
            + local_offset
        )
    )

    local_exact_state = (
        local_exact_result
        .eigenstate
        .to_dict()
    )

    local_exact_bitstrings = sorted(
        {
            str(
                bitstring
            ).replace(
                " ",
                "",
            )
            for bitstring, amplitude
            in local_exact_state.items()
            if abs(
                amplitude
            )
            > 1e-12
        }
    )

    if not local_exact_bitstrings:
        raise RuntimeError(
            "A cópia embaralhada não produziu bitstring ótimo."
        )

    local_exact_bitstring = (
        local_exact_bitstrings[
            0
        ]
    )

    # O mesmo seed mantém a estrutura e a ordem dos theta
    # idênticas entre todas as permutações.
    local_ansatz, local_structure = (
        build_tracked_dicke_ansatz(
            n_value=n_value,
            k_value=int(
                k_value
            ),
            seed=int(
                context_seed
            ),
        )
    )

    local_simulator = AerSimulator(
        method="statevector",
        device="CPU",
    )

    local_estimator = BackendEstimatorV2(
        backend=local_simulator
    )

    return {
        "n": n_value,
        "k": int(
            k_value
        ),
        "tickers": list(
            copied_prices.columns
        ),
        "assets_returns": (
            local_assets_returns
        ),
        "covariance": (
            local_covariance
        ),
        "ising": local_ising,
        "offset": float(
            local_offset
        ),
        "exact_energy": (
            local_exact_energy
        ),
        "exact_bitstring": (
            local_exact_bitstring
        ),
        "exact_bitstrings": (
            local_exact_bitstrings
        ),
        "cvxpy_result": (
            local_cvx_result
        ),
        "ansatz": local_ansatz,
        "n_parameters": int(
            local_ansatz.num_parameters
        ),
        "structure_df": (
            local_structure
        ),
        "simulator": local_simulator,
        "estimator": local_estimator,
        "source_prices": (
            copied_prices
        ),
    }


def run_shuffle_cobyla(
    context,
    run_index,
):
    """
    Uma rodada COBYLA para uma cópia embaralhada.

    O ponto inicial depende apenas de run_index.
    Assim, todas as ordens usam os mesmos pontos iniciais
    e a comparação fica pareada.
    """
    local_rng = np.random.default_rng(
        shuffle_seed
        + 8_000_000
        + int(
            run_index
        )
    )

    theta_initial = (
        0.5
        * np.pi
        * local_rng.random(
            context[
                "n_parameters"
            ]
        )
    )

    callback_energies = []

    def objective_function(
        theta,
    ):
        theta = np.asarray(
            theta,
            dtype=float,
        ).reshape(-1)

        result = context[
            "estimator"
        ].run(
            pubs=[
                (
                    context[
                        "ansatz"
                    ],
                    context[
                        "ising"
                    ],
                    theta,
                )
            ]
        ).result()

        energy = float(
            np.real(
                np.asarray(
                    result[
                        0
                    ].data.evs
                ).reshape(-1)[
                    0
                ]
            )
            + context[
                "offset"
            ]
        )

        callback_energies.append(
            energy
        )

        return energy

    optimizer = COBYLA(
        maxiter=int(
            shuffle_cobyla_maxiter
        )
    )

    start_time = perf_counter()

    result = optimizer.minimize(
        fun=objective_function,
        x0=theta_initial,
    )

    elapsed = (
        perf_counter()
        - start_time
    )

    theta_final = np.asarray(
        result.x,
        dtype=float,
    ).reshape(-1)

    exact_metrics = exact_theta_metrics(
        context,
        theta_final,
    )

    return {
        "run": int(
            run_index
        ),
        "initial_point": (
            theta_initial
        ),
        "best_parameters": (
            theta_final
        ),
        "objective_function_value": float(
            result.fun
        ),
        "nfev": int(
            len(
                callback_energies
            )
        ),
        "optimizer_time": float(
            elapsed
        ),
        **exact_metrics,
    }


def circular_center(
    angles,
    period,
):
    """
    Centro circular respeitando o período físico da porta.
    """
    angles = np.asarray(
        angles,
        dtype=float,
    )

    phase = (
        2
        * np.pi
        * angles
        / float(
            period
        )
    )

    center_phase = np.arctan2(
        np.mean(
            np.sin(
                phase
            )
        ),
        np.mean(
            np.cos(
                phase
            )
        ),
    )

    return float(
        center_phase
        * float(
            period
        )
        / (
            2
            * np.pi
        )
    )


def circular_distance(
    angle_a,
    angle_b,
    period,
):
    difference = (
        float(
            angle_a
        )
        - float(
            angle_b
        )
        + float(
            period
        )
        / 2
    ) % float(
        period
    ) - float(
        period
    ) / 2

    return abs(
        float(
            difference
        )
    )


def discover_shuffle_regions(
    discovery_df,
    candidate_table,
):
    """
    Descobre regiões boas e ruins por centros circulares dos
    melhores e piores vetores. É mais estável que muitas faixas
    quando o banco possui apenas 30-50 restarts.
    """
    if len(
        discovery_df
    ) < 6:
        raise ValueError(
            "São necessários pelo menos seis vetores "
            "na metade de descoberta."
        )

    ordered_good = discovery_df.sort_values(
        [
            "gap_exact_eval",
            "p_exact_eval",
        ],
        ascending=[
            True,
            False,
        ],
    )

    ordered_bad = discovery_df.sort_values(
        [
            "gap_exact_eval",
            "p_exact_eval",
        ],
        ascending=[
            False,
            True,
        ],
    )

    group_size = max(
        3,
        int(
            np.floor(
                0.30
                * len(
                    discovery_df
                )
            )
        ),
    )

    group_size = min(
        group_size,
        len(
            discovery_df
        )
        // 2,
    )

    good_group = ordered_good.head(
        group_size
    )

    bad_group = ordered_bad.head(
        group_size
    )

    rows = []

    for _, parameter_row in candidate_table.iterrows():
        theta_index = int(
            parameter_row[
                "theta_index"
            ]
        )

        period = float(
            parameter_row[
                "angular_period"
            ]
        )

        good_angle = circular_center(
            [
                theta[
                    theta_index
                ]
                for theta
                in good_group[
                    "best_parameters"
                ]
            ],
            period=period,
        )

        bad_angle = circular_center(
            [
                theta[
                    theta_index
                ]
                for theta
                in bad_group[
                    "best_parameters"
                ]
            ],
            period=period,
        )

        separation = circular_distance(
            good_angle,
            bad_angle,
            period,
        )

        rows.append(
            {
                "theta_index": (
                    theta_index
                ),
                "good_angle": (
                    good_angle
                ),
                "bad_angle": (
                    bad_angle
                ),
                "angular_separation": (
                    separation
                ),
                "region_valid": bool(
                    separation
                    >= 0.03
                    * period
                ),
            }
        )

    return pd.DataFrame(
        rows
    )


def choose_low_exposure_controls(
    parameter_table,
):
    """
    Para cada tipo físico previsto, escolhe controles do mesmo
    tipo de porta, mas fora do par e com baixa exposição financeira.
    """
    predicted = (
        parameter_table[
            parameter_table[
                "predicted_by_finance"
            ]
        ]
        .copy()
    )

    non_predicted = (
        parameter_table[
            ~parameter_table[
                "predicted_by_finance"
            ]
        ]
        .copy()
    )

    selected_controls = []
    used_control_indices = set()

    for _, predicted_row in predicted.iterrows():
        same_type = non_predicted[
            non_predicted[
                "physical_type"
            ]
            == predicted_row[
                "physical_type"
            ]
        ].copy()

        same_type = same_type[
            ~same_type[
                "theta_index"
            ].isin(
                used_control_indices
            )
        ]

        if same_type.empty:
            continue

        same_type[
            "control_priority"
        ] = (
            same_type[
                "financial_exposure_mean"
            ]
            + 1e-6
            * np.abs(
                same_type[
                    "n_occurrences"
                ]
                - predicted_row[
                    "n_occurrences"
                ]
            )
        )

        selected = (
            same_type
            .sort_values(
                [
                    "control_priority",
                    "theta_index",
                ]
            )
            .iloc[
                0
            ]
        )

        used_control_indices.add(
            int(
                selected[
                    "theta_index"
                ]
            )
        )

        selected_controls.append(
            selected
        )

    if selected_controls:
        control_df = pd.DataFrame(
            selected_controls
        )
    else:
        control_df = pd.DataFrame(
            columns=parameter_table.columns
        )

    return (
        predicted,
        control_df,
    )


def causal_effect_for_candidate(
    context,
    good_validation_df,
    bad_validation_df,
    theta_index,
    good_angle,
    bad_angle,
    energy_span,
):
    """
    Teste de necessidade e suficiência de um único theta.
    """
    necessity_delta_energy = []
    necessity_delta_probability = []

    for _, base_row in good_validation_df.iterrows():
        after = intervene_one_parameter(
            context=context,
            theta=base_row[
                "best_parameters"
            ],
            theta_index=int(
                theta_index
            ),
            new_angle=float(
                bad_angle
            ),
        )

        necessity_delta_energy.append(
            after[
                "energy_exact_eval"
            ]
            - float(
                base_row[
                    "energy_exact_eval"
                ]
            )
        )

        necessity_delta_probability.append(
            after[
                "p_exact_eval"
            ]
            - float(
                base_row[
                    "p_exact_eval"
                ]
            )
        )

    sufficiency_delta_energy = []
    sufficiency_delta_probability = []

    for _, base_row in bad_validation_df.iterrows():
        after = intervene_one_parameter(
            context=context,
            theta=base_row[
                "best_parameters"
            ],
            theta_index=int(
                theta_index
            ),
            new_angle=float(
                good_angle
            ),
        )

        sufficiency_delta_energy.append(
            after[
                "energy_exact_eval"
            ]
            - float(
                base_row[
                    "energy_exact_eval"
                ]
            )
        )

        sufficiency_delta_probability.append(
            after[
                "p_exact_eval"
            ]
            - float(
                base_row[
                    "p_exact_eval"
                ]
            )
        )

    necessity_energy_median = float(
        np.median(
            necessity_delta_energy
        )
    )

    necessity_probability_median = float(
        np.median(
            necessity_delta_probability
        )
    )

    sufficiency_energy_median = float(
        np.median(
            sufficiency_delta_energy
        )
    )

    sufficiency_probability_median = float(
        np.median(
            sufficiency_delta_probability
        )
    )

    # Score dimensionalmente comparável entre todas as ordens:
    # o span de energia é invariante sob permutação.
    causal_score = float(
        0.25
        * (
            max(
                necessity_energy_median,
                0.0,
            )
            / max(
                float(
                    energy_span
                ),
                1e-15,
            )
            + max(
                -necessity_probability_median,
                0.0,
            )
            + max(
                -sufficiency_energy_median,
                0.0,
            )
            / max(
                float(
                    energy_span
                ),
                1e-15,
            )
            + max(
                sufficiency_probability_median,
                0.0,
            )
        )
    )

    return {
        "necessity_delta_energy_median": (
            necessity_energy_median
        ),
        "necessity_delta_probability_median": (
            necessity_probability_median
        ),
        "sufficiency_delta_energy_median": (
            sufficiency_energy_median
        ),
        "sufficiency_delta_probability_median": (
            sufficiency_probability_median
        ),
        "causal_score_comparable": (
            causal_score
        ),
        "necessity_energy_p": safe_wilcoxon(
            necessity_delta_energy,
            alternative="greater",
        ),
        "necessity_probability_p": safe_wilcoxon(
            necessity_delta_probability,
            alternative="less",
        ),
        "sufficiency_energy_p": safe_wilcoxon(
            sufficiency_delta_energy,
            alternative="less",
        ),
        "sufficiency_probability_p": safe_wilcoxon(
            sufficiency_delta_probability,
            alternative="greater",
        ),
    }


def empirical_spearman_pvalue(
    x_values,
    y_values,
    n_permutations=5000,
    seed=456,
):
    """
    P-valor empírico para a correlação entre margem e efeito causal.
    """
    x_values = np.asarray(
        x_values,
        dtype=float,
    )

    y_values = np.asarray(
        y_values,
        dtype=float,
    )

    valid = (
        np.isfinite(
            x_values
        )
        & np.isfinite(
            y_values
        )
    )

    x_values = x_values[
        valid
    ]

    y_values = y_values[
        valid
    ]

    if (
        len(
            x_values
        )
        < 5
        or np.unique(
            x_values
        ).size < 2
        or np.unique(
            y_values
        ).size < 2
    ):
        return {
            "rho": np.nan,
            "p_asymptotic": np.nan,
            "p_empirical": np.nan,
        }

    observed = spearmanr(
        x_values,
        y_values,
    )

    observed_rho = float(
        observed.statistic
    )

    local_rng = np.random.default_rng(
        seed
    )

    null_rhos = []

    for _ in range(
        int(
            n_permutations
        )
    ):
        permuted_y = local_rng.permutation(
            y_values
        )

        null_rhos.append(
            float(
                spearmanr(
                    x_values,
                    permuted_y,
                ).statistic
            )
        )

    null_rhos = np.asarray(
        null_rhos,
        dtype=float,
    )

    p_empirical = float(
        (
            1
            + np.sum(
                np.abs(
                    null_rhos
                )
                >= abs(
                    observed_rho
                )
            )
        )
        / (
            len(
                null_rhos
            )
            + 1
        )
    )

    return {
        "rho": observed_rho,
        "p_asymptotic": float(
            observed.pvalue
        ),
        "p_empirical": (
            p_empirical
        ),
    }


# ------------------------------------------------------------
# EXECUÇÃO
# ------------------------------------------------------------

shuffle_summary_df = pd.DataFrame()
shuffle_causal_effects_df = pd.DataFrame()
shuffle_correlation_df = pd.DataFrame()
shuffle_verdict_df = pd.DataFrame()

if not run_csv_permutation_test:
    print(
        "Teste de permutação definido, mas não executado."
    )

    print(
        "Para executar, defina:"
    )

    print(
        "run_csv_permutation_test = True"
    )

else:
    import hashlib

    original_prices_snapshot = (
        stocks_prices.copy(
            deep=True
        )
    )

    original_columns = list(
        original_prices_snapshot.columns
    )

    original_analysis = (
        financial_context_analysis[
            (
                n_assets,
                shuffle_k,
            )
        ]
    )

    original_enumeration_objectives = np.sort(
        original_analysis[
            "enumeration_df"
        ][
            "objective"
        ].to_numpy(
            dtype=float
        )
    )

    original_selected_tickers = sorted(
        original_analysis[
            "decision_df"
        ].loc[
            original_analysis[
                "decision_df"
            ][
                "selected_exact"
            ]
            == 1,
            "ticker",
        ].astype(str)
    )

    original_exact_energy = float(
        nk_contexts[
            (
                n_assets,
                shuffle_k,
            )
        ][
            "exact_energy"
        ]
    )

    shuffle_rng = np.random.default_rng(
        shuffle_seed
    )

    order_records = []
    seen_orders = {
        tuple(
            original_columns
        )
    }

    while len(
        order_records
    ) < int(
        shuffle_n_random_orders
    ):
        proposed_order = tuple(
            shuffle_rng.permutation(
                original_columns
            ).tolist()
        )

        if proposed_order in seen_orders:
            continue

        seen_orders.add(
            proposed_order
        )

        order_records.append(
            {
                "order_type": "random",
                "order_id": (
                    f"r{len(order_records):02d}"
                ),
                "column_order": list(
                    proposed_order
                ),
            }
        )

    if shuffle_include_deliberate_order:
        # ----------------------------------------------------
        # ORDEM DELIBERADA ESPECÍFICA: AMD/COST ADJACENTES
        # ----------------------------------------------------
        deliberate_left, deliberate_right = (
            shuffle_deliberate_pair
        )

        if deliberate_left == deliberate_right:
            raise ValueError(
                "O par deliberado deve conter dois tickers distintos."
            )

        missing_deliberate_tickers = [
            ticker
            for ticker
            in shuffle_deliberate_pair
            if ticker not in original_columns
        ]

        if missing_deliberate_tickers:
            raise KeyError(
                "Tickers do par deliberado ausentes no CSV: "
                f"{missing_deliberate_tickers}"
            )

        remaining = [
            ticker
            for ticker
            in original_columns
            if ticker not in {
                deliberate_left,
                deliberate_right,
            }
        ]

        if shuffle_deliberate_insertion_position is None:
            # Coloca o bloco no centro da cadeia.
            insertion_position = (
                len(
                    remaining
                )
                // 2
            )
        else:
            insertion_position = int(
                shuffle_deliberate_insertion_position
            )

        if not (
            0
            <= insertion_position
            <= len(
                remaining
            )
        ):
            raise ValueError(
                "shuffle_deliberate_insertion_position fora do intervalo "
                f"[0, {len(remaining)}]."
            )

        deliberate_order = (
            remaining[
                :insertion_position
            ]
            + [
                deliberate_left,
                deliberate_right,
            ]
            + remaining[
                insertion_position:
            ]
        )

        deliberate_positions = (
            deliberate_order.index(
                deliberate_left
            ),
            deliberate_order.index(
                deliberate_right
            ),
        )

        deliberate_pair_is_adjacent = bool(
            abs(
                deliberate_positions[
                    0
                ]
                - deliberate_positions[
                    1
                ]
            )
            == 1
        )

        # Esta asserção transforma a intenção científica em uma
        # condição executável. O teste não continua se o par não
        # estiver realmente lado a lado.
        if not deliberate_pair_is_adjacent:
            raise RuntimeError(
                "Falha na construção deliberada: "
                f"{deliberate_left}/{deliberate_right} ficaram nas posições "
                f"{deliberate_positions}, e não adjacentes."
            )

        order_records.append(
            {
                "order_type": (
                    "deliberate"
                ),
                "order_id": "d00",
                "column_order": (
                    deliberate_order
                ),
                "deliberate_target_pair": (
                    f"{deliberate_left}/"
                    f"{deliberate_right}"
                ),
                "deliberate_target_positions": str(
                    deliberate_positions
                ),
                "deliberate_target_adjacent": (
                    deliberate_pair_is_adjacent
                ),
            }
        )

        print(
            "Ordem deliberada construída:",
            deliberate_order,
        )

        print(
            "Par deliberado:",
            f"{deliberate_left}/{deliberate_right}",
            "posições:",
            deliberate_positions,
            "adjacente:",
            deliberate_pair_is_adjacent,
        )

    all_summary_rows = []
    all_causal_rows = []

    # O mesmo ansatz é usado em todas as ordens.
    shared_ansatz_seed = (
        random_seed
        + 100
        * n_assets
        + shuffle_k
    )

    for order_record in order_records:
        order_type = str(
            order_record[
                "order_type"
            ]
        )

        order_id = str(
            order_record[
                "order_id"
            ]
        )

        column_order = list(
            order_record[
                "column_order"
            ]
        )

        deliberate_target_pair = (
            order_record.get(
                "deliberate_target_pair"
            )
        )

        deliberate_target_positions = (
            order_record.get(
                "deliberate_target_positions"
            )
        )

        deliberate_target_adjacent = (
            order_record.get(
                "deliberate_target_adjacent"
            )
        )

        order_hash = hashlib.sha1(
            "|".join(
                column_order
            ).encode(
                "utf-8"
            )
        ).hexdigest()[
            :8
        ]

        short_id = (
            f"{order_id}_{order_hash}"
        )

        copy_path = (
            shuffle_csv_dir
            / f"{short_id}.csv"
        )

        shuffled_prices = (
            original_prices_snapshot[
                column_order
            ]
            .copy(
                deep=True
            )
        )

        # Confirma que é apenas permutação de colunas.
        pd.testing.assert_frame_equal(
            shuffled_prices.sort_index(
                axis=1
            ),
            original_prices_snapshot.sort_index(
                axis=1
            ),
            check_exact=True,
        )

        shuffled_prices.to_csv(
            copy_path,
            index=True,
        )

        # O teste usa a cópia relida do disco.
        copied_prices = pd.read_csv(
            copy_path,
            index_col=0,
        )

        context = (
            build_problem_context_from_price_copy(
                copied_prices=copied_prices,
                k_value=shuffle_k,
                context_seed=shared_ansatz_seed,
            )
        )

        local_analysis = financial_decision_table(
            context
        )

        local_enumeration_objectives = np.sort(
            local_analysis[
                "enumeration_df"
            ][
                "objective"
            ].to_numpy(
                dtype=float
            )
        )

        spectrum_difference = float(
            np.max(
                np.abs(
                    local_enumeration_objectives
                    - original_enumeration_objectives
                )
            )
        )

        local_selected_tickers = sorted(
            local_analysis[
                "decision_df"
            ].loc[
                local_analysis[
                    "decision_df"
                ][
                    "selected_exact"
                ]
                == 1,
                "ticker",
            ].astype(str)
        )

        selected_set_invariant = bool(
            local_selected_tickers
            == original_selected_tickers
        )

        exact_energy_difference = abs(
            float(
                context[
                    "exact_energy"
                ]
            )
            - original_exact_energy
        )

        if (
            spectrum_difference
            > 1e-9
            or exact_energy_difference
            > 1e-8
            or not selected_set_invariant
        ):
            raise RuntimeError(
                "A cópia embaralhada alterou o problema físico. "
                f"order_id={order_id}, "
                f"spectrum_difference={spectrum_difference:.3e}, "
                f"energy_difference={exact_energy_difference:.3e}, "
                f"selected_set_invariant={selected_set_invariant}"
            )

        physical_map, _ = (
            build_physical_parameter_map(
                context[
                    "ansatz"
                ]
            )
        )

        parameter_table = (
            annotate_parameter_financial_exposure(
                parameter_map=physical_map,
                decision_df=local_analysis[
                    "decision_df"
                ],
                decisive_pair=local_analysis[
                    "decisive_pair"
                ],
            )
        )

        predicted_df, control_df = (
            choose_low_exposure_controls(
                parameter_table
            )
        )

        candidate_df = pd.concat(
            [
                predicted_df.assign(
                    candidate_group=(
                        "predicted"
                    )
                ),
                control_df.assign(
                    candidate_group=(
                        "control"
                    )
                ),
            ],
            ignore_index=True,
        ).drop_duplicates(
            subset=[
                "theta_index"
            ]
        )

        local_bank_directory = (
            shuffle_bank_dir
            / short_id
        )

        local_bank_directory.mkdir(
            parents=True,
            exist_ok=True,
        )

        checkpoint_path = (
            local_bank_directory
            / "bank.pkl"
        )

        if checkpoint_path.exists():
            bank_rows = pd.read_pickle(
                checkpoint_path
            )

            if not isinstance(
                bank_rows,
                list,
            ):
                bank_rows = []
        else:
            bank_rows = []

        completed_runs = {
            int(
                row[
                    "run"
                ]
            )
            for row
            in bank_rows
            if row.get(
                "status",
                "ok",
            )
            == "ok"
        }

        for run_index in range(
            int(
                shuffle_restarts_per_order
            )
        ):
            if run_index in completed_runs:
                continue

            run_row = run_shuffle_cobyla(
                context=context,
                run_index=run_index,
            )

            run_row[
                "status"
            ] = "ok"

            run_row[
                "order_id"
            ] = order_id

            run_row[
                "order_type"
            ] = order_type

            bank_rows.append(
                run_row
            )

            pd.to_pickle(
                bank_rows,
                checkpoint_path,
            )

            print(
                short_id,
                f"run {run_index + 1}/"
                f"{shuffle_restarts_per_order}",
                f"gap={run_row['gap_exact_eval']:.6e}",
            )

        bank_df = pd.DataFrame(
            bank_rows
        )

        bank_df = (
            bank_df[
                bank_df[
                    "status"
                ]
                == "ok"
            ]
            .sort_values(
                "run"
            )
            .drop_duplicates(
                subset=[
                    "run"
                ],
                keep="last",
            )
            .head(
                int(
                    shuffle_restarts_per_order
                )
            )
            .reset_index(
                drop=True
            )
        )

        if len(
            bank_df
        ) < 8:
            raise RuntimeError(
                f"O banco {short_id} possui apenas "
                f"{len(bank_df)} vetores válidos."
            )

        local_rng = np.random.default_rng(
            shuffle_seed
            + 900_000
            + sum(
                ord(
                    character
                )
                for character
                in short_id
            )
        )

        permutation = local_rng.permutation(
            len(
                bank_df
            )
        )

        discovery_size = int(
            np.floor(
                shuffle_discovery_fraction
                * len(
                    bank_df
                )
            )
        )

        discovery_size = min(
            max(
                discovery_size,
                4,
            ),
            len(
                bank_df
            )
            - 4,
        )

        discovery_df = (
            bank_df.iloc[
                permutation[
                    :discovery_size
                ]
            ]
            .copy()
            .reset_index(
                drop=True
            )
        )

        validation_df = (
            bank_df.iloc[
                permutation[
                    discovery_size:
                ]
            ]
            .copy()
            .reset_index(
                drop=True
            )
        )

        regions_df = discover_shuffle_regions(
            discovery_df=discovery_df,
            candidate_table=candidate_df,
        )

        candidate_regions = (
            candidate_df.merge(
                regions_df,
                on="theta_index",
                how="left",
                validate="one_to_one",
            )
        )

        # Seleção local e sem sobreposição:
        # melhores vetores para necessidade e piores para suficiência.
        ordered_good = validation_df.sort_values(
            [
                "gap_exact_eval",
                "p_exact_eval",
            ],
            ascending=[
                True,
                False,
            ],
        )

        ordered_bad = validation_df.sort_values(
            [
                "gap_exact_eval",
                "p_exact_eval",
            ],
            ascending=[
                False,
                True,
            ],
        )

        max_intervention_vectors = min(
            int(
                shuffle_n_intervention_vectors
            ),
            max(
                1,
                len(
                    validation_df
                )
                // 2,
            ),
        )

        good_validation_df = (
            ordered_good.head(
                max_intervention_vectors
            )
            .copy()
            .reset_index(
                drop=True
            )
        )

        bad_validation_df = (
            ordered_bad.head(
                max_intervention_vectors
            )
            .copy()
            .reset_index(
                drop=True
            )
        )

        good_validation_df = (
            good_validation_df.head(
                max_intervention_vectors
            )
        )

        bad_validation_df = (
            bad_validation_df.head(
                max_intervention_vectors
            )
        )

        energy_values = (
            local_analysis[
                "enumeration_df"
            ][
                "objective"
            ].to_numpy(
                dtype=float
            )
        )

        energy_span = float(
            np.max(
                energy_values
            )
            - np.min(
                energy_values
            )
        )

        local_effect_rows = []

        for _, candidate_row in candidate_regions.iterrows():
            if not bool(
                candidate_row.get(
                    "region_valid",
                    False,
                )
            ):
                continue

            effect = causal_effect_for_candidate(
                context=context,
                good_validation_df=(
                    good_validation_df
                ),
                bad_validation_df=(
                    bad_validation_df
                ),
                theta_index=int(
                    candidate_row[
                        "theta_index"
                    ]
                ),
                good_angle=float(
                    candidate_row[
                        "good_angle"
                    ]
                ),
                bad_angle=float(
                    candidate_row[
                        "bad_angle"
                    ]
                ),
                energy_span=energy_span,
            )

            local_effect_rows.append(
                {
                    "order_id": order_id,
                    "order_type": order_type,
                    "order_hash": order_hash,
                    "theta_index": int(
                        candidate_row[
                            "theta_index"
                        ]
                    ),
                    "candidate_group": (
                        candidate_row[
                            "candidate_group"
                        ]
                    ),
                    "predicted_role": (
                        candidate_row[
                            "predicted_role"
                        ]
                    ),
                    "physical_type": (
                        candidate_row[
                            "physical_type"
                        ]
                    ),
                    "physical_qubits": str(
                        candidate_row[
                            "physical_qubits"
                        ]
                    ),
                    "financial_exposure_mean": float(
                        candidate_row[
                            "financial_exposure_mean"
                        ]
                    ),
                    "angular_separation": float(
                        candidate_row[
                            "angular_separation"
                        ]
                    ),
                    **effect,
                }
            )

        local_effect_df = pd.DataFrame(
            local_effect_rows
        )

        if local_effect_df.empty:
            predicted_score_mean = np.nan
            predicted_score_max = np.nan
            control_score_mean = np.nan
            predicted_minus_control = np.nan
        else:
            predicted_scores = (
                local_effect_df.loc[
                    local_effect_df[
                        "candidate_group"
                    ]
                    == "predicted",
                    "causal_score_comparable",
                ]
            )

            control_scores = (
                local_effect_df.loc[
                    local_effect_df[
                        "candidate_group"
                    ]
                    == "control",
                    "causal_score_comparable",
                ]
            )

            predicted_score_mean = (
                float(
                    predicted_scores.mean()
                )
                if not predicted_scores.empty
                else np.nan
            )

            predicted_score_max = (
                float(
                    predicted_scores.max()
                )
                if not predicted_scores.empty
                else np.nan
            )

            control_score_mean = (
                float(
                    control_scores.mean()
                )
                if not control_scores.empty
                else np.nan
            )

            predicted_minus_control = (
                predicted_score_mean
                - control_score_mean
                if np.isfinite(
                    predicted_score_mean
                )
                and np.isfinite(
                    control_score_mean
                )
                else np.nan
            )

            all_causal_rows.extend(
                local_effect_df.to_dict(
                    orient="records"
                )
            )

        decisive_pair = (
            local_analysis[
                "decisive_pair"
            ]
        )

        decisive_pair_row = (
            local_analysis[
                "decisive_pair_row"
            ]
        )

        pair_tickers = [
            context[
                "tickers"
            ][
                decisive_pair[
                    0
                ]
            ],
            context[
                "tickers"
            ][
                decisive_pair[
                    1
                ]
            ],
        ]

        # Usa posição explícita, sem depender de conjunto,
        # para documentar onde AMD/COST caíram na cadeia.
        amd_index = (
            context[
                "tickers"
            ].index(
                "AMD"
            )
        )

        cost_index = (
            context[
                "tickers"
            ].index(
                "COST"
            )
        )

        amd_cost_distance = int(
            abs(
                amd_index
                - cost_index
            )
        )

        amd_cost_adjacent = bool(
            amd_cost_distance
            == 1
        )

        all_summary_rows.append(
            {
                "order_id": order_id,
                "order_type": order_type,
                "order_hash": order_hash,
                "csv_copy": str(
                    copy_path.resolve()
                ),
                "column_order": "|".join(
                    column_order
                ),
                "deliberate_target_pair": (
                    deliberate_target_pair
                ),
                "deliberate_target_positions": (
                    deliberate_target_positions
                ),
                "deliberate_target_adjacent": (
                    deliberate_target_adjacent
                ),
                "exact_bitstring_new_order": (
                    context[
                        "exact_bitstring"
                    ]
                ),
                "exact_energy": float(
                    context[
                        "exact_energy"
                    ]
                ),
                "exact_energy_difference": (
                    exact_energy_difference
                ),
                "energy_spectrum_max_difference": (
                    spectrum_difference
                ),
                "selected_ticker_set_invariant": (
                    selected_set_invariant
                ),
                "decisive_pair_indices": str(
                    decisive_pair
                ),
                "decisive_pair_tickers": (
                    "/".join(
                        pair_tickers
                    )
                ),
                "decisive_pair_margin_min": float(
                    decisive_pair_row[
                        "pair_margin_min"
                    ]
                ),
                "decisive_pair_margin_sum": float(
                    decisive_pair_row[
                        "pair_margin_sum"
                    ]
                ),
                "amd_position": int(
                    amd_index
                ),
                "cost_position": int(
                    cost_index
                ),
                "amd_cost_distance": int(
                    amd_cost_distance
                ),
                "amd_cost_adjacent": bool(
                    amd_cost_adjacent
                ),
                "bank_runs": int(
                    len(
                        bank_df
                    )
                ),
                "bank_gap_median": float(
                    bank_df[
                        "gap_exact_eval"
                    ].median()
                ),
                "bank_p_exact_median": float(
                    bank_df[
                        "p_exact_eval"
                    ].median()
                ),
                "bank_exact_dominant_rate_pct": float(
                    100
                    * bank_df[
                        "is_exact_dominant_eval"
                    ].mean()
                ),
                "bank_nfev_median": float(
                    pd.to_numeric(
                        bank_df[
                            "nfev"
                        ],
                        errors="coerce",
                    ).median()
                ),
                "bank_optimizer_time_median": float(
                    pd.to_numeric(
                        bank_df[
                            "optimizer_time"
                        ],
                        errors="coerce",
                    ).median()
                ),
                "cvxpy_solver": (
                    context[
                        "cvxpy_result"
                    ][
                        "solver"
                    ]
                ),
                "cvxpy_status": (
                    context[
                        "cvxpy_result"
                    ][
                        "status"
                    ]
                ),
                "predicted_parameter_count": int(
                    (
                        candidate_df[
                            "candidate_group"
                        ]
                        == "predicted"
                    ).sum()
                ),
                "control_parameter_count": int(
                    (
                        candidate_df[
                            "candidate_group"
                        ]
                        == "control"
                    ).sum()
                ),
                "valid_effect_count": int(
                    len(
                        local_effect_df
                    )
                ),
                "predicted_causal_score_mean": (
                    predicted_score_mean
                ),
                "predicted_causal_score_max": (
                    predicted_score_max
                ),
                "control_causal_score_mean": (
                    control_score_mean
                ),
                "predicted_minus_control": (
                    predicted_minus_control
                ),
            }
        )

        pd.DataFrame(
            all_summary_rows
        ).to_csv(
            shuffle_table_dir
            / "shuffle_summary_checkpoint.csv",
            index=False,
        )

        pd.DataFrame(
            all_causal_rows
        ).to_csv(
            shuffle_table_dir
            / "shuffle_causal_effects_checkpoint.csv",
            index=False,
        )

    # Confirma que o DataFrame original permaneceu intacto.
    pd.testing.assert_frame_equal(
        stocks_prices,
        original_prices_snapshot,
        check_exact=True,
    )

    shuffle_summary_df = pd.DataFrame(
        all_summary_rows
    )

    shuffle_causal_effects_df = pd.DataFrame(
        all_causal_rows
    )

    random_summary = (
        shuffle_summary_df[
            shuffle_summary_df[
                "order_type"
            ]
            == "random"
        ]
        .copy()
    )

    correlation_valid_mask = (
        random_summary[
            [
                "decisive_pair_margin_sum",
                "predicted_causal_score_mean",
            ]
        ]
        .notna()
        .all(
            axis=1
        )
    )

    correlation_valid_summary = (
        random_summary.loc[
            correlation_valid_mask
        ]
        .copy()
        .reset_index(
            drop=True
        )
    )

    correlation_excluded_orders = (
        random_summary.loc[
            ~correlation_valid_mask,
            "order_id",
        ]
        .astype(str)
        .tolist()
    )

    n_random_orders_total = int(
        len(
            random_summary
        )
    )

    n_random_orders_effective = int(
        len(
            correlation_valid_summary
        )
    )

    correlation_result = (
        empirical_spearman_pvalue(
            x_values=correlation_valid_summary[
                "decisive_pair_margin_sum"
            ],
            y_values=correlation_valid_summary[
                "predicted_causal_score_mean"
            ],
            n_permutations=5000,
            seed=(
                shuffle_seed
                + 99
            ),
        )
    )

    print(
        "Ordens aleatórias totais:",
        n_random_orders_total,
    )

    print(
        "N efetivo da correlação:",
        n_random_orders_effective,
    )

    print(
        "Ordens excluídas por NaN:",
        correlation_excluded_orders,
    )

    predicted_control_valid = random_summary[
        [
            "predicted_causal_score_mean",
            "control_causal_score_mean",
        ]
    ].dropna()

    if predicted_control_valid.empty:
        predicted_beats_control_rate = np.nan
    else:
        predicted_beats_control_rate = float(
            100
            * (
                predicted_control_valid[
                    "predicted_causal_score_mean"
                ]
                > predicted_control_valid[
                    "control_causal_score_mean"
                ]
            ).mean()
        )

    shuffle_correlation_df = pd.DataFrame(
        [
            {
                "n_random_orders_total": (
                    n_random_orders_total
                ),
                "n_random_orders_effective": (
                    n_random_orders_effective
                ),
                "excluded_order_ids": "|".join(
                    correlation_excluded_orders
                ),
                "restarts_per_order": int(
                    shuffle_restarts_per_order
                ),
                "spearman_margin_vs_causal": (
                    correlation_result[
                        "rho"
                    ]
                ),
                "spearman_p_asymptotic": (
                    correlation_result[
                        "p_asymptotic"
                    ]
                ),
                "spearman_p_empirical": (
                    correlation_result[
                        "p_empirical"
                    ]
                ),
                "predicted_beats_control_rate_pct": (
                    predicted_beats_control_rate
                ),
            }
        ]
    )

    robust_evidence = bool(
        shuffle_robust_mode
        and n_random_orders_effective
        >= 13
        and shuffle_restarts_per_order
        >= 30
    )

    rho_value = (
        correlation_result[
            "rho"
        ]
    )

    empirical_p = (
        correlation_result[
            "p_empirical"
        ]
    )

    if not robust_evidence:
        shuffle_verdict = (
            "PRELIMINARY_ONLY"
        )
    elif (
        np.isfinite(
            rho_value
        )
        and rho_value >= 0.50
        and np.isfinite(
            empirical_p
        )
        and empirical_p <= 0.05
        and np.isfinite(
            predicted_beats_control_rate
        )
        and predicted_beats_control_rate
        >= 66.0
    ):
        shuffle_verdict = (
            "SUPPORTS_FINANCIAL_LOCATION_HYPOTHESIS"
        )
    elif (
        np.isfinite(
            rho_value
        )
        and rho_value <= 0.10
        and np.isfinite(
            predicted_beats_control_rate
        )
        and predicted_beats_control_rate
        <= 50.0
    ):
        shuffle_verdict = (
            "DOES_NOT_SUPPORT_HYPOTHESIS"
        )
    else:
        shuffle_verdict = (
            "MIXED_EVIDENCE"
        )

    deliberate_summary = shuffle_summary_df[
        shuffle_summary_df[
            "order_type"
        ]
        == "deliberate"
    ].copy()

    if deliberate_summary.empty:
        deliberate_gap_percentile = np.nan
        deliberate_gain_percentile = np.nan

    else:
        deliberate_row = (
            deliberate_summary.iloc[
                0
            ]
        )

        deliberate_gap_percentile = float(
            100
            * (
                random_summary[
                    "bank_gap_median"
                ]
                >= deliberate_row[
                    "bank_gap_median"
                ]
            ).mean()
        )

        deliberate_gain_percentile = float(
            100
            * (
                random_summary[
                    "predicted_minus_control"
                ]
                <= deliberate_row[
                    "predicted_minus_control"
                ]
            ).mean()
        )

    shuffle_verdict_df = pd.DataFrame(
        [
            {
                "verdict": (
                    shuffle_verdict
                ),
                "robust_evidence": (
                    robust_evidence
                ),
                "n_random_orders_total": (
                    n_random_orders_total
                ),
                "n_random_orders_effective": (
                    n_random_orders_effective
                ),
                "excluded_order_ids": "|".join(
                    correlation_excluded_orders
                ),
                "restarts_per_order": int(
                    shuffle_restarts_per_order
                ),
                "deliberate_gap_percentile": (
                    deliberate_gap_percentile
                ),
                "deliberate_gain_percentile": (
                    deliberate_gain_percentile
                ),
            }
        ]
    )

    shuffle_summary_df.to_csv(
        shuffle_table_dir
        / "shuffle_summary.csv",
        index=False,
    )

    shuffle_causal_effects_df.to_csv(
        shuffle_table_dir
        / "shuffle_causal_effects.csv",
        index=False,
    )

    shuffle_correlation_df.to_csv(
        shuffle_table_dir
        / "shuffle_correlation.csv",
        index=False,
    )

    shuffle_verdict_df.to_csv(
        shuffle_table_dir
        / "shuffle_verdict.csv",
        index=False,
    )

    display(
        shuffle_summary_df
    )

    display(
        shuffle_correlation_df
    )

    display(
        shuffle_verdict_df
    )


    # ------------------------------------------------------------
    # GRÁFICO 1 — margem financeira x efeito causal (melhorado)
    # ------------------------------------------------------------
    #
    # Agora inclui a ordem deliberada (marcada à parte, não como um
    # ponto qualquer do ruído aleatório), destaca quando AMD/COST
    # calharam vizinhos por acaso, e mostra a reta de tendência e o
    # coeficiente de Spearman quando há pontos suficientes.

    fig, ax = plt.subplots(figsize=(9, 6.5))

    non_adjacent = random_summary[~random_summary["amd_cost_adjacent"]]
    adjacent = random_summary[random_summary["amd_cost_adjacent"]]

    ax.scatter(
        non_adjacent["decisive_pair_margin_sum"],
        non_adjacent["predicted_causal_score_mean"],
        color="#2b6cb0",
        label="ordem aleatória",
        zorder=3,
    )

    if not adjacent.empty:
        ax.scatter(
            adjacent["decisive_pair_margin_sum"],
            adjacent["predicted_causal_score_mean"],
            color="#d97706",
            marker="D",
            s=70,
            label="ordem aleatória (AMD/COST ficaram adjacentes por acaso)",
            zorder=4,
        )

    deliberate_rows = shuffle_summary_df[shuffle_summary_df["order_type"] == "deliberate"]

    if not deliberate_rows.empty:
        ax.scatter(
            deliberate_rows["decisive_pair_margin_sum"],
            deliberate_rows["predicted_causal_score_mean"],
            color="#dc2626",
            marker="*",
            s=280,
            label="ordem deliberada — score do par automático",
            zorder=5,
            edgecolors="black",
            linewidths=0.6,
        )

    for _, row in pd.concat([random_summary, deliberate_rows]).iterrows():
        ax.annotate(
            str(row["order_id"]),
            (row["decisive_pair_margin_sum"], row["predicted_causal_score_mean"]),
            xytext=(5, 5),
            textcoords="offset points",
            fontsize=8,
            color="#555555",
        )

    if len(random_summary) >= 3:
        valid = random_summary[["decisive_pair_margin_sum", "predicted_causal_score_mean"]].dropna()
        if valid["decisive_pair_margin_sum"].nunique() > 1:
            coeffs = np.polyfit(
                valid["decisive_pair_margin_sum"],
                valid["predicted_causal_score_mean"],
                1,
            )
            x_line = np.linspace(
                valid["decisive_pair_margin_sum"].min(),
                valid["decisive_pair_margin_sum"].max(),
                50,
            )
            ax.plot(
                x_line,
                np.polyval(coeffs, x_line),
                linestyle="--",
                color="#555555",
                alpha=0.7,
                label="tendência (só ordens aleatórias)",
            )

    rho_text = (
        f"ρ de Spearman = {correlation_result['rho']:.3f}"
        f"  (p empírico = {correlation_result['p_empirical']:.3f})"
        if np.isfinite(correlation_result["rho"])
        else "ρ de Spearman: dados insuficientes"
    )

    ax.text(
        0.02, 0.98, rho_text,
        transform=ax.transAxes,
        fontsize=9,
        va="top",
        color="#333333",
        bbox=dict(boxstyle="round", facecolor="white", alpha=0.75, edgecolor="#cccccc"),
    )

    ax.set_xlabel("Margem financeira do melhor par vizinho")
    ax.set_ylabel("Score causal médio do par automático previsto")
    ax.set_title("1. Margem financeira versus efeito causal, entre embaralhamentos")
    ax.grid(alpha=0.25)
    ax.legend(fontsize=8.5, loc="lower right")
    fig.tight_layout()
    fig.savefig(shuffle_figure_dir / "01_margin_vs_causal.png", dpi=250, bbox_inches="tight")
    plt.show()


    # ------------------------------------------------------------
    # GRÁFICO 2 — previstos x controles (melhorado)
    # ------------------------------------------------------------
    #
    # Agora inclui a ordem deliberada lado a lado com as aleatórias,
    # com rótulos de valor e uma cor distinta para ela.

    comparison_plot = (
        shuffle_summary_df[["order_id", "order_type", "predicted_causal_score_mean", "control_causal_score_mean"]]
        .sort_values(["order_type", "order_id"])
        .set_index("order_id")
    )

    fig, ax = plt.subplots(figsize=(11, 5.5))

    x_positions = np.arange(len(comparison_plot))
    bar_width = 0.36

    colors_predicted = [
        "#dc2626" if t == "deliberate" else "#2b6cb0"
        for t in comparison_plot["order_type"]
    ]

    bars_predicted = ax.bar(
        x_positions - bar_width / 2,
        comparison_plot["predicted_causal_score_mean"],
        width=bar_width,
        color=colors_predicted,
        label="par automático previsto",
    )

    bars_control = ax.bar(
        x_positions + bar_width / 2,
        comparison_plot["control_causal_score_mean"],
        width=bar_width,
        color="#94a3b8",
        label="controles de baixa exposição",
    )

    for bar, value in zip(bars_predicted, comparison_plot["predicted_causal_score_mean"]):
        if np.isfinite(value):
            ax.text(bar.get_x() + bar.get_width()/2, bar.get_height(), f"{value:.3f}",
                    ha="center", va="bottom", fontsize=7.5)

    ax.set_xticks(x_positions)
    ax.set_xticklabels(comparison_plot.index, rotation=0)
    ax.set_xlabel("Permutação (vermelho = ordem deliberada)")
    ax.set_ylabel("Score causal comparável")
    ax.set_title("2. Par automático previsto versus controles, em cada ordem")
    ax.grid(axis="y", alpha=0.25)
    ax.legend(fontsize=8.5)
    fig.tight_layout()
    fig.savefig(shuffle_figure_dir / "02_predicted_vs_controls.png", dpi=250, bbox_inches="tight")
    plt.show()


    # ------------------------------------------------------------
    # GRÁFICO 3 — a ordem deliberada se destaca do ruído aleatório?
    # ------------------------------------------------------------
    #
    # Este é o gráfico que responde à pergunta central do teste:
    # distribui a vantagem (previsto - controle) das ordens aleatórias
    # e marca onde a ordem deliberada cai nessa distribuição.

    fig, ax = plt.subplots(figsize=(9, 5.5))

    random_gaps = random_summary["predicted_minus_control"].dropna()

    if len(random_gaps) >= 2:
        ax.hist(
            random_gaps,
            bins=min(10, max(3, len(random_gaps))),
            color="#94a3b8",
            alpha=0.75,
            edgecolor="white",
            label=f"ordens aleatórias (n={len(random_gaps)})",
        )
    elif len(random_gaps) == 1:
        ax.axvline(random_gaps.iloc[0], color="#94a3b8", linewidth=3,
                    label="única ordem aleatória disponível")

    if not deliberate_rows.empty:
        deliberate_gap = float(deliberate_rows["predicted_minus_control"].iloc[0])
        ax.axvline(
            deliberate_gap,
            color="#dc2626",
            linewidth=2.5,
            linestyle="--",
            label=f"ordem deliberada ({deliberate_gap:.4f})",
        )

        if len(random_gaps) >= 2:
            percentile = float(100 * (random_gaps < deliberate_gap).mean())
            ax.text(
                0.02, 0.98,
                f"a ordem deliberada supera {percentile:.0f}% das\nordens aleatórias nesta métrica",
                transform=ax.transAxes, fontsize=9, va="top", color="#333333",
                bbox=dict(boxstyle="round", facecolor="white", alpha=0.8, edgecolor="#cccccc"),
            )

    ax.axvline(0, color="#333333", linewidth=1, alpha=0.5)
    ax.set_xlabel("Score causal: par automático previsto − controle")
    ax.set_ylabel("Número de ordens aleatórias")
    ax.set_title("3. A ordem deliberada se destaca do ruído aleatório?")
    ax.grid(axis="y", alpha=0.25)
    ax.legend(fontsize=8.5)
    fig.tight_layout()
    fig.savefig(shuffle_figure_dir / "03_deliberate_vs_random_noise.png", dpi=250, bbox_inches="tight")
    plt.show()


    # ------------------------------------------------------------
    # GRÁFICO 4 — margem financeira do par decisivo, por ordem
    # ------------------------------------------------------------
    #
    # Mostra por que a ordem deliberada deveria ter efeito maior:
    # ela foi construída para ter a maior margem financeira possível
    # entre pares vizinhos, o que as ordens aleatórias só atingem
    # por acaso.

    fig, ax = plt.subplots(figsize=(10, 5.5))

    plot_order = shuffle_summary_df.sort_values(
        ["order_type", "decisive_pair_margin_sum"], ascending=[True, False]
    )

    bar_colors = [
        "#dc2626" if t == "deliberate"
        else ("#d97706" if adj else "#2b6cb0")
        for t, adj in zip(plot_order["order_type"], plot_order["amd_cost_adjacent"])
    ]

    bars = ax.bar(
        plot_order["order_id"],
        plot_order["decisive_pair_margin_sum"],
        color=bar_colors,
    )

    for bar, ticker_pair in zip(bars, plot_order["decisive_pair_tickers"]):
        ax.text(
            bar.get_x() + bar.get_width() / 2, bar.get_height(),
            ticker_pair, ha="center", va="bottom", fontsize=7.5, rotation=60,
        )

    from matplotlib.patches import Patch
    legend_handles = [
        Patch(color="#2b6cb0", label="ordem aleatória"),
        Patch(color="#d97706", label="ordem aleatória (AMD/COST vizinhos por acaso)"),
        Patch(color="#dc2626", label="ordem deliberada"),
    ]

    ax.set_xlabel("Ordem (identificador)")
    ax.set_ylabel("Margem financeira do par decisivo")
    ax.set_title("4. Robustez financeira do par decisivo encontrado em cada ordem")
    ax.grid(axis="y", alpha=0.25)
    ax.legend(handles=legend_handles, fontsize=8, loc="upper right")
    fig.tight_layout()
    fig.savefig(shuffle_figure_dir / "04_decisive_pair_margin_by_order.png", dpi=250, bbox_inches="tight")
    plt.show()


    # ------------------------------------------------------------
    # GRÁFICO 5 — checagem de invariância física entre embaralhamentos
    # ------------------------------------------------------------
    #
    # Confirma visualmente que todo embaralhamento manteve o mesmo
    # problema físico: mesma energia exata, mesmo conjunto de ativos
    # selecionados. Isso não é decorativo — é a evidência de que o
    # teste de permutação está medindo o que diz medir.

    fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

    ax = axes[0]
    ax.bar(
        shuffle_summary_df["order_id"],
        shuffle_summary_df["exact_energy_difference"].clip(lower=1e-18),
        color=["#dc2626" if t == "deliberate" else "#2b6cb0" for t in shuffle_summary_df["order_type"]],
    )
    ax.set_yscale("log")
    ax.axhline(1e-8, linestyle="--", color="#333333", linewidth=1, label="tolerância aceita (1e-8)")
    ax.set_ylabel("|diferença de energia exata|")
    ax.set_title("Energia exata permaneceu igual?")
    ax.grid(axis="y", alpha=0.25)
    ax.legend(fontsize=8)

    ax = axes[1]
    invariant_counts = shuffle_summary_df["selected_ticker_set_invariant"].value_counts()
    bar_labels = [str(v) for v in invariant_counts.index]
    ax.bar(
        bar_labels,
        invariant_counts.values,
        color=["#16a34a" if v else "#dc2626" for v in invariant_counts.index],
    )
    ax.set_ylabel("Número de ordens")
    ax.set_title("Mesmo conjunto de ativos selecionado?")
    ax.grid(axis="y", alpha=0.25)

    fig.suptitle("5. Checagem de invariância física entre embaralhamentos", y=1.02)
    fig.tight_layout()
    fig.savefig(shuffle_figure_dir / "05_invariance_check.png", dpi=250, bbox_inches="tight")
    plt.show()


    # ------------------------------------------------------------
    # GRÁFICO 6 — score causal por parâmetro, previsto x controle
    # ------------------------------------------------------------
    #
    # Granularidade de parâmetro individual, não só a média por ordem:
    # todo ponto de parâmetro previsto (por ordem) contra todo ponto
    # de parâmetro controle, para ver se a separação entre os dois
    # grupos é consistente parâmetro a parâmetro, não só na média.

    fig, ax = plt.subplots(figsize=(9, 5.5))

    group_order = ["control", "predicted"]
    positions = {"control": 0, "predicted": 1}
    jitter_rng = np.random.default_rng(7)

    for order_id, order_type in shuffle_summary_df[["order_id", "order_type"]].itertuples(index=False):
        subset = shuffle_causal_effects_df[shuffle_causal_effects_df["order_id"] == order_id]
        color = "#dc2626" if order_type == "deliberate" else "#2b6cb0"
        alpha = 0.9 if order_type == "deliberate" else 0.45

        for group in group_order:
            values = subset.loc[subset["candidate_group"] == group, "causal_score_comparable"]
            if values.empty:
                continue
            x_jitter = positions[group] + jitter_rng.uniform(-0.12, 0.12, len(values))
            ax.scatter(x_jitter, values, color=color, alpha=alpha, s=35, zorder=3)

    ax.set_xticks([0, 1])
    ax.set_xticklabels(["controle\n(baixa exposição)", "previsto pelo\npar automático"])
    ax.set_ylabel("Score causal comparável (por parâmetro)")
    ax.set_title("6. Score causal do par automático — vermelho = ordem deliberada")
    ax.grid(axis="y", alpha=0.25)
    fig.tight_layout()
    fig.savefig(shuffle_figure_dir / "06_per_parameter_scores.png", dpi=250, bbox_inches="tight")
    plt.show()


# Parte VI — comparação causal direta na ordem deliberada

## 22. Par automático AIG/DAL versus par forçado AMD/COST

A célula principal de permutação responde à hipótese **automática**:

> o par vizinho que maximiza a margem financeira em cada ordem deve indicar as
> portas causalmente relevantes.

Na ordem `d00`, entretanto, há uma segunda pergunta:

> ao colocar explicitamente AMD e COST em posições adjacentes, as portas que
> tocam AMD/COST ganham efeito causal, mesmo quando o algoritmo automático
> continua escolhendo AIG/DAL?

A versão 19.4 mede essas duas hipóteses separadamente. São construídos quatro
grupos:

1. parâmetros previstos pelo par automático;
2. controles casados para o par automático;
3. parâmetros previstos especificamente para AMD/COST;
4. controles casados especificamente para AMD/COST.

Os controles têm o mesmo tipo físico de porta (`RY` ou `CRY`) e são escolhidos
fora da união dos dois conjuntos previstos. Um mesmo theta pode pertencer aos
dois grupos previstos; esse cruzamento é registrado explicitamente.

### Custo computacional

Esta etapa **não executa COBYLA novamente**. Ela reutiliza o banco robusto de
40 restarts da ordem `d00`, repete apenas:

- a separação descoberta/validação;
- a descoberta das regiões angulares;
- as intervenções `Statevector` nos theta candidatos.

As saídas são gravadas separadamente para não sobrescrever a análise anterior.


In [ ]:
# ============================================================
# 49. COMPARAÇÃO DIRETA: PAR AUTOMÁTICO x AMD/COST
# ============================================================

run_direct_deliberate_pair_comparison = True

direct_table_dir = (
    shuffle_output_dir
    / "tables_v19_4_direct"
)

direct_figure_dir = (
    shuffle_output_dir
    / "figures_v19_4_direct"
)

for directory in [
    direct_table_dir,
    direct_figure_dir,
]:
    directory.mkdir(
        parents=True,
        exist_ok=True,
    )


def choose_matched_controls_excluding(
    parameter_table,
    excluded_theta_indices,
):
    """
    Seleciona um controle para cada parâmetro previsto.

    Regras:
    - mesmo tipo físico de porta;
    - não pode pertencer a nenhum conjunto previsto;
    - não repete theta dentro do mesmo grupo de controles;
    - prioriza baixa exposição financeira;
    - em empate, aproxima o número de ocorrências da porta prevista.
    """
    predicted = (
        parameter_table[
            parameter_table[
                "predicted_by_finance"
            ]
        ]
        .copy()
        .reset_index(
            drop=True
        )
    )

    excluded = {
        int(
            theta_index
        )
        for theta_index
        in excluded_theta_indices
    }

    eligible_pool = (
        parameter_table[
            ~parameter_table[
                "theta_index"
            ]
            .astype(int)
            .isin(
                excluded
            )
        ]
        .copy()
    )

    selected_controls = []
    used_control_indices = set()

    for _, predicted_row in predicted.iterrows():
        same_type = eligible_pool[
            eligible_pool[
                "physical_type"
            ]
            == predicted_row[
                "physical_type"
            ]
        ].copy()

        same_type = same_type[
            ~same_type[
                "theta_index"
            ]
            .astype(int)
            .isin(
                used_control_indices
            )
        ]

        if same_type.empty:
            continue

        same_type[
            "control_priority"
        ] = (
            same_type[
                "financial_exposure_mean"
            ]
            + 1e-6
            * np.abs(
                same_type[
                    "n_occurrences"
                ]
                - predicted_row[
                    "n_occurrences"
                ]
            )
        )

        selected = (
            same_type
            .sort_values(
                [
                    "control_priority",
                    "theta_index",
                ]
            )
            .iloc[
                0
            ]
            .copy()
        )

        selected[
            "matched_to_theta_index"
        ] = int(
            predicted_row[
                "theta_index"
            ]
        )

        selected_controls.append(
            selected
        )

        used_control_indices.add(
            int(
                selected[
                    "theta_index"
                ]
            )
        )

    if selected_controls:
        controls = (
            pd.DataFrame(
                selected_controls
            )
            .reset_index(
                drop=True
            )
        )
    else:
        controls = pd.DataFrame(
            columns=[
                *parameter_table.columns,
                "matched_to_theta_index",
            ]
        )

    return (
        predicted,
        controls,
    )


def exact_label_permutation_test(
    values_a,
    values_b,
):
    """
    Teste exploratório unilateral para diferença de médias.

    H1:
        média(A) > média(B)

    As observações são scores por parâmetro e compartilham o mesmo banco;
    portanto, o p-valor deve ser interpretado como diagnóstico exploratório,
    não como prova independente definitiva.
    """
    values_a = np.asarray(
        values_a,
        dtype=float,
    )

    values_b = np.asarray(
        values_b,
        dtype=float,
    )

    values_a = values_a[
        np.isfinite(
            values_a
        )
    ]

    values_b = values_b[
        np.isfinite(
            values_b
        )
    ]

    if (
        len(
            values_a
        )
        < 2
        or len(
            values_b
        )
        < 2
    ):
        return {
            "mean_difference": np.nan,
            "p_one_sided": np.nan,
            "n_a": int(
                len(
                    values_a
                )
            ),
            "n_b": int(
                len(
                    values_b
                )
            ),
            "n_permutations": 0,
        }

    pooled = np.concatenate(
        [
            values_a,
            values_b,
        ]
    )

    n_a = int(
        len(
            values_a
        )
    )

    observed = float(
        np.mean(
            values_a
        )
        - np.mean(
            values_b
        )
    )

    total_combinations = math.comb(
        len(
            pooled
        ),
        n_a,
    )

    exceedances = 0
    evaluated = 0

    if total_combinations <= 50_000:
        index_range = range(
            len(
                pooled
            )
        )

        for group_a_indices in combinations(
            index_range,
            n_a,
        ):
            group_a_indices = set(
                group_a_indices
            )

            permuted_a = np.asarray(
                [
                    pooled[
                        index
                    ]
                    for index
                    in index_range
                    if index
                    in group_a_indices
                ],
                dtype=float,
            )

            permuted_b = np.asarray(
                [
                    pooled[
                        index
                    ]
                    for index
                    in index_range
                    if index
                    not in group_a_indices
                ],
                dtype=float,
            )

            permuted_difference = float(
                np.mean(
                    permuted_a
                )
                - np.mean(
                    permuted_b
                )
            )

            exceedances += int(
                permuted_difference
                >= observed
                - 1e-15
            )

            evaluated += 1

    else:
        local_rng = np.random.default_rng(
            shuffle_seed
            + 19_400
        )

        evaluated = 50_000

        for _ in range(
            evaluated
        ):
            permuted = local_rng.permutation(
                pooled
            )

            permuted_difference = float(
                np.mean(
                    permuted[
                        :n_a
                    ]
                )
                - np.mean(
                    permuted[
                        n_a:
                    ]
                )
            )

            exceedances += int(
                permuted_difference
                >= observed
            )

    p_value = float(
        (
            exceedances
            + 1
        )
        / (
            evaluated
            + 1
        )
    )

    return {
        "mean_difference": observed,
        "p_one_sided": p_value,
        "n_a": int(
            len(
                values_a
            )
        ),
        "n_b": int(
            len(
                values_b
            )
        ),
        "n_permutations": int(
            evaluated
        ),
    }


direct_pair_group_summary_df = pd.DataFrame()
direct_pair_effects_df = pd.DataFrame()
direct_pair_comparison_df = pd.DataFrame()
direct_pair_verdict_df = pd.DataFrame()

if not run_direct_deliberate_pair_comparison:
    print(
        "Comparação direta desativada."
    )

elif (
    "shuffle_summary_df"
    not in globals()
    or shuffle_summary_df.empty
):
    print(
        "Execute primeiro a célula 48 do teste de permutação."
    )

else:
    deliberate_rows = (
        shuffle_summary_df[
            shuffle_summary_df[
                "order_id"
            ]
            == "d00"
        ]
        .copy()
    )

    if deliberate_rows.empty:
        raise RuntimeError(
            "A ordem deliberada d00 não foi encontrada."
        )

    deliberate_row = (
        deliberate_rows.iloc[
            0
        ]
    )

    if not bool(
        deliberate_row[
            "deliberate_target_adjacent"
        ]
    ):
        raise RuntimeError(
            "A ordem d00 não possui o par deliberado adjacente."
        )

    deliberate_order = str(
        deliberate_row[
            "column_order"
        ]
    ).split(
        "|"
    )

    if set(
        deliberate_order
    ) != set(
        stocks_prices.columns
    ):
        raise RuntimeError(
            "A ordem d00 não contém exatamente os tickers do CSV original."
        )

    deliberate_prices = (
        stocks_prices[
            deliberate_order
        ]
        .copy(
            deep=True
        )
    )

    shared_ansatz_seed = (
        random_seed
        + 100
        * n_assets
        + shuffle_k
    )

    direct_context = (
        build_problem_context_from_price_copy(
            copied_prices=deliberate_prices,
            k_value=shuffle_k,
            context_seed=shared_ansatz_seed,
        )
    )

    direct_analysis = (
        financial_decision_table(
            direct_context
        )
    )

    automatic_pair = tuple(
        int(
            value
        )
        for value
        in direct_analysis[
            "decisive_pair"
        ]
    )

    amd_index = int(
        direct_context[
            "tickers"
        ].index(
            "AMD"
        )
    )

    cost_index = int(
        direct_context[
            "tickers"
        ].index(
            "COST"
        )
    )

    forced_amd_cost_pair = tuple(
        sorted(
            [
                amd_index,
                cost_index,
            ]
        )
    )

    if abs(
        amd_index
        - cost_index
    ) != 1:
        raise RuntimeError(
            "AMD e COST não estão adjacentes na ordem d00."
        )

    automatic_pair_tickers = "/".join(
        direct_context[
            "tickers"
        ][
            index
        ]
        for index
        in automatic_pair
    )

    forced_pair_tickers = "/".join(
        direct_context[
            "tickers"
        ][
            index
        ]
        for index
        in forced_amd_cost_pair
    )

    pair_table = (
        direct_analysis[
            "pair_df"
        ]
    )

    forced_pair_match = pair_table[
        (
            pair_table[
                "left_qubit"
            ]
            == forced_amd_cost_pair[
                0
            ]
        )
        & (
            pair_table[
                "right_qubit"
            ]
            == forced_amd_cost_pair[
                1
            ]
        )
    ]

    if forced_pair_match.empty:
        raise RuntimeError(
            "O par AMD/COST não foi encontrado na tabela de pares adjacentes."
        )

    forced_pair_row = (
        forced_pair_match.iloc[
            0
        ]
    )

    automatic_pair_row = (
        direct_analysis[
            "decisive_pair_row"
        ]
    )

    direct_physical_map, _ = (
        build_physical_parameter_map(
            direct_context[
                "ansatz"
            ]
        )
    )

    automatic_parameter_table = (
        annotate_parameter_financial_exposure(
            parameter_map=direct_physical_map,
            decision_df=direct_analysis[
                "decision_df"
            ],
            decisive_pair=automatic_pair,
        )
    )

    forced_parameter_table = (
        annotate_parameter_financial_exposure(
            parameter_map=direct_physical_map,
            decision_df=direct_analysis[
                "decision_df"
            ],
            decisive_pair=forced_amd_cost_pair,
        )
    )

    automatic_predicted_seed = (
        automatic_parameter_table[
            automatic_parameter_table[
                "predicted_by_finance"
            ]
        ]
        .copy()
    )

    forced_predicted_seed = (
        forced_parameter_table[
            forced_parameter_table[
                "predicted_by_finance"
            ]
        ]
        .copy()
    )

    predicted_union = {
        int(
            theta_index
        )
        for theta_index
        in pd.concat(
            [
                automatic_predicted_seed[
                    "theta_index"
                ],
                forced_predicted_seed[
                    "theta_index"
                ],
            ],
            ignore_index=True,
        )
    }

    automatic_predicted, automatic_controls = (
        choose_matched_controls_excluding(
            parameter_table=automatic_parameter_table,
            excluded_theta_indices=predicted_union,
        )
    )

    forced_predicted, forced_controls = (
        choose_matched_controls_excluding(
            parameter_table=forced_parameter_table,
            excluded_theta_indices=predicted_union,
        )
    )

    membership_frames = [
        automatic_predicted.assign(
            candidate_group=(
                "automatic_pair_predicted"
            ),
            target_pair=(
                automatic_pair_tickers
            ),
            target_pair_indices=str(
                automatic_pair
            ),
        ),
        automatic_controls.assign(
            candidate_group=(
                "automatic_pair_control"
            ),
            target_pair=(
                automatic_pair_tickers
            ),
            target_pair_indices=str(
                automatic_pair
            ),
        ),
        forced_predicted.assign(
            candidate_group=(
                "forced_amd_cost_predicted"
            ),
            target_pair=(
                forced_pair_tickers
            ),
            target_pair_indices=str(
                forced_amd_cost_pair
            ),
        ),
        forced_controls.assign(
            candidate_group=(
                "forced_amd_cost_control"
            ),
            target_pair=(
                forced_pair_tickers
            ),
            target_pair_indices=str(
                forced_amd_cost_pair
            ),
        ),
    ]

    direct_membership_df = (
        pd.concat(
            membership_frames,
            ignore_index=True,
            sort=False,
        )
        .reset_index(
            drop=True
        )
    )

    candidate_unique_df = (
        direct_membership_df[
            [
                "theta_index",
                "angular_period",
            ]
        ]
        .drop_duplicates(
            subset=[
                "theta_index"
            ]
        )
        .sort_values(
            "theta_index"
        )
        .reset_index(
            drop=True
        )
    )

    deliberate_short_id = (
        f"d00_"
        f"{deliberate_row['order_hash']}"
    )

    deliberate_checkpoint = (
        shuffle_bank_dir
        / deliberate_short_id
        / "bank.pkl"
    )

    if not deliberate_checkpoint.exists():
        raise FileNotFoundError(
            "Checkpoint robusto de d00 não encontrado: "
            f"{deliberate_checkpoint}"
        )

    deliberate_bank_rows = (
        pd.read_pickle(
            deliberate_checkpoint
        )
    )

    if not isinstance(
        deliberate_bank_rows,
        list,
    ):
        raise TypeError(
            "O checkpoint d00 deveria conter uma lista de execuções."
        )

    deliberate_bank_df = (
        pd.DataFrame(
            deliberate_bank_rows
        )
    )

    if "status" in deliberate_bank_df.columns:
        deliberate_bank_df = (
            deliberate_bank_df[
                deliberate_bank_df[
                    "status"
                ]
                == "ok"
            ]
            .copy()
        )

    deliberate_bank_df = (
        deliberate_bank_df
        .sort_values(
            "run"
        )
        .drop_duplicates(
            subset=[
                "run"
            ],
            keep="last",
        )
        .head(
            int(
                shuffle_restarts_per_order
            )
        )
        .reset_index(
            drop=True
        )
    )

    required_bank_columns = {
        "best_parameters",
        "gap_exact_eval",
        "p_exact_eval",
        "energy_exact_eval",
    }

    missing_bank_columns = (
        required_bank_columns
        - set(
            deliberate_bank_df.columns
        )
    )

    if missing_bank_columns:
        raise KeyError(
            "Banco d00 sem colunas necessárias: "
            f"{sorted(missing_bank_columns)}"
        )

    if len(
        deliberate_bank_df
    ) < 12:
        raise RuntimeError(
            "O banco d00 possui poucos vetores para separar "
            "descoberta e validação."
        )

    split_rng = np.random.default_rng(
        shuffle_seed
        + 900_000
        + sum(
            ord(
                character
            )
            for character
            in deliberate_short_id
        )
    )

    split_permutation = split_rng.permutation(
        len(
            deliberate_bank_df
        )
    )

    discovery_size = int(
        np.floor(
            shuffle_discovery_fraction
            * len(
                deliberate_bank_df
            )
        )
    )

    discovery_size = min(
        max(
            discovery_size,
            6,
        ),
        len(
            deliberate_bank_df
        )
        - 6,
    )

    direct_discovery_df = (
        deliberate_bank_df.iloc[
            split_permutation[
                :discovery_size
            ]
        ]
        .copy()
        .reset_index(
            drop=True
        )
    )

    direct_validation_df = (
        deliberate_bank_df.iloc[
            split_permutation[
                discovery_size:
            ]
        ]
        .copy()
        .reset_index(
            drop=True
        )
    )

    direct_regions_df = (
        discover_shuffle_regions(
            discovery_df=direct_discovery_df,
            candidate_table=candidate_unique_df,
        )
    )

    ordered_good = (
        direct_validation_df
        .sort_values(
            [
                "gap_exact_eval",
                "p_exact_eval",
            ],
            ascending=[
                True,
                False,
            ],
        )
    )

    ordered_bad = (
        direct_validation_df
        .sort_values(
            [
                "gap_exact_eval",
                "p_exact_eval",
            ],
            ascending=[
                False,
                True,
            ],
        )
    )

    n_intervention_vectors = min(
        int(
            shuffle_n_intervention_vectors
        ),
        max(
            1,
            len(
                direct_validation_df
            )
            // 2,
        ),
    )

    direct_good_df = (
        ordered_good.head(
            n_intervention_vectors
        )
        .copy()
        .reset_index(
            drop=True
        )
    )

    direct_bad_df = (
        ordered_bad.head(
            n_intervention_vectors
        )
        .copy()
        .reset_index(
            drop=True
        )
    )

    direct_energy_values = (
        direct_analysis[
            "enumeration_df"
        ][
            "objective"
        ].to_numpy(
            dtype=float
        )
    )

    direct_energy_span = float(
        np.max(
            direct_energy_values
        )
        - np.min(
            direct_energy_values
        )
    )

    direct_effect_rows = []

    for _, region_row in direct_regions_df.iterrows():
        if not bool(
            region_row[
                "region_valid"
            ]
        ):
            continue

        direct_effect = (
            causal_effect_for_candidate(
                context=direct_context,
                good_validation_df=direct_good_df,
                bad_validation_df=direct_bad_df,
                theta_index=int(
                    region_row[
                        "theta_index"
                    ]
                ),
                good_angle=float(
                    region_row[
                        "good_angle"
                    ]
                ),
                bad_angle=float(
                    region_row[
                        "bad_angle"
                    ]
                ),
                energy_span=direct_energy_span,
            )
        )

        direct_effect_rows.append(
            {
                "theta_index": int(
                    region_row[
                        "theta_index"
                    ]
                ),
                "good_angle": float(
                    region_row[
                        "good_angle"
                    ]
                ),
                "bad_angle": float(
                    region_row[
                        "bad_angle"
                    ]
                ),
                "angular_separation": float(
                    region_row[
                        "angular_separation"
                    ]
                ),
                **direct_effect,
            }
        )

    direct_effect_base_df = pd.DataFrame(
        direct_effect_rows
    )

    direct_pair_effects_df = (
        direct_membership_df.merge(
            direct_effect_base_df,
            on="theta_index",
            how="left",
            validate="many_to_one",
        )
    )

    group_order = [
        "automatic_pair_predicted",
        "automatic_pair_control",
        "forced_amd_cost_predicted",
        "forced_amd_cost_control",
    ]

    group_labels = {
        "automatic_pair_predicted": (
            f"automático previsto\n"
            f"({automatic_pair_tickers})"
        ),
        "automatic_pair_control": (
            "controle casado\n"
            "do automático"
        ),
        "forced_amd_cost_predicted": (
            "AMD/COST\n"
            "previsto direto"
        ),
        "forced_amd_cost_control": (
            "controle casado\n"
            "de AMD/COST"
        ),
    }

    group_summary_rows = []

    for group_name in group_order:
        group_membership = direct_pair_effects_df[
            direct_pair_effects_df[
                "candidate_group"
            ]
            == group_name
        ].copy()

        valid_scores = (
            group_membership[
                "causal_score_comparable"
            ]
            .dropna()
            .to_numpy(
                dtype=float
            )
        )

        group_summary_rows.append(
            {
                "candidate_group": group_name,
                "display_label": group_labels[
                    group_name
                ],
                "target_pair": (
                    group_membership[
                        "target_pair"
                    ].iloc[
                        0
                    ]
                    if not group_membership.empty
                    else None
                ),
                "parameter_count_total": int(
                    len(
                        group_membership
                    )
                ),
                "parameter_count_valid": int(
                    len(
                        valid_scores
                    )
                ),
                "causal_score_mean": (
                    float(
                        np.mean(
                            valid_scores
                        )
                    )
                    if len(
                        valid_scores
                    )
                    else np.nan
                ),
                "causal_score_median": (
                    float(
                        np.median(
                            valid_scores
                        )
                    )
                    if len(
                        valid_scores
                    )
                    else np.nan
                ),
                "causal_score_max": (
                    float(
                        np.max(
                            valid_scores
                        )
                    )
                    if len(
                        valid_scores
                    )
                    else np.nan
                ),
                "theta_indices_all": "|".join(
                    str(
                        int(
                            value
                        )
                    )
                    for value
                    in sorted(
                        group_membership[
                            "theta_index"
                        ]
                        .astype(int)
                        .unique()
                    )
                ),
                "theta_indices_valid": "|".join(
                    str(
                        int(
                            value
                        )
                    )
                    for value
                    in sorted(
                        group_membership.loc[
                            group_membership[
                                "causal_score_comparable"
                            ].notna(),
                            "theta_index",
                        ]
                        .astype(int)
                        .unique()
                    )
                ),
            }
        )

    direct_pair_group_summary_df = (
        pd.DataFrame(
            group_summary_rows
        )
    )

    summary_lookup = (
        direct_pair_group_summary_df
        .set_index(
            "candidate_group"
        )
    )

    def group_mean(
        group_name,
    ):
        return float(
            summary_lookup.loc[
                group_name,
                "causal_score_mean",
            ]
        )

    automatic_predicted_mean = group_mean(
        "automatic_pair_predicted"
    )

    automatic_control_mean = group_mean(
        "automatic_pair_control"
    )

    forced_predicted_mean = group_mean(
        "forced_amd_cost_predicted"
    )

    forced_control_mean = group_mean(
        "forced_amd_cost_control"
    )

    automatic_minus_control = (
        automatic_predicted_mean
        - automatic_control_mean
        if np.isfinite(
            automatic_predicted_mean
        )
        and np.isfinite(
            automatic_control_mean
        )
        else np.nan
    )

    forced_minus_control = (
        forced_predicted_mean
        - forced_control_mean
        if np.isfinite(
            forced_predicted_mean
        )
        and np.isfinite(
            forced_control_mean
        )
        else np.nan
    )

    forced_minus_automatic = (
        forced_predicted_mean
        - automatic_predicted_mean
        if np.isfinite(
            forced_predicted_mean
        )
        and np.isfinite(
            automatic_predicted_mean
        )
        else np.nan
    )

    automatic_predicted_indices = set(
        automatic_predicted[
            "theta_index"
        ].astype(int)
    )

    forced_predicted_indices = set(
        forced_predicted[
            "theta_index"
        ].astype(int)
    )

    predicted_overlap = sorted(
        automatic_predicted_indices
        & forced_predicted_indices
    )

    automatic_control_indices = set(
        automatic_controls[
            "theta_index"
        ].astype(int)
    )

    forced_control_indices = set(
        forced_controls[
            "theta_index"
        ].astype(int)
    )

    control_overlap = sorted(
        automatic_control_indices
        & forced_control_indices
    )

    automatic_scores = (
        direct_pair_effects_df.loc[
            direct_pair_effects_df[
                "candidate_group"
            ]
            == "automatic_pair_predicted",
            "causal_score_comparable",
        ]
        .dropna()
        .to_numpy(
            dtype=float
        )
    )

    automatic_control_scores = (
        direct_pair_effects_df.loc[
            direct_pair_effects_df[
                "candidate_group"
            ]
            == "automatic_pair_control",
            "causal_score_comparable",
        ]
        .dropna()
        .to_numpy(
            dtype=float
        )
    )

    forced_scores = (
        direct_pair_effects_df.loc[
            direct_pair_effects_df[
                "candidate_group"
            ]
            == "forced_amd_cost_predicted",
            "causal_score_comparable",
        ]
        .dropna()
        .to_numpy(
            dtype=float
        )
    )

    forced_control_scores = (
        direct_pair_effects_df.loc[
            direct_pair_effects_df[
                "candidate_group"
            ]
            == "forced_amd_cost_control",
            "causal_score_comparable",
        ]
        .dropna()
        .to_numpy(
            dtype=float
        )
    )

    automatic_control_test = (
        exact_label_permutation_test(
            automatic_scores,
            automatic_control_scores,
        )
    )

    forced_control_test = (
        exact_label_permutation_test(
            forced_scores,
            forced_control_scores,
        )
    )

    forced_automatic_test = (
        exact_label_permutation_test(
            forced_scores,
            automatic_scores,
        )
    )

    direct_pair_comparison_df = pd.DataFrame(
        [
            {
                "order_id": "d00",
                "automatic_pair_indices": str(
                    automatic_pair
                ),
                "automatic_pair_tickers": (
                    automatic_pair_tickers
                ),
                "automatic_pair_margin_min": float(
                    automatic_pair_row[
                        "pair_margin_min"
                    ]
                ),
                "automatic_pair_margin_sum": float(
                    automatic_pair_row[
                        "pair_margin_sum"
                    ]
                ),
                "forced_pair_indices": str(
                    forced_amd_cost_pair
                ),
                "forced_pair_tickers": (
                    forced_pair_tickers
                ),
                "forced_pair_margin_min": float(
                    forced_pair_row[
                        "pair_margin_min"
                    ]
                ),
                "forced_pair_margin_sum": float(
                    forced_pair_row[
                        "pair_margin_sum"
                    ]
                ),
                "forced_pair_is_automatic": bool(
                    automatic_pair
                    == forced_amd_cost_pair
                ),
                "automatic_predicted_mean": (
                    automatic_predicted_mean
                ),
                "automatic_control_mean": (
                    automatic_control_mean
                ),
                "automatic_minus_control": (
                    automatic_minus_control
                ),
                "forced_amd_cost_mean": (
                    forced_predicted_mean
                ),
                "forced_amd_cost_control_mean": (
                    forced_control_mean
                ),
                "forced_minus_control": (
                    forced_minus_control
                ),
                "forced_minus_automatic": (
                    forced_minus_automatic
                ),
                "predicted_overlap_count": int(
                    len(
                        predicted_overlap
                    )
                ),
                "predicted_overlap_thetas": "|".join(
                    str(
                        value
                    )
                    for value
                    in predicted_overlap
                ),
                "control_overlap_count": int(
                    len(
                        control_overlap
                    )
                ),
                "control_overlap_thetas": "|".join(
                    str(
                        value
                    )
                    for value
                    in control_overlap
                ),
                "automatic_vs_control_p_exploratory": (
                    automatic_control_test[
                        "p_one_sided"
                    ]
                ),
                "forced_vs_control_p_exploratory": (
                    forced_control_test[
                        "p_one_sided"
                    ]
                ),
                "forced_vs_automatic_p_exploratory": (
                    forced_automatic_test[
                        "p_one_sided"
                    ]
                ),
                "discovery_vectors": int(
                    len(
                        direct_discovery_df
                    )
                ),
                "validation_vectors": int(
                    len(
                        direct_validation_df
                    )
                ),
                "intervention_vectors_per_direction": int(
                    n_intervention_vectors
                ),
            }
        ]
    )

    if not np.isfinite(
        forced_predicted_mean
    ):
        direct_verdict = (
            "FORCED_AMD_COST_INSUFFICIENT_VALID_EFFECTS"
        )

    elif (
        np.isfinite(
            forced_control_mean
        )
        and forced_predicted_mean
        > forced_control_mean
        and np.isfinite(
            automatic_predicted_mean
        )
        and forced_predicted_mean
        > automatic_predicted_mean
    ):
        direct_verdict = (
            "AMD_COST_ABOVE_AUTOMATIC_AND_MATCHED_CONTROL"
        )

    elif (
        np.isfinite(
            forced_control_mean
        )
        and forced_predicted_mean
        > forced_control_mean
    ):
        direct_verdict = (
            "AMD_COST_ABOVE_MATCHED_CONTROL_ONLY"
        )

    elif (
        np.isfinite(
            automatic_predicted_mean
        )
        and automatic_predicted_mean
        > forced_predicted_mean
    ):
        direct_verdict = (
            "AUTOMATIC_PAIR_STRONGER_THAN_AMD_COST"
        )

    else:
        direct_verdict = (
            "NO_CLEAR_LOCAL_SEPARATION"
        )

    direct_pair_verdict_df = pd.DataFrame(
        [
            {
                "direct_verdict": (
                    direct_verdict
                ),
                "automatic_pair": (
                    automatic_pair_tickers
                ),
                "forced_pair": (
                    forced_pair_tickers
                ),
                "forced_pair_adjacent": True,
                "forced_pair_is_automatic": bool(
                    automatic_pair
                    == forced_amd_cost_pair
                ),
                "automatic_minus_control": (
                    automatic_minus_control
                ),
                "forced_minus_control": (
                    forced_minus_control
                ),
                "forced_minus_automatic": (
                    forced_minus_automatic
                ),
            }
        ]
    )

    direct_pair_group_summary_df.to_csv(
        direct_table_dir
        / "d00_group_summary.csv",
        index=False,
    )

    direct_pair_effects_df.to_csv(
        direct_table_dir
        / "d00_parameter_effects.csv",
        index=False,
    )

    direct_pair_comparison_df.to_csv(
        direct_table_dir
        / "d00_automatic_vs_amd_cost.csv",
        index=False,
    )

    direct_pair_verdict_df.to_csv(
        direct_table_dir
        / "d00_direct_verdict.csv",
        index=False,
    )

    print(
        "Par automático:",
        automatic_pair_tickers,
        automatic_pair,
    )

    print(
        "Par forçado:",
        forced_pair_tickers,
        forced_amd_cost_pair,
    )

    print(
        "Theta previstos em comum:",
        predicted_overlap,
    )

    display(
        direct_pair_group_summary_df
    )

    display(
        direct_pair_comparison_df
    )

    display(
        direct_pair_verdict_df
    )

    # --------------------------------------------------------
    # GRÁFICO 7 — MÉDIA DOS QUATRO GRUPOS
    # --------------------------------------------------------
    plot_summary = (
        direct_pair_group_summary_df
        .set_index(
            "candidate_group"
        )
        .loc[
            group_order
        ]
        .reset_index()
    )

    fig, ax = plt.subplots(
        figsize=(
            10,
            5.5,
        )
    )

    bars = ax.bar(
        np.arange(
            len(
                plot_summary
            )
        ),
        plot_summary[
            "causal_score_mean"
        ],
    )

    for bar, value in zip(
        bars,
        plot_summary[
            "causal_score_mean"
        ],
    ):
        if np.isfinite(
            value
        ):
            ax.text(
                bar.get_x()
                + bar.get_width()
                / 2,
                bar.get_height(),
                f"{value:.4f}",
                ha="center",
                va="bottom",
                fontsize=9,
            )

    ax.set_xticks(
        np.arange(
            len(
                plot_summary
            )
        )
    )

    ax.set_xticklabels(
        plot_summary[
            "display_label"
        ]
    )

    ax.set_ylabel(
        "Score causal comparável médio"
    )

    ax.set_title(
        "7. Ordem d00: AIG/DAL automático versus AMD/COST forçado"
    )

    ax.grid(
        axis="y",
        alpha=0.25,
    )

    fig.tight_layout()

    fig.savefig(
        direct_figure_dir
        / "07_d00_group_means.png",
        dpi=250,
        bbox_inches="tight",
    )

    plt.show()

    # --------------------------------------------------------
    # GRÁFICO 8 — SCORES POR THETA NOS QUATRO GRUPOS
    # --------------------------------------------------------
    fig, ax = plt.subplots(
        figsize=(
            11,
            6,
        )
    )

    jitter_rng = np.random.default_rng(
        shuffle_seed
        + 19_408
    )

    group_positions = {
        group_name: position
        for position, group_name
        in enumerate(
            group_order
        )
    }

    for group_name in group_order:
        subset = direct_pair_effects_df[
            (
                direct_pair_effects_df[
                    "candidate_group"
                ]
                == group_name
            )
            & direct_pair_effects_df[
                "causal_score_comparable"
            ].notna()
        ].copy()

        if subset.empty:
            continue

        x_values = (
            group_positions[
                group_name
            ]
            + jitter_rng.uniform(
                -0.10,
                0.10,
                len(
                    subset
                ),
            )
        )

        ax.scatter(
            x_values,
            subset[
                "causal_score_comparable"
            ],
            s=50,
            alpha=0.8,
        )

        for x_value, (_, row) in zip(
            x_values,
            subset.iterrows(),
        ):
            ax.annotate(
                f"θ{int(row['theta_index'])}",
                (
                    x_value,
                    row[
                        "causal_score_comparable"
                    ],
                ),
                xytext=(
                    3,
                    4,
                ),
                textcoords="offset points",
                fontsize=7,
            )

    ax.set_xticks(
        list(
            group_positions.values()
        )
    )

    ax.set_xticklabels(
        [
            group_labels[
                group_name
            ]
            for group_name
            in group_order
        ]
    )

    ax.set_ylabel(
        "Score causal comparável por parâmetro"
    )

    ax.set_title(
        "8. Ordem d00: distribuição causal por grupo e theta"
    )

    ax.grid(
        axis="y",
        alpha=0.25,
    )

    fig.tight_layout()

    fig.savefig(
        direct_figure_dir
        / "08_d00_per_parameter.png",
        dpi=250,
        bbox_inches="tight",
    )

    plt.show()

    # --------------------------------------------------------
    # GRÁFICO 9 — MARGEM DOS DOIS PARES
    # --------------------------------------------------------
    pair_margin_df = pd.DataFrame(
        [
            {
                "pair": (
                    f"automático\n"
                    f"{automatic_pair_tickers}"
                ),
                "margin_sum": float(
                    automatic_pair_row[
                        "pair_margin_sum"
                    ]
                ),
            },
            {
                "pair": (
                    "forçado\nAMD/COST"
                ),
                "margin_sum": float(
                    forced_pair_row[
                        "pair_margin_sum"
                    ]
                ),
            },
        ]
    )

    fig, ax = plt.subplots(
        figsize=(
            7,
            5,
        )
    )

    pair_bars = ax.bar(
        pair_margin_df[
            "pair"
        ],
        pair_margin_df[
            "margin_sum"
        ],
    )

    for bar, value in zip(
        pair_bars,
        pair_margin_df[
            "margin_sum"
        ],
    ):
        ax.text(
            bar.get_x()
            + bar.get_width()
            / 2,
            bar.get_height(),
            f"{value:.4f}",
            ha="center",
            va="bottom",
        )

    ax.set_ylabel(
        "Soma das margens de decisão"
    )

    ax.set_title(
        "9. Robustez financeira: par automático versus AMD/COST"
    )

    ax.grid(
        axis="y",
        alpha=0.25,
    )

    fig.tight_layout()

    fig.savefig(
        direct_figure_dir
        / "09_d00_pair_margins.png",
        dpi=250,
        bbox_inches="tight",
    )

    plt.show()


# Parte VII — tabelas e gráficos finais

## 23. Saída multi-$k$

A tabela final reúne previsão, efeito causal, qualidade dos bancos e veredito.

As saídas do teste de permutação são salvas separadamente em:

```text
shuffle_summary.csv
shuffle_causal_effects.csv
shuffle_correlation.csv
shuffle_verdict.csv
```

Assim, uma frente pode ser executada sem sobrescrever a outra.


In [ ]:
# ============================================================
# 49. TABELA FINAL ÚNICA DA ANÁLISE MULTI-K
# ============================================================

final_rows = []

if not run_multi_k_analysis:
    final_results_df = pd.DataFrame()

    print(
        "Tabela multi-k não gerada porque "
        "run_multi_k_analysis=False."
    )

else:
    for _, summary_row in hypothesis_summary_df.iterrows():
        configuration = (
            int(
                summary_row[
                    "n"
                ]
            ),
            int(
                summary_row[
                    "k"
                ]
            ),
        )

        local_table = ranked_causal_tables[
            configuration
        ]

        top_parameter = local_table.iloc[
            0
        ]

        final_rows.append(
            {
                "n": configuration[
                    0
                ],
                "k": configuration[
                    1
                ],
                "decisive_pair": summary_row[
                    "decisive_pair"
                ],
                "decisive_tickers": summary_row[
                    "decisive_tickers"
                ],
                "predicted_parameter_count": int(
                    summary_row[
                        "predicted_parameter_count"
                    ]
                ),
                "causal_signal_present": bool(
                    summary_row[
                        "causal_signal_present"
                    ]
                ),
                "max_causal_score": float(
                    summary_row[
                        "max_causal_score"
                    ]
                ),
                "top_causal_thetas": summary_row[
                    "top_causal_thetas"
                ],
                "top_theta": int(
                    top_parameter[
                        "theta_index"
                    ]
                ),
                "top_theta_qubits": str(
                    top_parameter[
                        "physical_qubits"
                    ]
                ),
                "top_theta_predicted": bool(
                    top_parameter[
                        "predicted_by_finance"
                    ]
                ),
                "top3_hit_rate_pct": summary_row[
                    "top3_hit_rate_pct"
                ],
                "predicted_top_quartile_rate_pct": (
                    summary_row[
                        "predicted_top_quartile_rate_pct"
                    ]
                ),
                "spearman_finance_causal": (
                    summary_row[
                        "spearman_finance_causal"
                    ]
                ),
                "bank_source": summary_row[
                    "bank_source"
                ],
                "cobyla_budget": int(
                    summary_row[
                        "cobyla_budget"
                    ]
                ),
                "bank_runs": int(
                    summary_row[
                        "bank_runs"
                    ]
                ),
                "final_evidence": bool(
                    summary_row[
                        "final_evidence"
                    ]
                ),
                "verdict": final_verdict,
            }
        )

    final_results_df = pd.DataFrame(
        final_rows
    )

    final_results_df.to_csv(
        financial_output_dir
        / "final_results_v19_2.csv",
        index=False,
    )

    display(
        final_results_df
    )


## 24. Visualizações

Os quatro gráficos multi-$k$ mostram:

1. margem financeira por ativo;
2. score causal por theta;
3. exposição financeira versus score causal;
4. taxa de acerto da previsão por $k$.

A célula de permutação produz visualizações adicionais para:

- margem versus efeito causal;
- previstos versus controles;
- ganho previsto-controle;
- invariância física;
- desempenho da ordem deliberada;
- score por parâmetro.


In [ ]:
if not run_multi_k_analysis:
    print(
        "Gráficos multi-k ignorados porque "
        "run_multi_k_analysis=False."
    )
else:
    # ============================================================
    # 50. QUATRO GRÁFICOS FINAIS DA ANÁLISE MULTI-K
    # ============================================================

    # ------------------------------------------------------------
    # GRÁFICO 1 — margem financeira dos ativos
    # ------------------------------------------------------------
    asset_plot_rows = []

    for configuration, analysis in financial_context_analysis.items():
        n_value, k_value = configuration

        local = analysis[
            "decision_df"
        ].copy()

        local[
            "configuration"
        ] = f"k={k_value}"

        asset_plot_rows.append(
            local
        )

    asset_plot_df = pd.concat(
        asset_plot_rows,
        ignore_index=True,
    )

    fig, ax = plt.subplots(
        figsize=(
            11,
            6,
        )
    )

    for k_value, group in asset_plot_df.groupby(
        asset_plot_df[
            "configuration"
        ]
    ):
        ax.plot(
            group[
                "asset_index"
            ],
            group[
                "decision_margin"
            ],
            marker="o",
            label=k_value,
        )

    ax.set_xticks(
        range(
            n_assets
        )
    )

    ax.set_xticklabels(
        tickers,
        rotation=35,
    )

    ax.set_xlabel(
        "Ativo / qubit"
    )

    ax.set_ylabel(
        "Margem mínima de troca"
    )

    ax.set_title(
        "1. Robustez financeira de cada decisão"
    )

    ax.grid(
        alpha=0.25
    )

    ax.legend()

    fig.tight_layout()

    fig.savefig(
        financial_figure_dir
        / "01_asset_decision_margin.png",
        dpi=250,
        bbox_inches="tight",
    )

    plt.show()


    # ------------------------------------------------------------
    # GRÁFICO 2 — score causal por theta
    # ------------------------------------------------------------
    fig, ax = plt.subplots(
        figsize=(
            11,
            6,
        )
    )

    for configuration, local_table in ranked_causal_tables.items():
        _, k_value = configuration

        ax.plot(
            local_table[
                "theta_index"
            ],
            local_table[
                "causal_score"
            ],
            marker="o",
            label=f"k={k_value}",
        )

    ax.set_xlabel(
        "Índice do theta"
    )

    ax.set_ylabel(
        "Score causal normalizado"
    )

    ax.set_title(
        "2. Hierarquia causal em cada valor de k"
    )

    ax.grid(
        alpha=0.25
    )

    ax.legend()

    fig.tight_layout()

    fig.savefig(
        financial_figure_dir
        / "02_causal_score_by_theta.png",
        dpi=250,
        bbox_inches="tight",
    )

    plt.show()


    # ------------------------------------------------------------
    # GRÁFICO 3 — exposição financeira x efeito causal
    # ------------------------------------------------------------
    fig, ax = plt.subplots(
        figsize=(
            9,
            6,
        )
    )

    for configuration, local_table in ranked_causal_tables.items():
        _, k_value = configuration

        ax.scatter(
            local_table[
                "financial_exposure_mean"
            ],
            local_table[
                "causal_score"
            ],
            label=f"k={k_value}",
        )

    ax.set_xlabel(
        "Exposição financeira média dos qubits tocados"
    )

    ax.set_ylabel(
        "Score causal"
    )

    ax.set_title(
        "3. A importância causal acompanha a margem financeira?"
    )

    ax.grid(
        alpha=0.25
    )

    ax.legend()

    fig.tight_layout()

    fig.savefig(
        financial_figure_dir
        / "03_financial_exposure_vs_causal.png",
        dpi=250,
        bbox_inches="tight",
    )

    plt.show()


    # ------------------------------------------------------------
    # GRÁFICO 4 — acerto da previsão por k
    # ------------------------------------------------------------
    fig, ax = plt.subplots(
        figsize=(
            9,
            5,
        )
    )

    summary_plot = hypothesis_summary_df.sort_values(
        "k"
    )

    x_positions = np.arange(
        len(
            summary_plot
        )
    )

    bar_width = 0.36

    ax.bar(
        x_positions
        - bar_width / 2,
        summary_plot[
            "top3_hit_rate_pct"
        ],
        width=bar_width,
        label="Top-3 causal previsto",
    )

    ax.bar(
        x_positions
        + bar_width / 2,
        summary_plot[
            "predicted_top_quartile_rate_pct"
        ],
        width=bar_width,
        label="Parâmetros previstos no quartil causal",
    )

    ax.set_xticks(
        x_positions
    )

    ax.set_xticklabels(
        [
            f"k={int(value)}"
            for value
            in summary_plot[
                "k"
            ]
        ]
    )

    ax.set_ylim(
        0,
        105,
    )

    ax.set_xlabel(
        "Quantidade de ativos escolhidos"
    )

    ax.set_ylabel(
        "Taxa de acerto (%)"
    )

    ax.set_title(
        "4. Validação da hipótese financeira entre valores de k"
    )

    ax.grid(
        axis="y",
        alpha=0.25,
    )

    ax.legend()

    fig.tight_layout()

    fig.savefig(
        financial_figure_dir
        / "04_hypothesis_hit_rate.png",
        dpi=250,
        bbox_inches="tight",
    )

    plt.show()


# Guia de execução

## A. Verificar apenas a reconstrução original

Execute as células até a solução exata. Todas as auditorias devem retornar
`True`.

## B. Rodar somente a análise multi-$k$

Na configuração inicial:

```python
run_multi_k_analysis = True
run_csv_permutation_test = False
```

### Usar bancos existentes

```python
generate_financial_banks = False
```

### Gerar os quatro bancos robustos

```python
financial_robust_mode = True
generate_financial_banks = True
```

Depois da geração:

```python
generate_financial_banks = False
```

Mantenha `financial_robust_mode=True` para carregar a pasta robusta.

## C. Rodar somente o teste de permutação

Na primeira célula:

```python
run_multi_k_analysis = False
generate_financial_banks = False
```

Na célula 48:

```python
run_csv_permutation_test = True
shuffle_robust_mode = False
```

O modo piloto agora usa 14 restarts e não quebra a separação
descoberta/validação.

## D. Teste de permutação robusto

```python
run_csv_permutation_test = True
shuffle_robust_mode = True
```

Isso executa 15 ordens aleatórias, uma ordem deliberada e 40 restarts por ordem.

## E. Retomada

Cada ordem tem seu próprio `bank.pkl`. Ao reiniciar, o notebook ignora
`run_index` já concluídos.

## F. Interpretação dos vereditos

### Multi-$k$

```text
SUPPORTED_ACROSS_K
NOT_SUPPORTED_ACROSS_K
MIXED_EVIDENCE
PRELIMINARY_OR_FAILED_POSITIVE_CONTROL
SKIPPED_MULTI_K_ANALYSIS
```

### Permutação

```text
SUPPORTS_FINANCIAL_LOCATION_HYPOTHESIS
DOES_NOT_SUPPORT_HYPOTHESIS
MIXED_EVIDENCE
PRELIMINARY_ONLY
```

Nenhum veredito substitui a inspeção de:

- tamanho do efeito;
- quantidade de ordens válidas;
- parâmetros previstos e controles;
- invariância física;
- dispersão entre restarts.


## Reteste corrigido AMD/COST

Use:

```python
run_csv_permutation_test = True
shuffle_robust_mode = True
shuffle_deliberate_pair = ("AMD", "COST")
```

Não apague os checkpoints anteriores. As 15 ordens aleatórias serão reutilizadas
e apenas a nova ordem deliberada será otimizada.

Antes de interpretar o resultado, confirme na tabela:

```text
deliberate_target_adjacent = True
amd_cost_distance = 1
n_random_orders_effective
```


## G. Comparação direta AIG/DAL versus AMD/COST

Depois de concluir a célula 48, mantenha:

```python
run_direct_deliberate_pair_comparison = True
```

A nova célula reutiliza o checkpoint `d00` e não executa COBYLA. Confirme nas
tabelas:

```text
automatic_pair_tickers
forced_pair_tickers
forced_pair_is_automatic
automatic_minus_control
forced_minus_control
forced_minus_automatic
```

Os p-valores com sufixo `_exploratory` usam scores por parâmetro que compartilham
o mesmo banco; devem ser lidos como diagnóstico complementar, não como teste
independente definitivo.
